# 💳 Diagnóstico, Calibração e Governança do Motor de Trilhas — CPGF (2013–2026) — Versão 1.3.2

Este notebook executa **integralmente** o pipeline do CPGF a partir do consolidado bruto e preserva a separação entre:

- `VERSAO_REGRAS = 1.2.0`;
- `VERSAO_MOTOR = 1.3.2`.

A V1.3.2 é um **PATCH final da camada de governança da exposição**. Nenhuma regra T01–T09 é alterada.

---

## 🆕 Motivo do patch 1.3.2

A V1.3.1 mostrou que o controle de exposição em `UG × ano` funcionou adequadamente com decis anuais.

No nível `UG × fornecedor × ano`, porém, a distribuição de compras é extremamente discreta e concentrada em poucos valores inteiros. Como muitos fornecedor-anos possuem exatamente 1 ou 2 compras, a tentativa de formar decis preservando empates fez com que os dez estratos nominais colapsassem em poucas classes.

Isso não é erro dos dados. É uma consequência da distribuição.

### Solução

A V1.3.2 adota duas estratégias distintas:

#### `UG × ano`

Mantém:

`DECIL_EXPOSICAO_ANUAL`

calculado a partir de `N_OPERACOES_EFETIVAS`.

#### `UG × fornecedor × ano`

Substitui o decil por bandas fixas e substantivamente interpretáveis:

1. `1 compra`
2. `2 compras`
3. `3–4 compras`
4. `5–9 compras`
5. `10–19 compras`
6. `20+ compras`

A variável usada é:

`N_COMPRAS_FORNECEDOR`

---

## 🎯 Consequências metodológicas

A sobreposição e a contribuição marginal passam a ser examinadas:

- globalmente;
- ano a ano;
- por **banda de exposição do fornecedor**, no nível relacional;
- por **decil anual de exposição**, no nível da UG.

Isso evita chamar de “decil” uma classificação que não possui dez grupos efetivos.

---

## 🔒 Elementos preservados

A V1.3.2 mantém integralmente:

- T01–T09 em 1.2.0;
- famílias de evidência;
- T08/T09 como contextos;
- elegibilidade mínima para PCA/VIF;
- Jaccard, Phi e probabilidades condicionais;
- contribuição marginal;
- PCA e VIF exploratórios;
- contrato semântico de sensibilidade;
- amostragem por trilha × nível de triagem;
- pesos amostrais;
- validação humana separada da natureza da evidência;
- ausência de score opaco obrigatório.

A regressão T01–T09 continua sendo verificada de forma rígida.

## 1️⃣ Google Drive e dependências

Esta etapa monta o Google Drive e verifica as bibliotecas necessárias.

A instalação é feita somente para pacotes ausentes. Ao final, as versões utilizadas serão registradas no diretório de controle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import importlib
import subprocess
import sys

PACOTES = {
    'duckdb': 'duckdb',
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'psutil': 'psutil',
}

def garantir_pacote(modulo, pacote):
    try:
        return importlib.import_module(modulo)
    except ModuleNotFoundError:
        print(f'📦 Instalando {pacote}...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pacote
        ])
        return importlib.import_module(modulo)

for modulo, pacote in PACOTES.items():
    garantir_pacote(modulo, pacote)

print('✅ Dependências verificadas.')

## 2️⃣ Configuração geral e parâmetros versionados

Os parâmetros analíticos ficam concentrados em um único dicionário para que nenhuma regra relevante permaneça escondida no código.

### 🧪 Calibração completa de T05

A análise de sensibilidade utilizará:

- janelas: **15, 30, 45 e 60 dias**;
- mínimo de transações: **3, 5, 7 e 10**;
- mínimo de portadores: **2 e 3**;
- coeficiente de variação: **5%, 10%, 15%, 20% e 30%**.

Isso produz **160 combinações**.

> Os valores utilizados em T05 e T06 são parâmetros de pesquisa, e não limites normativos.

In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import product
from IPython.display import display

import csv
import gc
import hashlib
import json
import math
import os
import platform
import shutil
import time
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

from tqdm.auto import tqdm


# ============================================================
# 📁 CAMINHOS
# ============================================================

BASE_DIR = Path('/content/drive/MyDrive/Suprimentos de Fundos - CPGF')
INPUT_CSV = BASE_DIR / 'dados_consolidado' / 'CPGF_201301_a_202607.csv'

RESULT_DIR = BASE_DIR / 'Analise_Trilhas_v1_3_2'

CONTROLE_DIR = RESULT_DIR / '00_controle'
INTERMEDIARIOS_DIR = RESULT_DIR / '01_intermediarios_parquet'
DIAGNOSTICO_DIR = RESULT_DIR / '02_diagnostico'
TRILHAS_DIR = RESULT_DIR / '03_trilhas'
CONVERGENCIA_DIR = RESULT_DIR / '04_convergencia'
GRAFICOS_DIR = RESULT_DIR / '05_graficos'
EXPORTACOES_DIR = RESULT_DIR / '06_exportacoes'
LOGS_DIR = RESULT_DIR / '07_logs'

T01_DIR = TRILHAS_DIR / 'T01_fim_semana'
T02_DIR = TRILHAS_DIR / 'T02_compra_parcelada'
T03_DIR = TRILHAS_DIR / 'T03_repeticao_exata'
T04_DIR = TRILHAS_DIR / 'T04_multiportador'
T05_DIR = TRILHAS_DIR / 'T05_recorrencia'
T06_DIR = TRILHAS_DIR / 'T06_concentracao'
T07_DIR = TRILHAS_DIR / 'T07_saques'
T08_DIR = TRILHAS_DIR / 'T08_benford'
T09_DIR = TRILHAS_DIR / 'T09_limites'

GOVERNANCA_DIR = RESULT_DIR / '08_governanca_motor'

FAMILIAS_DIR = (
    GOVERNANCA_DIR
    / '01_familias_evidencia'
)

MATRIZES_DIR = (
    GOVERNANCA_DIR
    / '02_matrizes_flags'
)

SOBREPOSICAO_DIR = (
    GOVERNANCA_DIR
    / '03_sobreposicao'
)

MULTICOL_DIR = (
    GOVERNANCA_DIR
    / '04_multicolinearidade'
)

MARGINAL_DIR = (
    GOVERNANCA_DIR
    / '05_contribuicao_marginal'
)

PCA_DIR = (
    GOVERNANCA_DIR
    / '06_pca'
)

SENSIBILIDADE_MOTOR_DIR = (
    GOVERNANCA_DIR
    / '07_sensibilidade_motor'
)

VALIDACAO_DIR = (
    GOVERNANCA_DIR
    / '08_validacao'
)

CONTRATO_DASHBOARD_DIR = (
    GOVERNANCA_DIR
    / '09_contrato_dashboard'
)

TMP_DIR = Path('/content/cpgf_analise_tmp')
DB_PATH = TMP_DIR / 'cpgf_analise.duckdb'

for pasta in [
    RESULT_DIR, CONTROLE_DIR, INTERMEDIARIOS_DIR, DIAGNOSTICO_DIR,
    TRILHAS_DIR, CONVERGENCIA_DIR, GRAFICOS_DIR, EXPORTACOES_DIR,
    LOGS_DIR, T01_DIR, T02_DIR, T03_DIR, T04_DIR, T05_DIR,
    T06_DIR, T07_DIR, T08_DIR, T09_DIR,
    GOVERNANCA_DIR, FAMILIAS_DIR, MATRIZES_DIR,
    SOBREPOSICAO_DIR, MULTICOL_DIR, MARGINAL_DIR,
    PCA_DIR, SENSIBILIDADE_MOTOR_DIR, VALIDACAO_DIR,
    CONTRATO_DASHBOARD_DIR, TMP_DIR,
]:
    pasta.mkdir(parents=True, exist_ok=True)


# ============================================================
# 🧾 VERSÕES
# ============================================================

VERSAO_REGRAS = '1.2.0'
VERSAO_MOTOR = '1.3.2'
VERSAO_PREPARACAO = '1.0.0'

REPROCESSAR_PREPARACAO = True
REPROCESSAR_TRILHAS = False
REPROCESSAR_BENFORD = False

GERAR_GRAFICOS = True
EXPORTAR_CSVS_COMPLETOS = True


# ============================================================
# ⚙️ PARÂMETROS
# ============================================================

CONFIG = {
    'VERSAO_REGRAS': VERSAO_REGRAS,

    'T03': {
        'min_ocorrencias': 2,
        'reforcado_ocorrencias': 3,
    },

    'T04': {
        'min_portadores': 2,
        'reforcado_portadores': 3,
        'muito_elevado_portadores': 5,
    },

    'T05': {
        'min_transacoes': 5,
        'min_portadores': 2,
        'janela_dias': 30,
        'cv_base': 0.20,
        'cv_reforcado': 0.10,
        'tolerancia_mediana': 0.20,
        'share_mediana_referencia': 0.80,
        'janelas_calibracao': [15, 30, 45, 60],
        'min_transacoes_calibracao': [3, 5, 7, 10],
        'min_portadores_calibracao': [2, 3],
        'cv_calibracao': [0.05, 0.10, 0.15, 0.20, 0.30],
    },

    'T06': {
        'min_compras_identificadas': 20,
        'min_fornecedores': 3,
        'cobertura_min': 0.80,
        'share_base': 0.50,
        'share_reforcado': 0.70,
        'share_muito_elevado': 0.80,
    },

    'T07': {
        'min_saques_dia': 2,
        'reforcado_saques_dia': 3,
        'min_dias_recorrencia': 3,
        'percentil_priorizacao': 0.90,
        'min_comparaveis_ano': 10,
    },

    'T08': {
        'min_n_nao_aplicar': 300,
        'min_n_formal': 1000,
        'min_n_robusto': 3000,
        'min_valor_d12': 10.00,
        'sensibilidade_incluir_positivos': True,
        'persistencia_min_anos': 3,
        'persistencia_min_ratio': 0.50,
        'top_digitos_drilldown': 10,
        'top_valores_por_digito': 20,
        'min_ugs_comparaveis': 10,
        'anos_completos_inicio': 2013,
        'anos_completos_fim': 2025,
    },

    'T09': {
        'faixa_inferior': 0.90,
        'faixa_superior': 1.00,
        'habilitado_automaticamente': True,
        'modo': 'CENARIOS_PARALELOS_SEM_CATEGORIA',
        'classificar_no_limite_separadamente': True,
        'comparacao_monetaria': 'CENTAVOS_INTEIROS',
    },
}



# ============================================================
# 🧭 GOVERNANÇA DO MOTOR — V1.3.2
# ============================================================

MOTOR_CONFIG = {
    'VERSAO_MOTOR':
        VERSAO_MOTOR,

    'familias': {
        'F1': {
            'nome':
                'Conformidade operacional observável',
            'trilhas':
                ['T01', 'T02'],
        },

        'F2': {
            'nome':
                'Repetição e recorrência de aquisições',
            'trilhas':
                ['T03', 'T04', 'T05'],
        },

        'F3': {
            'nome':
                'Estrutura e concentração de fornecedor',
            'trilhas':
                ['T06'],
        },

        'F4': {
            'nome':
                'Comportamento de saque',
            'trilhas':
                ['T07'],
        },

        'F5': {
            'nome':
                'Contexto estatístico forense',
            'trilhas':
                ['T08'],
        },

        'F6': {
            'nome':
                'Contexto normativo-financeiro',
            'trilhas':
                ['T09'],
        },
    },

    'matriz_fornecedor': {
        'trilhas':
            ['T01', 'T02', 'T03', 'T04', 'T05', 'T06'],

        'familias':
            ['F1', 'F2', 'F3'],

        'unidade':
            'UG_FORNECEDOR_ANO',
    },

    'matriz_ug': {
        'trilhas':
            ['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07'],

        'familias':
            ['F1', 'F2', 'F3', 'F4'],

        'contextos':
            ['T08', 'T09'],

        'unidade':
            'UG_ANO',
    },

    'diagnostico': {
        'anos_completos_inicio':
            2013,

        'anos_completos_fim':
            2025,

        'vif_referencia': {
            'baixo_ate':
                2.5,

            'moderado_ate':
                5.0,

            'atencao_ate':
                10.0,
        },

        'pca':
            'EXPLORATORIA_NAO_EXCLUI_REGRAS',

        'sobreposicao_principal': [
            'JACCARD',
            'PHI',
            'P_A_DADO_B',
            'P_B_DADO_A',
            'CONTRIBUICAO_MARGINAL',
        ],

        # Regra de governança, não teste de hipótese:
        # flags com menos de 30 positivos ou 30 negativos
        # permanecem descritivas, mas não entram em PCA/VIF.
        'min_positivos_estatistica':
            30,

        'min_negativos_estatistica':
            30,

        # UG × ano: decis anuais preservados.
        'n_decis_exposicao_ug':
            10,

        # UG × fornecedor × ano:
        # bandas fixas para respeitar a distribuição discreta.
        'bandas_exposicao_fornecedor': [
            {
                'ordem': 1,
                'codigo': 'B01_1',
                'rotulo': '1 compra',
                'min': 1,
                'max': 1,
            },
            {
                'ordem': 2,
                'codigo': 'B02_2',
                'rotulo': '2 compras',
                'min': 2,
                'max': 2,
            },
            {
                'ordem': 3,
                'codigo': 'B03_3_4',
                'rotulo': '3–4 compras',
                'min': 3,
                'max': 4,
            },
            {
                'ordem': 4,
                'codigo': 'B04_5_9',
                'rotulo': '5–9 compras',
                'min': 5,
                'max': 9,
            },
            {
                'ordem': 5,
                'codigo': 'B05_10_19',
                'rotulo': '10–19 compras',
                'min': 10,
                'max': 19,
            },
            {
                'ordem': 6,
                'codigo': 'B06_20_MAIS',
                'rotulo': '20+ compras',
                'min': 20,
                'max': None,
            },
        ],
    },

    'sensibilidade': {
        'T03_min_ocorrencias':
            [2, 3, 4, 5],

        'T04_min_portadores':
            [2, 3, 4, 5],

        'T06_top1_share':
            [0.40, 0.50, 0.60, 0.70, 0.80],

        'T07_min_dias':
            [3, 5, 10],

        'T07_percentis':
            [0.75, 0.80, 0.90, 0.95],

        'T09_faixa_proximidade':
            [0.80, 0.85, 0.90, 0.95],
    },

    'validacao': {
        'status_permitidos': [
            'NAO_VALIDADO',
            'EM_ANALISE',
            'CONFIRMADO',
            'JUSTIFICADO',
            'FALSO_POSITIVO',
            'ERRO_DADO',
            'INCONCLUSIVO',
        ],

        'n_amostra_por_trilha':
            30,

        'estratificar_por':
            'NIVEL_TRIAGEM',

        'ponderar_por_estrato':
            True,
    },
}


# Baseline esperada da metodologia 1.2.0
# sobre o consolidado de referência.
BASELINE_V12 = {
    'T01': 49675,
    'T02': 14,
    'T03': 7534,
    'T04': 1384,
    'T05': 1693,
    'T06': 233,
    'T07': 1089,
    'T08': 12,
    'T09': 46941,
}


# ============================================================
# 🧾 CÓDIGOS DE TRANSAÇÃO
# ============================================================

CODIGO_COMPRA_NACIONAL = 'COMPRA A/V - R$ - APRES'
CODIGO_COMPRA_INTERNACIONAL = 'COMPRA A/V - INT$ - APRES'
CODIGO_COMPRA_PARCELADA = 'CPP LOJISTA TRF P/FATURA - REAL'

SAQUES_EFETIVOS = [
    'SAQUE CASH/ATM BB',
    'SAQUE - INT$ - APRES',
    'SAQUE MANUAL - CARTOES BB NA AGENCIA',
    'SAQUE - R$ - APRES',
]

CODIGOS_AJUSTE_CONTESTACAO = [
    'COMP A/V-SOL DISP C/CLI-R$ ANT VENC',
    'COMP A/V-SOL DISP C/CLI-R$ APOS VENC',
    'SAQUE BB B24HORAS-SOL C/CLIENTE',
    'VOUCHER - R$ - REVRS REAPR',
]


# ============================================================
# 📚 SCHEMA
# ============================================================

COLUNAS_ESPERADAS = [
    'CÓDIGO ÓRGÃO SUPERIOR',
    'NOME ÓRGÃO SUPERIOR',
    'CÓDIGO ÓRGÃO',
    'NOME ÓRGÃO',
    'CÓDIGO UNIDADE GESTORA',
    'NOME UNIDADE GESTORA',
    'ANO EXTRATO',
    'MÊS EXTRATO',
    'CPF PORTADOR',
    'NOME PORTADOR',
    'CNPJ OU CPF FAVORECIDO',
    'NOME FAVORECIDO',
    'TRANSAÇÃO',
    'DATA TRANSAÇÃO',
    'VALOR TRANSAÇÃO',
    'COMPETENCIA_ARQUIVO',
    'ARQUIVO_ORIGEM',
]

COMPETENCIA_INICIAL = '201301'
COMPETENCIA_FINAL = '202607'
N_COMPETENCIAS_ESPERADAS = 163
N_REGISTROS_REFERENCIA = 1_876_087

print('📥 Arquivo de entrada:', INPUT_CSV)
print('📤 Pasta de resultados:', RESULT_DIR)
print('🧪 Versão das regras:', VERSAO_REGRAS)
print('🧭 Versão do motor:', VERSAO_MOTOR)
print('✅ Configuração carregada.')

## 3️⃣ Funções auxiliares e rastreabilidade

São definidas funções para:

- exportação padronizada;
- IDs determinísticos;
- SHA-256;
- metadados;
- gráficos;
- configuração do DuckDB;
- checkpoints de etapas pesadas.

In [ ]:
def salvar_csv(df, caminho, index=False):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        caminho,
        sep=';',
        decimal=',',
        index=index,
        encoding='utf-8-sig',
        lineterminator='\n',
    )
    return caminho


def salvar_parquet(df, caminho, index=False):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(caminho, index=index)
    return caminho


def salvar_fig(caminho):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(caminho, dpi=220, bbox_inches='tight')
    plt.close()
    return caminho


def hash_texto(*partes):
    texto = '|'.join('' if p is None else str(p) for p in partes)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


def id_sinal(codigo_trilha, *partes):
    return f'{codigo_trilha}_{hash_texto(codigo_trilha, VERSAO_REGRAS, *partes)[:24]}'


def calcular_sha256_arquivo(caminho, bloco_mb=16):
    caminho = Path(caminho)

    if not caminho.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')

    tamanho = caminho.stat().st_size
    bloco = bloco_mb * 1024 * 1024
    h = hashlib.sha256()

    with open(caminho, 'rb') as f, tqdm(
        total=tamanho,
        unit='B',
        unit_scale=True,
        desc='🔐 SHA-256'
    ) as pbar:

        while True:
            parte = f.read(bloco)

            if not parte:
                break

            h.update(parte)
            pbar.update(len(parte))

    return h.hexdigest()


def salvar_json(obj, caminho):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)

    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

    return caminho


def registrar_ambiente():
    pacotes = {}

    for nome in [
        'duckdb', 'pandas', 'numpy',
        'pyarrow', 'matplotlib', 'openpyxl'
    ]:
        try:
            modulo = importlib.import_module(nome)
            pacotes[nome] = getattr(modulo, '__version__', 'desconhecida')
        except Exception:
            pacotes[nome] = 'indisponível'

    ambiente = {
        'data_hora': datetime.now().isoformat(),
        'python': sys.version,
        'plataforma': platform.platform(),
        'cpu_count': os.cpu_count(),
        'ram_total_gb': round(
            psutil.virtual_memory().total / (1024**3),
            2
        ),
        'pacotes': pacotes,
    }

    salvar_json(
        ambiente,
        CONTROLE_DIR / 'ambiente_execucao.json'
    )

    return ambiente


def configurar_duckdb(con):
    ram_total_gb = psutil.virtual_memory().total / (1024**3)
    memoria_duckdb_gb = max(2, int(ram_total_gb * 0.70))
    threads = max(1, min(os.cpu_count() or 2, 8))

    temp_duck = TMP_DIR / 'duckdb_temp'
    temp_duck.mkdir(parents=True, exist_ok=True)

    con.execute(
        f"PRAGMA memory_limit='{memoria_duckdb_gb}GB'"
    )
    con.execute(
        f"PRAGMA threads={threads}"
    )
    con.execute(
        f"SET temp_directory='{str(temp_duck)}'"
    )

    print(f'🧠 DuckDB memory_limit: {memoria_duckdb_gb} GB')
    print(f'🧵 DuckDB threads: {threads}')
    print(f'💽 DuckDB temp: {temp_duck}')


def checkpoint_valido(nome_etapa, fingerprint, arquivo_saida):
    arquivo_saida = Path(arquivo_saida)
    checkpoint_path = CONTROLE_DIR / 'checkpoints.json'

    if not arquivo_saida.exists() or not checkpoint_path.exists():
        return False

    try:
        checkpoints = json.loads(
            checkpoint_path.read_text(encoding='utf-8')
        )
    except Exception:
        return False

    return checkpoints.get(nome_etapa) == fingerprint


def gravar_checkpoint(nome_etapa, fingerprint):
    checkpoint_path = CONTROLE_DIR / 'checkpoints.json'

    if checkpoint_path.exists():
        try:
            checkpoints = json.loads(
                checkpoint_path.read_text(encoding='utf-8')
            )
        except Exception:
            checkpoints = {}
    else:
        checkpoints = {}

    checkpoints[nome_etapa] = fingerprint
    salvar_json(checkpoints, checkpoint_path)


AMBIENTE = registrar_ambiente()

print('✅ Funções auxiliares carregadas.')

## 4️⃣ Integridade, schema e hash da base

Antes de qualquer análise, o notebook verifica:

- existência do arquivo;
- cabeçalho com as 17 colunas esperadas;
- número de registros;
- 163 competências;
- competência inicial e final;
- tamanho do arquivo;
- hash SHA-256.

O schema é validado de forma estrita. Como o consolidado do CPGF já possui estrutura estabilizada, o notebook não tenta adivinhar aliases de colunas.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f'❌ Consolidado não encontrado:\n{INPUT_CSV}'
    )

print(f'✅ Arquivo encontrado: {INPUT_CSV}')
print(
    f'💾 Tamanho: '
    f'{INPUT_CSV.stat().st_size / (1024**3):.3f} GB'
)

cabecalho = pd.read_csv(
    INPUT_CSV,
    sep=';',
    encoding='utf-8-sig',
    nrows=0
).columns.tolist()

ausentes = [
    c for c in COLUNAS_ESPERADAS
    if c not in cabecalho
]

extras = [
    c for c in cabecalho
    if c not in COLUNAS_ESPERADAS
]

print(f'📋 Colunas encontradas: {len(cabecalho)}')

if ausentes:
    raise ValueError(
        '❌ Estrutura incompatível. Colunas ausentes:\n'
        + '\n'.join(f'- {c}' for c in ausentes)
    )

if extras:
    print('⚠️ Colunas extras encontradas:')

    for c in extras:
        print('  -', c)

if cabecalho == COLUNAS_ESPERADAS:
    print(
        '✅ Schema e ordem das colunas '
        'correspondem à referência.'
    )
else:
    print(
        'ℹ️ O conjunto de colunas está correto, '
        'mas a ordem difere da referência.'
    )

HASH_INPUT = calcular_sha256_arquivo(INPUT_CSV)

print('🔐 SHA-256:')
print(HASH_INPUT)

salvar_json(
    {
        'arquivo': str(INPUT_CSV),
        'tamanho_bytes': INPUT_CSV.stat().st_size,
        'sha256': HASH_INPUT,
        'cabecalho': cabecalho,
        'versao_regras': VERSAO_REGRAS,
        'versao_preparacao': VERSAO_PREPARACAO,
        'data_hora': datetime.now().isoformat(),
    },
    CONTROLE_DIR / 'integridade_arquivo.json'
)

RUN_FINGERPRINT = hash_texto(
    HASH_INPUT,
    VERSAO_REGRAS,
    json.dumps(
        CONFIG,
        sort_keys=True,
        ensure_ascii=False
    )
)

print(
    '🧬 Fingerprint da execução das regras:',
    RUN_FINGERPRINT[:24]
)

MOTOR_FINGERPRINT = hash_texto(
    RUN_FINGERPRINT,
    VERSAO_MOTOR,
    json.dumps(
        MOTOR_CONFIG,
        sort_keys=True,
        ensure_ascii=False
    )
)

print(
    '🧭 Fingerprint do motor V1.3:',
    MOTOR_FINGERPRINT[:24]
)

salvar_json(
    {
        'versao_regras':
            VERSAO_REGRAS,

        'versao_motor':
            VERSAO_MOTOR,

        'run_fingerprint_regras':
            RUN_FINGERPRINT,

        'motor_fingerprint':
            MOTOR_FINGERPRINT,

        'motor_config':
            MOTOR_CONFIG,
    },
    CONTROLE_DIR
    / 'governanca_motor_v1_3_2.json'
)

## 5️⃣ Abrir o consolidado com DuckDB

O DuckDB será a camada analítica principal.

A base bruta não é regravada nem modificada. O CSV é lido como texto e usado para criar uma camada normalizada em Parquet.

Essa estratégia reduz uso de memória e facilita reprocessamento no Colab.

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(str(DB_PATH))
configurar_duckdb(con)

csv_sql = str(INPUT_CSV).replace("'", "''")

con.execute(f"""
CREATE OR REPLACE VIEW raw_cpgf AS

SELECT *
FROM read_csv(
    '{csv_sql}',
    delim=';',
    header=true,
    all_varchar=true,
    encoding='utf-8',
    ignore_errors=false,
    quote='"',
    escape='"'
)
""")

controle_raw = con.execute("""
SELECT
    COUNT(*) AS n_registros,
    COUNT(DISTINCT COMPETENCIA_ARQUIVO)
        AS n_competencias,
    MIN(COMPETENCIA_ARQUIVO)
        AS competencia_min,
    MAX(COMPETENCIA_ARQUIVO)
        AS competencia_max
FROM raw_cpgf
""").df()

display(controle_raw)

n_registros = int(
    controle_raw.loc[0, 'n_registros']
)

n_comp = int(
    controle_raw.loc[0, 'n_competencias']
)

comp_min = str(
    controle_raw.loc[0, 'competencia_min']
)

comp_max = str(
    controle_raw.loc[0, 'competencia_max']
)

if n_registros != N_REGISTROS_REFERENCIA:
    print(
        f'⚠️ Registros: {n_registros:,}. '
        f'A referência conhecida era '
        f'{N_REGISTROS_REFERENCIA:,}.'
    )
else:
    print(
        f'✅ Registros conferidos: '
        f'{n_registros:,}'
    )

if n_comp != N_COMPETENCIAS_ESPERADAS:
    raise ValueError(
        f'❌ Foram encontradas {n_comp} competências; '
        f'eram esperadas '
        f'{N_COMPETENCIAS_ESPERADAS}.'
    )

if (
    comp_min != COMPETENCIA_INICIAL
    or comp_max != COMPETENCIA_FINAL
):
    raise ValueError(
        '❌ Intervalo de competências inesperado: '
        f'{comp_min}–{comp_max}.'
    )

print('✅ 163 competências localizadas.')
print(f'✅ Intervalo: {comp_min}–{comp_max}')

## 6️⃣ Preparação da camada analítica

A preparação cria `stg_cpgf_transacoes.parquet`.

### Transformações

- `ID_TRANSACAO`: identificador estável baseado na competência e na posição original dentro do arquivo mensal;
- `VALOR_NUM`: valor monetário normalizado;
- `VALOR_CENTAVOS`: inteiro em centavos para comparações exatas;
- `DATA_DT`: data efetiva da transação;
- ano, mês e dia da semana da transação;
- identificadores normalizados de UG, portador e favorecido;
- flags operacionais de compra, saque, contestação e sigilo.

### 💡 Por que centavos inteiros?

Comparações das trilhas T03 e T04 devem evitar erros de ponto flutuante.

`R$ 100,10 → 10010 centavos`

In [ ]:
STG_PARQUET = (
    INTERMEDIARIOS_DIR
    / 'stg_cpgf_transacoes.parquet'
)

PREP_META = (
    CONTROLE_DIR
    / 'preparacao_metadata.json'
)

prep_fingerprint = hash_texto(
    HASH_INPUT,
    'PREPARACAO',
    VERSAO_PREPARACAO
)

reutilizar_stg = (
    not REPROCESSAR_PREPARACAO
    and checkpoint_valido(
        'preparacao',
        prep_fingerprint,
        STG_PARQUET
    )
)

if reutilizar_stg:
    print(
        '⏭️ Camada de preparação compatível '
        'encontrada. Reutilizando Parquet.'
    )

else:
    print('🧹 Preparando a camada analítica...')

    compras_nacional_sql = (
        CODIGO_COMPRA_NACIONAL
        .replace("'", "''")
    )

    compras_internacional_sql = (
        CODIGO_COMPRA_INTERNACIONAL
        .replace("'", "''")
    )

    compra_parcelada_sql = (
        CODIGO_COMPRA_PARCELADA
        .replace("'", "''")
    )

    saques_sql = ', '.join(
        "'" + x.replace("'", "''") + "'"
        for x in SAQUES_EFETIVOS
    )

    ajustes_sql = ', '.join(
        "'" + x.replace("'", "''") + "'"
        for x in CODIGOS_AJUSTE_CONTESTACAO
    )

    stg_path_sql = (
        str(STG_PARQUET)
        .replace("'", "''")
    )

    stg_sql = f"""
    COPY (
        WITH raw_indexed AS (
            SELECT
                ROW_NUMBER() OVER ()
                    AS ID_LINHA_GLOBAL,
                *
            FROM raw_cpgf
        ),

        numerado AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY ARQUIVO_ORIGEM
                    ORDER BY ID_LINHA_GLOBAL
                ) AS ID_LINHA_ARQUIVO
            FROM raw_indexed
        ),

        preparado AS (
            SELECT
                *,

                COMPETENCIA_ARQUIVO
                    || ':'
                    || LPAD(
                        CAST(
                            ID_LINHA_ARQUIVO
                            AS VARCHAR
                        ),
                        8,
                        '0'
                    )
                    AS ID_TRANSACAO,

                TRIM(
                    "CÓDIGO UNIDADE GESTORA"
                ) AS UG_ID,

                CASE
                    WHEN TRIM(
                        COALESCE(
                            "CPF PORTADOR",
                            ''
                        )
                    ) IN ('', '-1')
                        THEN NULL

                    WHEN LOWER(
                        COALESCE(
                            "NOME PORTADOR",
                            ''
                        )
                    ) LIKE '%sigilo%'
                        THEN NULL

                    ELSE NULLIF(
                        REGEXP_REPLACE(
                            TRIM("CPF PORTADOR"),
                            '[^0-9]',
                            '',
                            'g'
                        ),
                        ''
                    )
                END AS PORTADOR_ID,

                CASE
                    WHEN TRIM(
                        COALESCE(
                            "CNPJ OU CPF FAVORECIDO",
                            ''
                        )
                    ) IN ('', '-1')
                        THEN NULL

                    WHEN LOWER(
                        COALESCE(
                            "NOME FAVORECIDO",
                            ''
                        )
                    ) LIKE '%sigilo%'
                        THEN NULL

                    WHEN LOWER(
                        COALESCE(
                            "NOME FAVORECIDO",
                            ''
                        )
                    ) LIKE '%sem inform%'
                        THEN NULL

                    ELSE NULLIF(
                        REGEXP_REPLACE(
                            TRIM(
                                "CNPJ OU CPF FAVORECIDO"
                            ),
                            '[^0-9]',
                            '',
                            'g'
                        ),
                        ''
                    )
                END AS FAVORECIDO_ID,

                TRY_CAST(
                    CASE
                        WHEN CONTAINS(
                            REPLACE(
                                REPLACE(
                                    TRIM(
                                        "VALOR TRANSAÇÃO"
                                    ),
                                    'R$',
                                    ''
                                ),
                                ' ',
                                ''
                            ),
                            ','
                        )
                        THEN REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        TRIM(
                                            "VALOR TRANSAÇÃO"
                                        ),
                                        'R$',
                                        ''
                                    ),
                                    ' ',
                                    ''
                                ),
                                '.',
                                ''
                            ),
                            ',',
                            '.'
                        )
                        ELSE REPLACE(
                            REPLACE(
                                TRIM(
                                    "VALOR TRANSAÇÃO"
                                ),
                                'R$',
                                ''
                            ),
                            ' ',
                            ''
                        )
                    END
                    AS DECIMAL(18,2)
                ) AS VALOR_NUM,

                TRY_STRPTIME(
                    NULLIF(
                        TRIM(
                            "DATA TRANSAÇÃO"
                        ),
                        ''
                    ),
                    '%d/%m/%Y'
                )::DATE AS DATA_DT

            FROM numerado
        )

        SELECT
            *,

            CAST(
                ROUND(
                    VALOR_NUM * 100,
                    0
                )
                AS BIGINT
            ) AS VALOR_CENTAVOS,

            EXTRACT(
                YEAR FROM DATA_DT
            )::INTEGER
                AS ANO_TRANSACAO,

            EXTRACT(
                MONTH FROM DATA_DT
            )::INTEGER
                AS MES_TRANSACAO,

            DAYOFWEEK(
                DATA_DT
            )::INTEGER
                AS DIA_SEMANA_DUCKDB,

            CASE DAYOFWEEK(DATA_DT)
                WHEN 0 THEN 'domingo'
                WHEN 1 THEN 'segunda-feira'
                WHEN 2 THEN 'terça-feira'
                WHEN 3 THEN 'quarta-feira'
                WHEN 4 THEN 'quinta-feira'
                WHEN 5 THEN 'sexta-feira'
                WHEN 6 THEN 'sábado'
                ELSE NULL
            END AS DIA_SEMANA_TXT,

            (
                "TRANSAÇÃO"
                = '{compras_nacional_sql}'
            ) AS EH_COMPRA_NACIONAL,

            (
                "TRANSAÇÃO"
                = '{compras_internacional_sql}'
            ) AS EH_COMPRA_INTERNACIONAL,

            (
                "TRANSAÇÃO"
                = '{compra_parcelada_sql}'
            ) AS EH_COMPRA_PARCELADA,

            (
                "TRANSAÇÃO" IN (
                    '{compras_nacional_sql}',
                    '{compras_internacional_sql}',
                    '{compra_parcelada_sql}'
                )
            ) AS EH_COMPRA_EFETIVA,

            (
                "TRANSAÇÃO"
                IN ({saques_sql})
            ) AS EH_SAQUE_EFETIVO,

            (
                "TRANSAÇÃO"
                IN ({ajustes_sql})
            ) AS EH_AJUSTE_CONTESTACAO,

            (
                LOWER(
                    COALESCE(
                        "TRANSAÇÃO",
                        ''
                    )
                ) LIKE '%sigilo%'
                OR LOWER(
                    COALESCE(
                        "NOME PORTADOR",
                        ''
                    )
                ) LIKE '%sigilo%'
                OR LOWER(
                    COALESCE(
                        "NOME FAVORECIDO",
                        ''
                    )
                ) LIKE '%sigilo%'
            ) AS EH_SIGILOSO,

            (
                FAVORECIDO_ID IS NOT NULL
                AND LENGTH(FAVORECIDO_ID) >= 3
            ) AS FAVORECIDO_IDENTIFICADO

        FROM preparado
    )
    TO '{stg_path_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """

    inicio = time.time()

    con.execute(stg_sql)

    duracao = (
        time.time()
        - inicio
    )

    gravar_checkpoint(
        'preparacao',
        prep_fingerprint
    )

    salvar_json(
        {
            'fingerprint':
                prep_fingerprint,
            'versao_preparacao':
                VERSAO_PREPARACAO,
            'hash_input':
                HASH_INPUT,
            'arquivo_saida':
                str(STG_PARQUET),
            'duracao_segundos':
                round(duracao, 2),
            'data_hora':
                datetime.now().isoformat(),
        },
        PREP_META
    )

    print(
        f'✅ Preparação concluída '
        f'em {duracao:.1f}s.'
    )

stg_path_sql = (
    str(STG_PARQUET)
    .replace("'", "''")
)

con.execute(f"""
CREATE OR REPLACE VIEW stg_cpgf_transacoes AS

SELECT *
FROM read_parquet(
    '{stg_path_sql}'
)
""")

resumo_stg = con.execute("""
SELECT
    COUNT(*) AS n,

    COUNT_IF(
        VALOR_NUM IS NULL
        AND TRIM(
            COALESCE(
                "VALOR TRANSAÇÃO",
                ''
            )
        ) <> ''
    ) AS valor_nao_parseado,

    COUNT_IF(
        DATA_DT IS NULL
        AND TRIM(
            COALESCE(
                "DATA TRANSAÇÃO",
                ''
            )
        ) <> ''
    ) AS data_nao_parseada,

    COUNT_IF(
        EH_COMPRA_NACIONAL
    ) AS compras_nacionais,

    COUNT_IF(
        EH_COMPRA_INTERNACIONAL
    ) AS compras_internacionais,

    COUNT_IF(
        EH_COMPRA_PARCELADA
    ) AS compras_parceladas,

    COUNT_IF(
        EH_SAQUE_EFETIVO
    ) AS saques_efetivos,

    COUNT_IF(
        EH_AJUSTE_CONTESTACAO
    ) AS ajustes_contestacao,

    COUNT_IF(
        EH_SIGILOSO
    ) AS sigilosos

FROM stg_cpgf_transacoes
""").df()

display(resumo_stg)

## 7️⃣ Diagnóstico exploratório do CPGF

Antes das trilhas, é importante compreender os processos geradores presentes na base.

Serão produzidos:

- distribuição dos tipos de transação;
- série histórica por ano da data da transação;
- cobertura de fornecedor identificado;
- estatísticas dos valores;
- valores mais frequentes;
- diagnóstico de arredondamento.

Esses resultados ajudam a evitar interpretações inadequadas, especialmente na Lei de Benford.

In [ ]:
tipos_transacao_df = con.execute("""
SELECT
    "TRANSAÇÃO" AS TRANSACAO,
    COUNT(*) AS N,
    SUM(
        COALESCE(
            VALOR_NUM,
            0
        )
    ) AS VALOR_TOTAL

FROM stg_cpgf_transacoes

GROUP BY 1

ORDER BY
    N DESC
""").df()

salvar_csv(
    tipos_transacao_df,
    DIAGNOSTICO_DIR
    / '01_tipos_transacao.csv'
)

display(tipos_transacao_df)


diagnostico_anual_df = con.execute("""
SELECT
    ANO_TRANSACAO,

    COUNT(*) AS N,

    COUNT_IF(
        EH_COMPRA_NACIONAL
    ) AS N_COMPRAS_NACIONAIS,

    COUNT_IF(
        EH_SAQUE_EFETIVO
    ) AS N_SAQUES,

    SUM(
        COALESCE(
            VALOR_NUM,
            0
        )
    ) AS VALOR_TOTAL,

    SUM(
        CASE
            WHEN EH_COMPRA_NACIONAL
            THEN COALESCE(
                VALOR_NUM,
                0
            )
            ELSE 0
        END
    ) AS VALOR_COMPRAS_NACIONAIS,

    SUM(
        CASE
            WHEN EH_SAQUE_EFETIVO
            THEN COALESCE(
                VALOR_NUM,
                0
            )
            ELSE 0
        END
    ) AS VALOR_SAQUES

FROM stg_cpgf_transacoes

WHERE
    ANO_TRANSACAO
    IS NOT NULL

GROUP BY 1

ORDER BY 1
""").df()

salvar_csv(
    diagnostico_anual_df,
    DIAGNOSTICO_DIR
    / '02_diagnostico_anual.csv'
)

display(diagnostico_anual_df)


cobertura_fornecedor_df = con.execute("""
SELECT
    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
    ) AS N_COMPRAS,

    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
        AND FAVORECIDO_IDENTIFICADO
    ) AS N_COMPRAS_IDENTIFICADAS,

    SUM(
        CASE
            WHEN
                EH_COMPRA_NACIONAL
                AND VALOR_NUM > 0
            THEN VALOR_NUM
            ELSE 0
        END
    ) AS VALOR_COMPRAS,

    SUM(
        CASE
            WHEN
                EH_COMPRA_NACIONAL
                AND VALOR_NUM > 0
                AND FAVORECIDO_IDENTIFICADO
            THEN VALOR_NUM
            ELSE 0
        END
    ) AS VALOR_IDENTIFICADO

FROM stg_cpgf_transacoes
""").df()

cobertura_fornecedor_df[
    'COBERTURA_N'
] = (
    cobertura_fornecedor_df[
        'N_COMPRAS_IDENTIFICADAS'
    ]
    / cobertura_fornecedor_df[
        'N_COMPRAS'
    ]
)

cobertura_fornecedor_df[
    'COBERTURA_VALOR'
] = (
    cobertura_fornecedor_df[
        'VALOR_IDENTIFICADO'
    ]
    / cobertura_fornecedor_df[
        'VALOR_COMPRAS'
    ]
)

salvar_csv(
    cobertura_fornecedor_df,
    DIAGNOSTICO_DIR
    / '03_cobertura_fornecedor.csv'
)

display(cobertura_fornecedor_df)


estatisticas_valores_df = con.execute("""
SELECT
    COUNT(*) AS N,

    COUNT(
        DISTINCT VALOR_CENTAVOS
    ) AS VALORES_UNICOS,

    MIN(
        VALOR_NUM
    ) AS MINIMO,

    MEDIAN(
        VALOR_NUM
    ) AS MEDIANA,

    AVG(
        VALOR_NUM
    ) AS MEDIA,

    MAX(
        VALOR_NUM
    ) AS MAXIMO,

    STDDEV_POP(
        VALOR_NUM
    ) AS DESVIO_PADRAO,

    SKEWNESS(
        VALOR_NUM
    ) AS ASSIMETRIA,

    COUNT_IF(
        VALOR_CENTAVOS % 100 = 0
    ) * 1.0 / COUNT(*)
        AS PCT_VALOR_INTEIRO,

    COUNT_IF(
        VALOR_CENTAVOS % 1000 = 0
    ) * 1.0 / COUNT(*)
        AS PCT_MULTIPLO_10,

    COUNT_IF(
        VALOR_CENTAVOS % 10000 = 0
    ) * 1.0 / COUNT(*)
        AS PCT_MULTIPLO_100

FROM stg_cpgf_transacoes

WHERE
    EH_COMPRA_NACIONAL
    AND VALOR_NUM > 0
""").df()

salvar_csv(
    estatisticas_valores_df,
    DIAGNOSTICO_DIR
    / '04_estatisticas_compras_nacionais.csv'
)

display(estatisticas_valores_df)


valores_frequentes_df = con.execute("""
SELECT
    VALOR_NUM,
    VALOR_CENTAVOS,
    COUNT(*) AS FREQUENCIA,
    SUM(
        VALOR_NUM
    ) AS VALOR_TOTAL

FROM stg_cpgf_transacoes

WHERE
    EH_COMPRA_NACIONAL
    AND VALOR_NUM > 0

GROUP BY
    1,
    2

ORDER BY
    FREQUENCIA DESC,
    VALOR_NUM DESC

LIMIT 200
""").df()

salvar_csv(
    valores_frequentes_df,
    DIAGNOSTICO_DIR
    / '05_valores_mais_frequentes.csv'
)

display(
    valores_frequentes_df.head(30)
)

print(
    '✅ Diagnóstico exploratório '
    'concluído.'
)

## 8️⃣ Testes sintéticos das regras

Antes de aplicar as trilhas à base real, executamos pequenos testes de comportamento esperado.

Eles não substituem a futura suíte `pytest`, mas funcionam como uma barreira de segurança dentro do notebook.

In [ ]:
def classificar_mad(mad, tipo):
    if tipo == 'D1':
        if mad <= 0.006:
            return 'Conformidade próxima'
        if mad <= 0.012:
            return 'Conformidade aceitável'
        if mad <= 0.015:
            return (
                'Conformidade '
                'marginalmente aceitável'
            )
        return 'Não conformidade'

    if tipo == 'D12':
        if mad <= 0.0012:
            return 'Conformidade próxima'
        if mad <= 0.0018:
            return 'Conformidade aceitável'
        if mad <= 0.0022:
            return (
                'Conformidade '
                'marginalmente aceitável'
            )
        return 'Não conformidade'

    raise ValueError(
        'Tipo deve ser D1 ou D12.'
    )


def benford_prob_d1(d):
    return math.log10(
        1 + 1 / d
    )


def benford_prob_d12(d):
    return math.log10(
        1 + 1 / d
    )


def cv_pop(valores):
    valores = np.asarray(
        valores,
        dtype=float
    )

    media = valores.mean()

    if media == 0:
        return np.nan

    return (
        valores.std(ddof=0)
        / media
    )


TESTES = []


def check(nome, condicao):
    if not bool(condicao):
        raise AssertionError(
            f'❌ Falhou: {nome}'
        )

    TESTES.append(nome)


check(
    'P(D1=1)',
    abs(
        benford_prob_d1(1)
        - 0.30103
    ) < 1e-5
)

check(
    'P(D12=10)',
    abs(
        benford_prob_d12(10)
        - 0.04139
    ) < 1e-5
)

check(
    'MAD D1 próximo',
    classificar_mad(
        0.005,
        'D1'
    ) == 'Conformidade próxima'
)

check(
    'MAD D1 aceitável',
    classificar_mad(
        0.010,
        'D1'
    ) == 'Conformidade aceitável'
)

check(
    'MAD D1 marginal',
    classificar_mad(
        0.013,
        'D1'
    ) == (
        'Conformidade '
        'marginalmente aceitável'
    )
)

check(
    'MAD D1 não conformidade',
    classificar_mad(
        0.016,
        'D1'
    ) == 'Não conformidade'
)

check(
    'MAD D12 próximo',
    classificar_mad(
        0.0010,
        'D12'
    ) == 'Conformidade próxima'
)

check(
    'MAD D12 não conformidade',
    classificar_mad(
        0.0023,
        'D12'
    ) == 'Não conformidade'
)

check(
    'CV 8% aproximadamente',
    (
        0.05
        < cv_pop(
            [100, 108, 92, 101, 99]
        )
        < 0.10
    )
)

check(
    'CV 0 em valores idênticos',
    cv_pop(
        [100, 100, 100, 100, 100]
    ) == 0
)

check(
    'R$ 100,10 em centavos',
    int(
        round(
            100.10 * 100
        )
    ) == 10010
)

check(
    'Diferença de um centavo',
    int(
        round(
            100.01 * 100
        )
    ) != int(
        round(
            100.00 * 100
        )
    )
)

print(
    f'🧪 Testes executados: '
    f'{len(TESTES)}'
)

print(
    f'✅ Aprovados: '
    f'{len(TESTES)}'
)

print(
    '❌ Falhas: 0'
)

## 9️⃣ T01 — Despesa realizada em final de semana

### Regra

Sinalizar uma compra efetiva quando `DATA TRANSAÇÃO` corresponder a sábado ou domingo.

### Interpretação

A ocorrência **não significa irregularidade**. A documentação administrativa prevê que despesas em finais de semana podem existir quando devidamente justificadas.

O CSV público não contém essa justificativa.

### Novidade da versão 1.2

Além das transações individuais, calcula-se a recorrência anual por `UG × portador`: quantidade de compras em finais de semana, número de dias distintos e proporção em relação a todas as compras do portador. Essa camada é **contextual** e não transforma frequência em irregularidade.

In [ ]:
T01_PARQUET = (
    T01_DIR
    / 't01_fim_semana.parquet'
)

T01_CSV = (
    T01_DIR
    / 't01_fim_semana.csv'
)

t01_path_sql = (
    str(T01_PARQUET)
    .replace("'", "''")
)

sql_t01 = f"""
COPY (
    SELECT
        'T01_' || MD5(
            'T01|'
            || '{VERSAO_REGRAS}'
            || '|'
            || ID_TRANSACAO
        ) AS ID_SINAL,

        ID_TRANSACAO,
        UG_ID,

        "NOME UNIDADE GESTORA"
            AS NOME_UG,

        PORTADOR_ID,

        "NOME PORTADOR"
            AS NOME_PORTADOR,

        FAVORECIDO_ID,

        "NOME FAVORECIDO"
            AS NOME_FAVORECIDO,

        DATA_DT,
        DIA_SEMANA_TXT,
        VALOR_NUM,
        VALOR_CENTAVOS,

        "TRANSAÇÃO"
            AS TRANSACAO,

        COMPETENCIA_ARQUIVO,
        ARQUIVO_ORIGEM,

        'ATENCAO'
            AS NIVEL_TRIAGEM,

        (
            'Compra realizada em final de semana; '
            'verificar justificativa documental.'
        ) AS EVIDENCIA,

        (
            'A base pública não contém '
            'a justificativa da despesa.'
        ) AS LIMITACAO

    FROM stg_cpgf_transacoes

    WHERE
        EH_COMPRA_EFETIVA
        AND NOT EH_AJUSTE_CONTESTACAO
        AND DATA_DT IS NOT NULL
        AND DAYOFWEEK(
            DATA_DT
        ) IN (0, 6)
)
TO '{t01_path_sql}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

con.execute(
    sql_t01
)

t01_resumo_df = con.execute(
    f"""
    SELECT
        EXTRACT(
            YEAR FROM DATA_DT
        )::INTEGER AS ANO,

        COUNT(*)
            AS N_SINAIS,

        SUM(
            VALOR_NUM
        ) AS VALOR_TOTAL

    FROM read_parquet(
        '{t01_path_sql}'
    )

    GROUP BY 1

    ORDER BY 1
    """
).df()

t01_resumo_df['STATUS_PERIODO'] = np.where(
    t01_resumo_df['ANO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

salvar_csv(
    t01_resumo_df,
    T01_DIR
    / 't01_resumo_anual.csv'
)

if EXPORTAR_CSVS_COMPLETOS:
    t01_csv_sql = (
        str(T01_CSV)
        .replace("'", "''")
    )

    con.execute(
        f"""
        COPY (
            SELECT *
            FROM read_parquet(
                '{t01_path_sql}'
            )
        )
        TO '{t01_csv_sql}'
        (
            HEADER,
            DELIMITER ';'
        )
        """
    )

display(
    t01_resumo_df
)

print(
    '✅ T01 concluída.'
)


# ============================================================
# 🔁 T01 — RECORRÊNCIA POR PORTADOR / UG / ANO
# ============================================================

T01_RECORRENCIA = T01_DIR / 't01_recorrencia_portador_ug_ano.parquet'
t01_rec_sql = str(T01_RECORRENCIA).replace("'", "''")

con.execute(
    f"""
    COPY (
        WITH compras AS (
            SELECT
                UG_ID,
                PORTADOR_ID,
                ANO_TRANSACAO,
                COUNT(*) AS N_COMPRAS,
                COUNT_IF(DAYOFWEEK(DATA_DT) IN (0,6)) AS N_FIM_SEMANA,
                COUNT(DISTINCT CASE WHEN DAYOFWEEK(DATA_DT) IN (0,6) THEN DATA_DT END) AS N_DIAS_FIM_SEMANA,
                SUM(CASE WHEN DAYOFWEEK(DATA_DT) IN (0,6) THEN VALOR_CENTAVOS ELSE 0 END) AS VALOR_FIM_SEMANA_CENTAVOS
            FROM stg_cpgf_transacoes
            WHERE EH_COMPRA_NACIONAL
              AND DATA_DT IS NOT NULL
              AND PORTADOR_ID IS NOT NULL
              AND UG_ID IS NOT NULL
            GROUP BY 1,2,3
        )
        SELECT
            *,
            N_FIM_SEMANA * 1.0 / NULLIF(N_COMPRAS,0) AS SHARE_FIM_SEMANA,
            PERCENT_RANK() OVER (
                PARTITION BY ANO_TRANSACAO
                ORDER BY N_FIM_SEMANA
            ) AS PERCENT_RANK_N_FIM_SEMANA
        FROM compras
        WHERE N_FIM_SEMANA > 0
    )
    TO '{t01_rec_sql}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

t01_recorrencia_resumo_df = con.execute(
    f"""
    SELECT
        ANO_TRANSACAO,
        COUNT(*) AS N_PORTADORES_COM_FIM_SEMANA,
        MEDIAN(N_FIM_SEMANA) AS MEDIANA_OCORRENCIAS,
        MAX(N_FIM_SEMANA) AS MAX_OCORRENCIAS,
        AVG(SHARE_FIM_SEMANA) AS SHARE_MEDIO
    FROM read_parquet('{t01_rec_sql}')
    GROUP BY 1
    ORDER BY 1
    """
).df()

t01_recorrencia_resumo_df['STATUS_PERIODO'] = np.where(
    t01_recorrencia_resumo_df['ANO_TRANSACAO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

salvar_csv(
    t01_recorrencia_resumo_df,
    T01_DIR / 't01_recorrencia_resumo_anual.csv'
)
display(t01_recorrencia_resumo_df)
print('✅ T01 — recorrência contextual calculada.')

## 🔟 T02 — Transação classificada como compra parcelada

A regra utiliza **correspondência exata** do código:

`CPP LOJISTA TRF P/FATURA - REAL`

O sinal é operacional e exige conferência da fatura/documentação. Não é utilizado casamento textual aproximado.

In [ ]:
T02_PARQUET = (
    T02_DIR
    / 't02_compra_parcelada.parquet'
)

t02_path_sql = (
    str(T02_PARQUET)
    .replace("'", "''")
)

codigo_t02_sql = (
    CODIGO_COMPRA_PARCELADA
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        SELECT
            'T02_' || MD5(
                'T02|'
                || '{VERSAO_REGRAS}'
                || '|'
                || ID_TRANSACAO
            ) AS ID_SINAL,

            ID_TRANSACAO,
            UG_ID,

            "NOME UNIDADE GESTORA"
                AS NOME_UG,

            PORTADOR_ID,

            "NOME PORTADOR"
                AS NOME_PORTADOR,

            FAVORECIDO_ID,

            "NOME FAVORECIDO"
                AS NOME_FAVORECIDO,

            DATA_DT,
            VALOR_NUM,
            VALOR_CENTAVOS,

            "TRANSAÇÃO"
                AS TRANSACAO,

            COMPETENCIA_ARQUIVO,
            ARQUIVO_ORIGEM,

            'ATENCAO'
                AS NIVEL_TRIAGEM,

            (
                'Código operacional classificado '
                'como compra parcelada.'
            ) AS EVIDENCIA,

            (
                'A classificação deve ser conferida '
                'com a fatura e os documentos da despesa.'
            ) AS LIMITACAO

        FROM stg_cpgf_transacoes

        WHERE
            "TRANSAÇÃO"
            = '{codigo_t02_sql}'
    )
    TO '{t02_path_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

t02_resumo_df = con.execute(
    f"""
    SELECT
        COUNT(*)
            AS N_SINAIS,

        SUM(
            COALESCE(
                VALOR_NUM,
                0
            )
        ) AS VALOR_TOTAL,

        MIN(
            DATA_DT
        ) AS PRIMEIRA_DATA,

        MAX(
            DATA_DT
        ) AS ULTIMA_DATA

    FROM read_parquet(
        '{t02_path_sql}'
    )
    """
).df()

salvar_csv(
    t02_resumo_df,
    T02_DIR
    / 't02_resumo.csv'
)

display(
    t02_resumo_df
)

print(
    '✅ T02 concluída.'
)

## 1️⃣1️⃣ T03 — Repetição exata de transações

### T03-A — repetição comportamental

A chave permanece:

`UG + portador + favorecido + data + valor em centavos + TRANSAÇÃO`

- `N = 2` → atenção;
- `N ≥ 3` → reforçado.

A inclusão do tipo de transação evita agrupar processos operacionais distintos apenas porque compartilham data e valor.

### T03-B — repetição integral **observável**

A V1.1 mostrou que a comparação de linhas integrais em toda a base pode ser distorcida por registros sigilosos ou incompletos: campos diferentes na origem podem aparecer publicamente com o mesmo conteúdo mascarado.

Por isso, a V1.2 limita T03-B a registros efetivamente comparáveis:

- compra efetiva;
- não ajuste/contestação;
- não sigilosa;
- valor positivo;
- data válida;
- UG válida;
- portador identificado;
- favorecido identificado.

A assinatura utiliza os **15 campos de negócio originalmente disponibilizados pelo Portal**. A saída continua sendo um diagnóstico de repetição do conteúdo observável, e não uma conclusão de pagamento duplicado.

In [ ]:
T03_GRUPOS = T03_DIR / 't03_repeticao_exata_grupos.parquet'
T03_PONTE = T03_DIR / 't03_repeticao_exata_transacoes.parquet'

T03_INTEGRAL = T03_DIR / 't03_repeticao_integral_observavel_grupos.parquet'
T03_INTEGRAL_PONTE = T03_DIR / 't03_repeticao_integral_observavel_transacoes.parquet'

for p in [
    T03_GRUPOS,
    T03_PONTE,
    T03_INTEGRAL,
    T03_INTEGRAL_PONTE,
]:
    p.parent.mkdir(parents=True, exist_ok=True)

t03_grupos_sql = str(T03_GRUPOS).replace("'", "''")
t03_ponte_sql = str(T03_PONTE).replace("'", "''")
t03_integral_sql = str(T03_INTEGRAL).replace("'", "''")
t03_integral_ponte_sql = str(T03_INTEGRAL_PONTE).replace("'", "''")

min_t03 = CONFIG['T03']['min_ocorrencias']
ref_t03 = CONFIG['T03']['reforcado_ocorrencias']


# ============================================================
# T03-A — REPETIÇÃO COMPORTAMENTAL
# ============================================================

con.execute(f"""
COPY (
    SELECT
        'T03_' || MD5(
            'T03|' || '{VERSAO_REGRAS}' || '|' || UG_ID || '|' ||
            PORTADOR_ID || '|' || FAVORECIDO_ID || '|' ||
            CAST(DATA_DT AS VARCHAR) || '|' ||
            CAST(VALOR_CENTAVOS AS VARCHAR) || '|' ||
            "TRANSAÇÃO"
        ) AS ID_SINAL,

        UG_ID,
        PORTADOR_ID,
        FAVORECIDO_ID,
        DATA_DT,
        "TRANSAÇÃO" AS TRANSACAO,
        VALOR_NUM AS VALOR_UNITARIO,
        VALOR_CENTAVOS,

        COUNT(*) AS N_TRANSACOES,
        SUM(VALOR_CENTAVOS) AS VALOR_TOTAL_CENTAVOS,

        CASE
            WHEN COUNT(*) >= {ref_t03} THEN 'REFORCADO'
            ELSE 'ATENCAO'
        END AS NIVEL_TRIAGEM

    FROM stg_cpgf_transacoes

    WHERE EH_COMPRA_EFETIVA
      AND NOT EH_AJUSTE_CONTESTACAO
      AND VALOR_CENTAVOS > 0
      AND DATA_DT IS NOT NULL
      AND UG_ID IS NOT NULL
      AND PORTADOR_ID IS NOT NULL
      AND FAVORECIDO_IDENTIFICADO

    GROUP BY
        UG_ID,
        PORTADOR_ID,
        FAVORECIDO_ID,
        DATA_DT,
        "TRANSAÇÃO",
        VALOR_NUM,
        VALOR_CENTAVOS

    HAVING COUNT(*) >= {min_t03}
)
TO '{t03_grupos_sql}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")


con.execute(f"""
COPY (
    SELECT
        g.ID_SINAL,
        t.ID_TRANSACAO,
        t.UG_ID,
        t.PORTADOR_ID,
        t.FAVORECIDO_ID,
        t.DATA_DT,
        t.VALOR_NUM,
        t.VALOR_CENTAVOS,
        t."TRANSAÇÃO" AS TRANSACAO,
        t.COMPETENCIA_ARQUIVO,
        t.ARQUIVO_ORIGEM

    FROM read_parquet('{t03_grupos_sql}') g

    JOIN stg_cpgf_transacoes t
      ON t.UG_ID = g.UG_ID
     AND t.PORTADOR_ID = g.PORTADOR_ID
     AND t.FAVORECIDO_ID = g.FAVORECIDO_ID
     AND t.DATA_DT = g.DATA_DT
     AND t.VALOR_CENTAVOS = g.VALOR_CENTAVOS
     AND t."TRANSAÇÃO" = g.TRANSACAO

    WHERE t.EH_COMPRA_EFETIVA
      AND NOT t.EH_AJUSTE_CONTESTACAO
)
TO '{t03_ponte_sql}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")


# ============================================================
# T03-B — REPETIÇÃO INTEGRAL OBSERVÁVEL
# ============================================================

con.execute("""
CREATE OR REPLACE TEMP VIEW t03_integral_base_observavel AS

SELECT
    ID_TRANSACAO,
    UG_ID,
    PORTADOR_ID,
    FAVORECIDO_ID,
    DATA_DT,
    VALOR_NUM,
    VALOR_CENTAVOS,
    "TRANSAÇÃO" AS TRANSACAO,

    MD5(
        CONCAT_WS(
            '|',
            COALESCE("CÓDIGO ÓRGÃO SUPERIOR",''),
            COALESCE("NOME ÓRGÃO SUPERIOR",''),
            COALESCE("CÓDIGO ÓRGÃO",''),
            COALESCE("NOME ÓRGÃO",''),
            COALESCE("CÓDIGO UNIDADE GESTORA",''),
            COALESCE("NOME UNIDADE GESTORA",''),
            COALESCE("ANO EXTRATO",''),
            COALESCE("MÊS EXTRATO",''),
            COALESCE("CPF PORTADOR",''),
            COALESCE("NOME PORTADOR",''),
            COALESCE("CNPJ OU CPF FAVORECIDO",''),
            COALESCE("NOME FAVORECIDO",''),
            COALESCE("TRANSAÇÃO",''),
            COALESCE("DATA TRANSAÇÃO",''),
            COALESCE("VALOR TRANSAÇÃO",'')
        )
    ) AS HASH_REGISTRO_NEGOCIO

FROM stg_cpgf_transacoes

WHERE EH_COMPRA_EFETIVA
  AND NOT EH_AJUSTE_CONTESTACAO
  AND NOT EH_SIGILOSO
  AND VALOR_CENTAVOS > 0
  AND DATA_DT IS NOT NULL
  AND UG_ID IS NOT NULL
  AND PORTADOR_ID IS NOT NULL
  AND FAVORECIDO_IDENTIFICADO
""")


con.execute(f"""
COPY (
    SELECT
        HASH_REGISTRO_NEGOCIO,

        MIN(ID_TRANSACAO) AS ID_TRANSACAO_EXEMPLO,

        ANY_VALUE(UG_ID) AS UG_ID,
        ANY_VALUE(PORTADOR_ID) AS PORTADOR_ID,
        ANY_VALUE(FAVORECIDO_ID) AS FAVORECIDO_ID,
        ANY_VALUE(DATA_DT) AS DATA_DT,
        ANY_VALUE(TRANSACAO) AS TRANSACAO,
        ANY_VALUE(VALOR_NUM) AS VALOR_NUM,
        ANY_VALUE(VALOR_CENTAVOS) AS VALOR_CENTAVOS,

        COUNT(*) AS N_REPETICOES

    FROM t03_integral_base_observavel

    GROUP BY HASH_REGISTRO_NEGOCIO

    HAVING COUNT(*) >= 2
)
TO '{t03_integral_sql}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")


con.execute(f"""
COPY (
    SELECT
        'T03B_' || MD5(
            'T03B|' || '{VERSAO_REGRAS}' || '|' || g.HASH_REGISTRO_NEGOCIO
        ) AS ID_GRUPO_INTEGRAL,

        g.HASH_REGISTRO_NEGOCIO,
        b.ID_TRANSACAO,
        b.UG_ID,
        b.PORTADOR_ID,
        b.FAVORECIDO_ID,
        b.DATA_DT,
        b.TRANSACAO,
        b.VALOR_NUM,
        b.VALOR_CENTAVOS

    FROM read_parquet('{t03_integral_sql}') g

    JOIN t03_integral_base_observavel b
      ON b.HASH_REGISTRO_NEGOCIO = g.HASH_REGISTRO_NEGOCIO
)
TO '{t03_integral_ponte_sql}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")


# ============================================================
# RESUMOS
# ============================================================

t03_resumo_df = con.execute(f"""
SELECT
    NIVEL_TRIAGEM,
    COUNT(*) AS N_GRUPOS,
    SUM(N_TRANSACOES) AS TRANSACOES_ENVOLVIDAS,
    SUM(VALOR_TOTAL_CENTAVOS) / 100.0 AS VALOR_TOTAL

FROM read_parquet('{t03_grupos_sql}')

GROUP BY 1
ORDER BY 1
""").df()


n_elegiveis_integral = int(
    con.execute(
        "SELECT COUNT(*) FROM t03_integral_base_observavel"
    ).fetchone()[0]
)

t03_integral_resumo_df = con.execute(f"""
SELECT
    COUNT(*) AS N_GRUPOS_INTEGRAIS_OBSERVAVEIS,
    SUM(N_REPETICOES) AS LINHAS_ENVOLVIDAS,
    MAX(N_REPETICOES) AS MAX_REPETICOES

FROM read_parquet('{t03_integral_sql}')
""").df()

t03_integral_resumo_df['N_REGISTROS_ELEGIVEIS_T03B'] = n_elegiveis_integral

t03_integral_resumo_df['SHARE_ELEGIVEIS_ENVOLVIDOS'] = (
    t03_integral_resumo_df['LINHAS_ENVOLVIDAS']
    / n_elegiveis_integral
    if n_elegiveis_integral
    else np.nan
)

salvar_csv(
    t03_resumo_df,
    T03_DIR / 't03_resumo.csv'
)

salvar_csv(
    t03_integral_resumo_df,
    T03_DIR / 't03_repeticao_integral_observavel_resumo.csv'
)

display(t03_resumo_df)
display(t03_integral_resumo_df)

print('✅ T03-A concluída.')
print('✅ T03-B concluída apenas sobre registros integralmente observáveis.')

## 1️⃣2️⃣ T04 — Repetição multiportador

### Chave

`UG + fornecedor + data + valor`

O sinal ocorre quando a chave possui **pelo menos dois portadores distintos**.

Essa trilha é relevante porque permite observar comportamentos conjuntos dentro da mesma UG. Entretanto, a base pública não informa o objeto adquirido; por isso, a ocorrência não permite afirmar que houve divisão de uma mesma despesa.

### Níveis da versão 1.2

- 2 portadores → `ATENCAO`;
- 3–4 portadores → `REFORCADO`;
- 5 ou mais portadores → `MUITO_ELEVADO`.

As faixas são analíticas e não representam gradação jurídica de irregularidade.

In [ ]:
T04_GRUPOS = (
    T04_DIR
    / 't04_multiportador_grupos.parquet'
)

T04_PONTE = (
    T04_DIR
    / 't04_multiportador_transacoes.parquet'
)

t04_grupos_sql = (
    str(T04_GRUPOS)
    .replace("'", "''")
)

t04_ponte_sql = (
    str(T04_PONTE)
    .replace("'", "''")
)

min_port_t04 = (
    CONFIG['T04']
    ['min_portadores']
)

con.execute(
    f"""
    COPY (
        SELECT
            'T04_' || MD5(
                'T04|'
                || '{VERSAO_REGRAS}'
                || '|'
                || UG_ID
                || '|'
                || FAVORECIDO_ID
                || '|'
                || CAST(
                    DATA_DT
                    AS VARCHAR
                )
                || '|'
                || CAST(
                    VALOR_CENTAVOS
                    AS VARCHAR
                )
            ) AS ID_SINAL,

            UG_ID,
            FAVORECIDO_ID,
            DATA_DT,

            VALOR_NUM
                AS VALOR_UNITARIO,

            VALOR_CENTAVOS,

            COUNT(*)
                AS N_TRANSACOES,

            COUNT(
                DISTINCT PORTADOR_ID
            ) AS N_PORTADORES,

            SUM(
                VALOR_CENTAVOS
            ) AS VALOR_TOTAL_CENTAVOS,

            CASE
                WHEN COUNT(DISTINCT PORTADOR_ID) >= {CONFIG['T04']['muito_elevado_portadores']} THEN 'MUITO_ELEVADO'
                WHEN COUNT(DISTINCT PORTADOR_ID) >= {CONFIG['T04']['reforcado_portadores']} THEN 'REFORCADO'
                ELSE 'ATENCAO'
            END AS NIVEL_TRIAGEM

        FROM stg_cpgf_transacoes

        WHERE
            EH_COMPRA_EFETIVA
            AND NOT EH_AJUSTE_CONTESTACAO
            AND VALOR_CENTAVOS > 0
            AND DATA_DT IS NOT NULL
            AND UG_ID IS NOT NULL
            AND PORTADOR_ID IS NOT NULL
            AND FAVORECIDO_IDENTIFICADO

        GROUP BY
            UG_ID,
            FAVORECIDO_ID,
            DATA_DT,
            VALOR_NUM,
            VALOR_CENTAVOS

        HAVING
            COUNT(*) >= 2
            AND COUNT(
                DISTINCT PORTADOR_ID
            ) >= {min_port_t04}
    )
    TO '{t04_grupos_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

con.execute(
    f"""
    COPY (
        SELECT
            g.ID_SINAL,
            t.ID_TRANSACAO,
            t.UG_ID,
            t.PORTADOR_ID,
            t.FAVORECIDO_ID,
            t.DATA_DT,
            t.VALOR_NUM,
            t.VALOR_CENTAVOS,

            t."TRANSAÇÃO"
                AS TRANSACAO,

            t.COMPETENCIA_ARQUIVO,
            t.ARQUIVO_ORIGEM

        FROM read_parquet(
            '{t04_grupos_sql}'
        ) g

        JOIN stg_cpgf_transacoes t
          ON t.UG_ID = g.UG_ID
         AND t.FAVORECIDO_ID = g.FAVORECIDO_ID
         AND t.DATA_DT = g.DATA_DT
         AND t.VALOR_CENTAVOS = g.VALOR_CENTAVOS

        WHERE
            t.EH_COMPRA_EFETIVA
            AND NOT t.EH_AJUSTE_CONTESTACAO
            AND t.PORTADOR_ID IS NOT NULL
    )
    TO '{t04_ponte_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

t04_resumo_df = con.execute(
    f"""
    SELECT
        NIVEL_TRIAGEM,
        COUNT(*) AS N_GRUPOS,
        SUM(N_TRANSACOES) AS TRANSACOES_ENVOLVIDAS,
        SUM(VALOR_TOTAL_CENTAVOS) / 100.0 AS VALOR_TOTAL,
        MAX(N_PORTADORES) AS MAX_PORTADORES
    FROM read_parquet('{t04_grupos_sql}')
    GROUP BY 1
    ORDER BY 1
    """
).df()

salvar_csv(
    t04_resumo_df,
    T04_DIR
    / 't04_resumo.csv'
)

display(
    t04_resumo_df
)

print(
    '✅ T04 concluída.'
)

## 1️⃣3️⃣ T05 — Recorrência de aquisições: métricas das janelas

T05 é a principal trilha de calibração comportamental.

Para cada combinação:

`UG + fornecedor + ano da transação`

o notebook cria janelas móveis iniciadas em cada data com compra e calcula:

- quantidade de transações;
- portadores distintos;
- média;
- desvio-padrão;
- coeficiente de variação;
- mínimo;
- máximo;
- valor acumulado.

As métricas são calculadas uma vez para cada janela de 15, 30, 45 e 60 dias.

### Novas métricas da versão 1.2

Além do CV, cada janela calcula **mediana**, **IQR** e a proporção de compras situada dentro de ±20% da mediana. Essa última medida é mais intuitiva e reduz a dependência exclusiva do CV.

In [ ]:
# ============================================================
# 🏪 T05 — MÉTRICAS DAS JANELAS
# Similaridade por CV + medida robusta em torno da mediana
# ============================================================

T05_METRICAS = (
    T05_DIR
    / 't05_metricas_janelas.parquet'
)

t05_metricas_sql = (
    str(T05_METRICAS)
    .replace("'", "''")
)

janelas = (
    CONFIG['T05']
    ['janelas_calibracao']
)

tol_mediana = (
    CONFIG['T05']
    ['tolerancia_mediana']
)


# ============================================================
# 🧱 TABELA TEMPORÁRIA
# ============================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE
    t05_metricas_janelas
(
    JANELA_DIAS INTEGER,
    UG_ID VARCHAR,
    FAVORECIDO_ID VARCHAR,
    ANO_TRANSACAO INTEGER,

    DT_INICIO DATE,
    DT_FIM DATE,

    N_TRANSACOES BIGINT,
    N_PORTADORES BIGINT,

    MEDIA_CENTAVOS DOUBLE,
    DP_CENTAVOS DOUBLE,
    CV DOUBLE,

    MEDIANA_CENTAVOS DOUBLE,
    Q1_CENTAVOS DOUBLE,
    Q3_CENTAVOS DOUBLE,
    IQR_CENTAVOS DOUBLE,

    SHARE_DENTRO_FAIXA_MEDIANA DOUBLE,

    MIN_CENTAVOS BIGINT,
    MAX_CENTAVOS BIGINT,

    VALOR_TOTAL_CENTAVOS HUGEINT
)
""")


# ============================================================
# 🔄 PROCESSAMENTO DAS JANELAS
# ============================================================

for janela in tqdm(
    janelas,
    desc='🏪 T05 — calculando janelas',
    unit='janela'
):

    con.execute(
        f"""
        INSERT INTO
            t05_metricas_janelas

        WITH grupos_elegiveis AS (

            SELECT
                UG_ID,
                FAVORECIDO_ID,
                ANO_TRANSACAO

            FROM stg_cpgf_transacoes

            WHERE EH_COMPRA_NACIONAL
              AND VALOR_CENTAVOS > 0
              AND DATA_DT IS NOT NULL
              AND UG_ID IS NOT NULL
              AND PORTADOR_ID IS NOT NULL
              AND FAVORECIDO_IDENTIFICADO

            GROUP BY
                UG_ID,
                FAVORECIDO_ID,
                ANO_TRANSACAO

            HAVING COUNT(*) >= 3
               AND COUNT(DISTINCT PORTADOR_ID) >= 2
        ),

        base AS (

            SELECT
                t.UG_ID,
                t.FAVORECIDO_ID,
                t.PORTADOR_ID,
                t.ANO_TRANSACAO,
                t.DATA_DT,
                t.VALOR_CENTAVOS

            FROM stg_cpgf_transacoes t

            INNER JOIN grupos_elegiveis g
              ON g.UG_ID = t.UG_ID
             AND g.FAVORECIDO_ID = t.FAVORECIDO_ID
             AND g.ANO_TRANSACAO = t.ANO_TRANSACAO

            WHERE t.EH_COMPRA_NACIONAL
              AND t.VALOR_CENTAVOS > 0
              AND t.DATA_DT IS NOT NULL
              AND t.PORTADOR_ID IS NOT NULL
        ),

        anchors AS (

            SELECT DISTINCT
                UG_ID,
                FAVORECIDO_ID,
                ANO_TRANSACAO,
                DATA_DT AS DT_INICIO

            FROM base
        ),

        pares AS (

            SELECT
                a.UG_ID,
                a.FAVORECIDO_ID,
                a.ANO_TRANSACAO,
                a.DT_INICIO,
                b.DATA_DT,
                b.PORTADOR_ID,
                b.VALOR_CENTAVOS

            FROM anchors a

            INNER JOIN base b
              ON b.UG_ID = a.UG_ID
             AND b.FAVORECIDO_ID = a.FAVORECIDO_ID
             AND b.ANO_TRANSACAO = a.ANO_TRANSACAO
             AND b.DATA_DT BETWEEN
                    a.DT_INICIO
                    AND a.DT_INICIO + INTERVAL {janela} DAY
        ),

        stats AS (

            SELECT
                UG_ID,
                FAVORECIDO_ID,
                ANO_TRANSACAO,
                DT_INICIO,

                MAX(DATA_DT) AS DT_FIM,
                COUNT(*) AS N_TRANSACOES,
                COUNT(DISTINCT PORTADOR_ID) AS N_PORTADORES,

                AVG(VALOR_CENTAVOS) AS MEDIA_CENTAVOS,
                STDDEV_POP(VALOR_CENTAVOS) AS DP_CENTAVOS,

                STDDEV_POP(VALOR_CENTAVOS)
                / NULLIF(AVG(VALOR_CENTAVOS), 0) AS CV,

                MEDIAN(VALOR_CENTAVOS) AS MEDIANA_CENTAVOS,

                QUANTILE_CONT(
                    VALOR_CENTAVOS,
                    0.25
                ) AS Q1_CENTAVOS,

                QUANTILE_CONT(
                    VALOR_CENTAVOS,
                    0.75
                ) AS Q3_CENTAVOS,

                MIN(VALOR_CENTAVOS) AS MIN_CENTAVOS,
                MAX(VALOR_CENTAVOS) AS MAX_CENTAVOS,
                SUM(VALOR_CENTAVOS) AS VALOR_TOTAL_CENTAVOS

            FROM pares

            GROUP BY
                UG_ID,
                FAVORECIDO_ID,
                ANO_TRANSACAO,
                DT_INICIO
        ),

        similaridade AS (

            SELECT
                s.UG_ID,
                s.FAVORECIDO_ID,
                s.ANO_TRANSACAO,
                s.DT_INICIO,

                SUM(
                    CASE
                        WHEN p.VALOR_CENTAVOS BETWEEN
                                s.MEDIANA_CENTAVOS * (1 - {tol_mediana})
                                AND
                                s.MEDIANA_CENTAVOS * (1 + {tol_mediana})
                        THEN 1
                        ELSE 0
                    END
                )
                * 1.0
                / NULLIF(COUNT(*), 0)
                    AS SHARE_DENTRO_FAIXA_MEDIANA

            FROM stats s

            INNER JOIN pares p
              ON p.UG_ID = s.UG_ID
             AND p.FAVORECIDO_ID = s.FAVORECIDO_ID
             AND p.ANO_TRANSACAO = s.ANO_TRANSACAO
             AND p.DT_INICIO = s.DT_INICIO

            GROUP BY
                s.UG_ID,
                s.FAVORECIDO_ID,
                s.ANO_TRANSACAO,
                s.DT_INICIO,
                s.MEDIANA_CENTAVOS
        )

        SELECT
            {janela} AS JANELA_DIAS,

            s.UG_ID,
            s.FAVORECIDO_ID,
            s.ANO_TRANSACAO,

            s.DT_INICIO,
            s.DT_FIM,

            s.N_TRANSACOES,
            s.N_PORTADORES,

            s.MEDIA_CENTAVOS,
            s.DP_CENTAVOS,
            s.CV,

            s.MEDIANA_CENTAVOS,
            s.Q1_CENTAVOS,
            s.Q3_CENTAVOS,

            (
                s.Q3_CENTAVOS
                - s.Q1_CENTAVOS
            ) AS IQR_CENTAVOS,

            sim.SHARE_DENTRO_FAIXA_MEDIANA,

            s.MIN_CENTAVOS,
            s.MAX_CENTAVOS,

            s.VALOR_TOTAL_CENTAVOS

        FROM stats s

        INNER JOIN similaridade sim
          ON sim.UG_ID = s.UG_ID
         AND sim.FAVORECIDO_ID = s.FAVORECIDO_ID
         AND sim.ANO_TRANSACAO = s.ANO_TRANSACAO
         AND sim.DT_INICIO = s.DT_INICIO
        """
    )


# ============================================================
# 💾 EXPORTAÇÃO PARA PARQUET
# ============================================================

con.execute(
    f"""
    COPY (
        SELECT *
        FROM t05_metricas_janelas
    )
    TO '{t05_metricas_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ============================================================
# 📊 RESUMO DAS JANELAS
# ============================================================

t05_metricas_resumo_df = con.execute(
    """
    SELECT
        JANELA_DIAS,

        COUNT(*) AS N_JANELAS,
        MAX(N_TRANSACOES) AS MAX_TRANSACOES,
        MAX(N_PORTADORES) AS MAX_PORTADORES,

        MEDIAN(
            SHARE_DENTRO_FAIXA_MEDIANA
        ) AS MEDIANA_SHARE_ROBUSTA,

        MEDIAN(CV) AS MEDIANA_CV,

        MEDIAN(IQR_CENTAVOS) / 100.0
            AS MEDIANA_IQR_REAIS

    FROM t05_metricas_janelas

    GROUP BY JANELA_DIAS

    ORDER BY JANELA_DIAS
    """
).df()

display(t05_metricas_resumo_df)

print(
    '✅ T05 — métricas de janelas com '
    'similaridade robusta concluídas.'
)

## 1️⃣4️⃣ T05 — Grade completa de sensibilidade

Para cada uma das 160 combinações, o notebook registra:

- quantidade de grupos `UG–fornecedor–ano` com ao menos uma janela elegível;
- quantidade representativa de transações;
- valor representativo;
- média de CV;
- mediana de CV.

Na calibração, cada `UG–fornecedor–ano` contribui com **uma janela representativa**, escolhida pela seguinte ordem:

1. maior número de transações;
2. maior número de portadores;
3. menor CV;
4. maior materialidade;
5. data inicial mais antiga.

Isso evita que dezenas de janelas quase idênticas do mesmo grupo inflem a comparação entre parâmetros.

In [ ]:
grade = list(
    product(
        CONFIG['T05']
        ['janelas_calibracao'],

        CONFIG['T05']
        ['min_transacoes_calibracao'],

        CONFIG['T05']
        ['min_portadores_calibracao'],

        CONFIG['T05']
        ['cv_calibracao']
    )
)

sensibilidade = []

for (
    janela,
    min_n,
    min_p,
    cv_limite
) in tqdm(
    grade,
    desc='🧪 T05 — sensibilidade',
    unit='combinação'
):
    linha = con.execute(
        f"""
        WITH qual AS (
            SELECT *

            FROM t05_metricas_janelas

            WHERE
                JANELA_DIAS = {janela}
                AND N_TRANSACOES >= {min_n}
                AND N_PORTADORES >= {min_p}
                AND CV <= {cv_limite}
        ),

        representativa AS (
            SELECT
                *,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        UG_ID,
                        FAVORECIDO_ID,
                        ANO_TRANSACAO

                    ORDER BY
                        N_TRANSACOES DESC,
                        N_PORTADORES DESC,
                        CV ASC,
                        VALOR_TOTAL_CENTAVOS DESC,
                        DT_INICIO ASC
                ) AS RN

            FROM qual
        )

        SELECT
            COUNT(*)
                AS N_GRUPOS,

            SUM(
                N_TRANSACOES
            ) AS TRANSACOES_REPRESENTATIVAS,

            SUM(
                VALOR_TOTAL_CENTAVOS
            ) / 100.0
                AS VALOR_REPRESENTATIVO,

            AVG(
                CV
            ) AS CV_MEDIO,

            MEDIAN(
                CV
            ) AS CV_MEDIANO

        FROM representativa

        WHERE
            RN = 1
        """
    ).df().iloc[0].to_dict()

    sensibilidade.append({
        'janela_dias':
            janela,
        'min_transacoes':
            min_n,
        'min_portadores':
            min_p,
        'cv_limite':
            cv_limite,
        **linha,
    })

t05_sensibilidade_df = pd.DataFrame(
    sensibilidade
)

salvar_csv(
    t05_sensibilidade_df,
    T05_DIR
    / 't05_sensibilidade_completa.csv'
)

salvar_parquet(
    t05_sensibilidade_df,
    T05_DIR
    / 't05_sensibilidade_completa.parquet'
)

display(
    t05_sensibilidade_df
    .sort_values(
        [
            'N_GRUPOS',
            'janela_dias'
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(30)
)

print(
    f'✅ Combinações avaliadas: '
    f'{len(t05_sensibilidade_df)}'
)

## 1️⃣5️⃣ T05 — Episódios da regra-base e deduplicação

Depois da calibração, o notebook também executa a regra-base prevista na matriz:

- 5 ou mais transações;
- 2 ou mais portadores;
- janela de 30 dias;
- CV ≤ 20%;
- nível reforçado quando CV ≤ 10%.

### 🔁 Deduplicação

Janelas sobrepostas do mesmo `UG–fornecedor–ano` são agrupadas em blocos de sobreposição. Em cada bloco permanece a janela representativa mais forte.

As janelas não são fundidas em um período artificialmente maior.

### Materialidade e similaridade robusta

A versão 1.2 não transforma materialidade em um score. Cada episódio recebe o percentil de valor acumulado dentro do próprio exercício (`PERCENTIL_MATERIALIDADE_ANO`) e o percentil de recorrência. Também é marcada a proporção de compras situada em ±20% da mediana.

In [ ]:
def deduplicar_episodios_t05(df):
    if df.empty:
        return df.copy()

    df = df.copy()

    df['DT_INICIO'] = pd.to_datetime(
        df['DT_INICIO']
    )

    df['DT_FIM'] = pd.to_datetime(
        df['DT_FIM']
    )

    escolhidos = []

    grupos = df.groupby(
        [
            'UG_ID',
            'FAVORECIDO_ID',
            'ANO_TRANSACAO'
        ],
        sort=False,
        dropna=False
    )

    total_grupos = grupos.ngroups

    for chave, g in tqdm(
        grupos,
        total=total_grupos,
        desc='🧹 T05 — deduplicando',
        unit='grupo'
    ):
        g = (
            g
            .sort_values(
                [
                    'DT_INICIO',
                    'DT_FIM',
                    'N_TRANSACOES'
                ],
                ascending=[
                    True,
                    True,
                    False
                ]
            )
            .reset_index(
                drop=True
            )
        )

        cluster = []
        cluster_fim = None

        def escolher_cluster(
            cluster_rows
        ):
            c = pd.DataFrame(
                cluster_rows
            )

            c = c.sort_values(
                [
                    'N_TRANSACOES',
                    'N_PORTADORES',
                    'CV',
                    'VALOR_TOTAL_CENTAVOS',
                    'DT_INICIO'
                ],
                ascending=[
                    False,
                    False,
                    True,
                    False,
                    True
                ]
            )

            return (
                c.iloc[0]
                .to_dict()
            )

        for _, row in g.iterrows():
            row_dict = row.to_dict()

            if not cluster:
                cluster = [
                    row_dict
                ]

                cluster_fim = (
                    row['DT_FIM']
                )

                continue

            if (
                row['DT_INICIO']
                <= cluster_fim
            ):
                cluster.append(
                    row_dict
                )

                cluster_fim = max(
                    cluster_fim,
                    row['DT_FIM']
                )

            else:
                escolhidos.append(
                    escolher_cluster(
                        cluster
                    )
                )

                cluster = [
                    row_dict
                ]

                cluster_fim = (
                    row['DT_FIM']
                )

        if cluster:
            escolhidos.append(
                escolher_cluster(
                    cluster
                )
            )

    out = pd.DataFrame(
        escolhidos
    )

    if not out.empty:
        out = (
            out
            .sort_values(
                [
                    'UG_ID',
                    'FAVORECIDO_ID',
                    'ANO_TRANSACAO',
                    'DT_INICIO'
                ]
            )
            .reset_index(
                drop=True
            )
        )

    return out


p_t05 = CONFIG['T05']

t05_candidatos_df = con.execute(
    f"""
    SELECT *

    FROM t05_metricas_janelas

    WHERE
        JANELA_DIAS
            = {p_t05['janela_dias']}
        AND N_TRANSACOES
            >= {p_t05['min_transacoes']}
        AND N_PORTADORES
            >= {p_t05['min_portadores']}
        AND CV
            <= {p_t05['cv_base']}
    """
).df()

print(
    '📌 Janelas candidatas '
    'antes da deduplicação:',
    f'{len(t05_candidatos_df):,}'
)

t05_episodios_df = (
    deduplicar_episodios_t05(
        t05_candidatos_df
    )
)

if not t05_episodios_df.empty:

    t05_episodios_df[
        'NIVEL_TRIAGEM'
    ] = np.where(
        t05_episodios_df[
            'CV'
        ] <= p_t05[
            'cv_reforcado'
        ],
        'REFORCADO',
        'ATENCAO'
    )

    t05_episodios_df['PERCENTIL_MATERIALIDADE_ANO'] = (
        t05_episodios_df.groupby('ANO_TRANSACAO')['VALOR_TOTAL_CENTAVOS']
        .rank(pct=True, method='average')
    )

    t05_episodios_df['PERCENTIL_RECORRENCIA_ANO'] = (
        t05_episodios_df.groupby('ANO_TRANSACAO')['N_TRANSACOES']
        .rank(pct=True, method='average')
    )

    t05_episodios_df['FAIXA_MATERIALIDADE'] = np.select(
        [
            t05_episodios_df['PERCENTIL_MATERIALIDADE_ANO'] >= 0.90,
            t05_episodios_df['PERCENTIL_MATERIALIDADE_ANO'] >= 0.75,
        ],
        ['ELEVADA', 'MODERADA'],
        default='BAIXA'
    )

    t05_episodios_df['SIMILARIDADE_ROBUSTA_ALTA'] = (
        t05_episodios_df['SHARE_DENTRO_FAIXA_MEDIANA']
        >= CONFIG['T05']['share_mediana_referencia']
    )

    t05_episodios_df[
        'ID_SINAL'
    ] = t05_episodios_df.apply(
        lambda r: id_sinal(
            'T05',
            r['UG_ID'],
            r['FAVORECIDO_ID'],
            r['ANO_TRANSACAO'],
            pd.Timestamp(
                r['DT_INICIO']
            ).date(),
            pd.Timestamp(
                r['DT_FIM']
            ).date(),
            r['N_TRANSACOES'],
            round(
                float(r['CV']),
                8
            )
        ),
        axis=1
    )

salvar_parquet(
    t05_episodios_df,
    T05_DIR
    / 't05_episodios.parquet'
)

salvar_csv(
    t05_episodios_df,
    T05_DIR
    / 't05_episodios.csv'
)

print(
    '✅ Episódios após deduplicação:',
    f'{len(t05_episodios_df):,}'
)

if len(t05_episodios_df):
    display(
        t05_episodios_df
        .sort_values(
            [
                'N_PORTADORES',
                'N_TRANSACOES',
                'VALOR_TOTAL_CENTAVOS'
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .head(30)
    )

In [ ]:
T05_PONTE = (
    T05_DIR
    / 't05_episodios_transacoes.parquet'
)

if len(t05_episodios_df):

    t05_registro = (
        t05_episodios_df[
            [
                'ID_SINAL',
                'UG_ID',
                'FAVORECIDO_ID',
                'ANO_TRANSACAO',
                'DT_INICIO',
                'DT_FIM',
            ]
        ]
        .copy()
    )

    con.register(
        't05_episodios_python',
        t05_registro
    )

    t05_ponte_sql = (
        str(T05_PONTE)
        .replace("'", "''")
    )

    con.execute(
        f"""
        COPY (
            SELECT
                e.ID_SINAL,
                t.ID_TRANSACAO,
                t.UG_ID,
                t.FAVORECIDO_ID,
                t.PORTADOR_ID,
                t.DATA_DT,
                t.VALOR_NUM,
                t.VALOR_CENTAVOS,
                t.COMPETENCIA_ARQUIVO,
                t.ARQUIVO_ORIGEM

            FROM
                t05_episodios_python e

            JOIN
                stg_cpgf_transacoes t

              ON t.UG_ID
                    = e.UG_ID
             AND t.FAVORECIDO_ID
                    = e.FAVORECIDO_ID
             AND t.ANO_TRANSACAO
                    = e.ANO_TRANSACAO
             AND t.DATA_DT
                    BETWEEN
                        e.DT_INICIO
                        AND e.DT_FIM

            WHERE
                t.EH_COMPRA_NACIONAL
                AND t.VALOR_CENTAVOS > 0
        )
        TO '{t05_ponte_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    con.unregister(
        't05_episodios_python'
    )

    print(
        '✅ Ponte T05 ↔ transações criada.'
    )

else:
    print(
        'ℹ️ Nenhum episódio T05 '
        'na regra-base.'
    )

## 1️⃣6️⃣ T06 — Concentração em fornecedor

T06 calcula a concentração financeira em `UG × ano da transação`.

### Requisitos para sinal automático

- 20 ou mais compras identificadas;
- 3 ou mais fornecedores;
- cobertura de fornecedor identificado ≥ 80% do valor;
- `Top-1 ≥ 50%`.

Também são calculados:

- participação do maior fornecedor;
- participação do Top-5;
- HHI.

Os limites de 50%, 70% e 80% são **faixas analíticas**, e não limites legais.

### Novidade da versão 1.2

A concentração passa a ser exibida em duas dimensões: **participação no valor** e **participação no número de transações**. Isso permite distinguir uma única compra de grande materialidade de uma relação operacional recorrente com o fornecedor.

In [ ]:
T06_INDICADORES = (
    T06_DIR
    / 't06_concentracao_ug_ano.parquet'
)

T06_SINAIS = (
    T06_DIR
    / 't06_concentracao_sinais.parquet'
)

T06_PONTE = (
    T06_DIR
    / 't06_concentracao_transacoes_top1.parquet'
)

t06_ind_sql = (
    str(T06_INDICADORES)
    .replace("'", "''")
)

t06_sinais_sql = (
    str(T06_SINAIS)
    .replace("'", "''")
)

t06_ponte_sql = (
    str(T06_PONTE)
    .replace("'", "''")
)

p_t06 = CONFIG['T06']

con.execute("""
CREATE OR REPLACE TEMP VIEW
    t06_total AS

SELECT
    UG_ID,
    ANO_TRANSACAO,

    COUNT(*)
        AS N_TOTAL_COMPRAS,

    SUM(
        VALOR_CENTAVOS
    ) AS TOTAL_UG_CENTAVOS

FROM stg_cpgf_transacoes

WHERE
    EH_COMPRA_NACIONAL
    AND VALOR_CENTAVOS > 0
    AND UG_ID IS NOT NULL
    AND ANO_TRANSACAO IS NOT NULL

GROUP BY
    1,
    2
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW
    t06_fornecedor AS

SELECT
    UG_ID,
    ANO_TRANSACAO,
    FAVORECIDO_ID,

    COUNT(*)
        AS N_FORN,

    SUM(
        VALOR_CENTAVOS
    ) AS V_FORN_CENTAVOS

FROM stg_cpgf_transacoes

WHERE
    EH_COMPRA_NACIONAL
    AND VALOR_CENTAVOS > 0
    AND UG_ID IS NOT NULL
    AND ANO_TRANSACAO IS NOT NULL
    AND FAVORECIDO_IDENTIFICADO

GROUP BY
    1,
    2,
    3
""")

con.execute(
    f"""
    COPY (
        WITH shares AS (
            SELECT
                *,

                SUM(
                    V_FORN_CENTAVOS
                ) OVER (
                    PARTITION BY
                        UG_ID,
                        ANO_TRANSACAO
                )
                    AS TOTAL_IDENTIFICADO_CENTAVOS,

                COUNT(*) OVER (
                    PARTITION BY
                        UG_ID,
                        ANO_TRANSACAO
                )
                    AS N_FORNECEDORES,

                SUM(
                    N_FORN
                ) OVER (
                    PARTITION BY
                        UG_ID,
                        ANO_TRANSACAO
                )
                    AS N_COMPRAS_IDENTIFICADAS,

                V_FORN_CENTAVOS * 1.0 / NULLIF(SUM(V_FORN_CENTAVOS) OVER (PARTITION BY UG_ID, ANO_TRANSACAO),0) AS SHARE,

                N_FORN * 1.0 / NULLIF(SUM(N_FORN) OVER (PARTITION BY UG_ID, ANO_TRANSACAO),0) AS SHARE_QTD,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        UG_ID,
                        ANO_TRANSACAO

                    ORDER BY
                        V_FORN_CENTAVOS DESC,
                        FAVORECIDO_ID
                )
                    AS RN

            FROM t06_fornecedor
        ),

        resumo AS (
            SELECT
                s.UG_ID,
                s.ANO_TRANSACAO,

                MAX(
                    s.N_FORNECEDORES
                ) AS N_FORNECEDORES,

                MAX(
                    s.N_COMPRAS_IDENTIFICADAS
                ) AS N_COMPRAS_IDENTIFICADAS,

                MAX(
                    s.TOTAL_IDENTIFICADO_CENTAVOS
                ) AS TOTAL_IDENTIFICADO_CENTAVOS,

                MAX(
                    t.TOTAL_UG_CENTAVOS
                ) AS TOTAL_UG_CENTAVOS,

                MAX(
                    s.TOTAL_IDENTIFICADO_CENTAVOS
                )
                * 1.0
                / NULLIF(
                    MAX(
                        t.TOTAL_UG_CENTAVOS
                    ),
                    0
                )
                    AS COBERTURA_VALOR,

                MAX(s.SHARE) AS TOP1_SHARE,

                MAX(s.SHARE_QTD) AS TOP1_SHARE_QTD,

                ARG_MAX(s.FAVORECIDO_ID, s.N_FORN) AS TOP1_FAVORECIDO_QTD,

                MAX(s.N_FORN) * 1.0 / NULLIF(SUM(s.N_FORN),0) AS TOP1_SHARE_QTD_CHECK,

                SUM(
                    CASE
                        WHEN s.RN <= 5
                        THEN s.SHARE
                        ELSE 0
                    END
                )
                    AS TOP5_SHARE,

                SUM(s.SHARE * s.SHARE) AS HHI,

                SUM(s.SHARE_QTD * s.SHARE_QTD) AS HHI_QTD,

                MAX(
                    CASE
                        WHEN s.RN = 1
                        THEN s.FAVORECIDO_ID
                    END
                )
                    AS TOP1_FAVORECIDO_ID,

                MAX(
                    CASE
                        WHEN s.RN = 1
                        THEN s.V_FORN_CENTAVOS
                    END
                )
                    AS TOP1_VALOR_CENTAVOS

            FROM shares s

            JOIN t06_total t
              ON t.UG_ID = s.UG_ID
             AND t.ANO_TRANSACAO
                    = s.ANO_TRANSACAO

            GROUP BY
                s.UG_ID,
                s.ANO_TRANSACAO
        )

        SELECT *
        FROM resumo
    )
    TO '{t06_ind_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

con.execute(
    f"""
    COPY (
        SELECT
            'T06_' || MD5(
                'T06|'
                || '{VERSAO_REGRAS}'
                || '|'
                || UG_ID
                || '|'
                || CAST(
                    ANO_TRANSACAO
                    AS VARCHAR
                )
                || '|'
                || TOP1_FAVORECIDO_ID
            ) AS ID_SINAL,

            *,

            CASE
                WHEN TOP1_SHARE
                        >= {p_t06['share_muito_elevado']}
                    THEN 'MUITO_ELEVADO'

                WHEN TOP1_SHARE
                        >= {p_t06['share_reforcado']}
                    THEN 'REFORCADO'

                ELSE 'ATENCAO'
            END AS NIVEL_TRIAGEM

        FROM read_parquet(
            '{t06_ind_sql}'
        )

        WHERE
            N_COMPRAS_IDENTIFICADAS
                >= {p_t06['min_compras_identificadas']}
            AND N_FORNECEDORES
                >= {p_t06['min_fornecedores']}
            AND COBERTURA_VALOR
                >= {p_t06['cobertura_min']}
            AND TOP1_SHARE
                >= {p_t06['share_base']}
    )
    TO '{t06_sinais_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

con.execute(
    f"""
    COPY (
        SELECT
            s.ID_SINAL,
            t.ID_TRANSACAO,
            t.UG_ID,
            t.FAVORECIDO_ID,
            t.PORTADOR_ID,
            t.DATA_DT,
            t.VALOR_NUM,
            t.VALOR_CENTAVOS

        FROM read_parquet(
            '{t06_sinais_sql}'
        ) s

        JOIN stg_cpgf_transacoes t
          ON t.UG_ID = s.UG_ID
         AND t.ANO_TRANSACAO
                = s.ANO_TRANSACAO
         AND t.FAVORECIDO_ID
                = s.TOP1_FAVORECIDO_ID

        WHERE
            t.EH_COMPRA_NACIONAL
            AND t.VALOR_CENTAVOS > 0
    )
    TO '{t06_ponte_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

t06_resumo_df = con.execute(
    f"""
    SELECT
        NIVEL_TRIAGEM,

        COUNT(*)
            AS N_UG_ANO,

        AVG(
            TOP1_SHARE
        ) AS TOP1_MEDIO,

        MAX(
            TOP1_SHARE
        ) AS TOP1_MAXIMO

    FROM read_parquet(
        '{t06_sinais_sql}'
    )

    GROUP BY 1

    ORDER BY 1
    """
).df()

salvar_csv(
    t06_resumo_df,
    T06_DIR
    / 't06_resumo.csv'
)

display(
    t06_resumo_df
)

print(
    '✅ T06 concluída.'
)

## 1️⃣7️⃣ T07 — Saques sucessivos e recorrência anual

A versão 1.2 separa duas camadas.

### T07-A — episódio diário

Mantém a regra descritiva `UG + portador + data` com dois ou mais saques. Essa saída continua disponível para consulta e rastreabilidade.

### T07-B — recorrência anual do comportamento

A priorização passa a considerar **quantos dias diferentes** do exercício o portador repetiu o comportamento de múltiplos saques. O ranking é relativo ao próprio ano e só é considerado para priorização quando existem pelo menos 10 portadores comparáveis.

> O resultado continua sem verificar a autorização do saque, que não está disponível no CSV público.

In [ ]:
T07_SINAIS = T07_DIR / 't07_saques_sucessivos.parquet'
T07_INDICADOR = T07_DIR / 't07_indicador_saques_ug_ano.parquet'
T07_PONTE = T07_DIR / 't07_saques_transacoes.parquet'
T07_RECORRENCIA = T07_DIR / 't07_recorrencia_portador_ano.parquet'
T07_PRIORITARIOS = T07_DIR / 't07_recorrencia_prioritaria.parquet'
T07_PRIORITARIOS_PONTE = T07_DIR / 't07_recorrencia_prioritaria_transacoes.parquet'

paths = {p: str(p).replace("'", "''") for p in [T07_SINAIS,T07_INDICADOR,T07_PONTE,T07_RECORRENCIA,T07_PRIORITARIOS,T07_PRIORITARIOS_PONTE]}
p_t07 = CONFIG['T07']

con.execute(f"""
COPY (
    SELECT
        'T07D_' || MD5('T07D|' || '{VERSAO_REGRAS}' || '|' || UG_ID || '|' || PORTADOR_ID || '|' || CAST(DATA_DT AS VARCHAR)) AS ID_SINAL_DIARIO,
        UG_ID, PORTADOR_ID, DATA_DT,
        COUNT(*) AS N_SAQUES,
        SUM(VALOR_CENTAVOS) AS TOTAL_CENTAVOS,
        MAX(VALOR_CENTAVOS) AS MAIOR_SAQUE_CENTAVOS,
        STRING_AGG(DISTINCT "TRANSAÇÃO", ' | ') AS TIPOS_SAQUE,
        CASE WHEN COUNT(*) >= {p_t07['reforcado_saques_dia']} THEN 'REFORCADO' ELSE 'ATENCAO' END AS NIVEL_EPISODIO
    FROM stg_cpgf_transacoes
    WHERE EH_SAQUE_EFETIVO
      AND DATA_DT IS NOT NULL
      AND UG_ID IS NOT NULL
      AND PORTADOR_ID IS NOT NULL
      AND VALOR_CENTAVOS IS NOT NULL
    GROUP BY 1,2,3,4
    HAVING COUNT(*) >= {p_t07['min_saques_dia']}
)
TO '{paths[T07_SINAIS]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    WITH base AS (
        SELECT
            UG_ID, ANO_TRANSACAO,
            SUM(CASE WHEN EH_SAQUE_EFETIVO AND VALOR_CENTAVOS>0 THEN VALOR_CENTAVOS ELSE 0 END) AS VALOR_SAQUES,
            SUM(CASE WHEN (EH_SAQUE_EFETIVO OR EH_COMPRA_EFETIVA) AND VALOR_CENTAVOS>0 THEN VALOR_CENTAVOS ELSE 0 END) AS VALOR_OBSERVADO
        FROM stg_cpgf_transacoes
        WHERE ANO_TRANSACAO IS NOT NULL AND UG_ID IS NOT NULL
        GROUP BY 1,2
    )
    SELECT *, VALOR_SAQUES * 1.0 / NULLIF(VALOR_OBSERVADO,0) AS SHARE_SAQUES_CPGF
    FROM base
)
TO '{paths[T07_INDICADOR]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        s.ID_SINAL_DIARIO,
        t.ID_TRANSACAO,
        t.UG_ID, t.PORTADOR_ID, t.DATA_DT, t.VALOR_NUM, t.VALOR_CENTAVOS,
        t."TRANSAÇÃO" AS TRANSACAO
    FROM read_parquet('{paths[T07_SINAIS]}') s
    JOIN stg_cpgf_transacoes t
      ON t.UG_ID=s.UG_ID AND t.PORTADOR_ID=s.PORTADOR_ID AND t.DATA_DT=s.DATA_DT
    WHERE t.EH_SAQUE_EFETIVO
)
TO '{paths[T07_PONTE]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

# T07-B — recorrência anual por portador
con.execute(f"""
COPY (
    WITH anual AS (
        SELECT
            UG_ID,
            PORTADOR_ID,
            EXTRACT(YEAR FROM DATA_DT)::INTEGER AS ANO_TRANSACAO,
            COUNT(*) AS N_DIAS_MULTISAQUE,
            SUM(N_SAQUES) AS N_SAQUES_EM_EPISODIOS,
            SUM(TOTAL_CENTAVOS) AS VALOR_EPISODIOS_CENTAVOS,
            MAX(N_SAQUES) AS MAX_SAQUES_DIA
        FROM read_parquet('{paths[T07_SINAIS]}')
        GROUP BY 1,2,3
    ),
    limites AS (
        SELECT
            ANO_TRANSACAO,
            COUNT(*) AS N_PORTADORES_COMPARAVEIS_ANO,
            QUANTILE_CONT(N_DIAS_MULTISAQUE, {p_t07['percentil_priorizacao']}) AS LIMIAR_PRIORIZACAO_DIAS
        FROM anual
        GROUP BY 1
    ),
    rankeado AS (
        SELECT
            a.*,
            l.N_PORTADORES_COMPARAVEIS_ANO,
            l.LIMIAR_PRIORIZACAO_DIAS,
            PERCENT_RANK() OVER (PARTITION BY a.ANO_TRANSACAO ORDER BY a.N_DIAS_MULTISAQUE) AS PERCENT_RANK_DIAS
        FROM anual a
        JOIN limites l USING (ANO_TRANSACAO)
    )
    SELECT
        *,
        CASE
            WHEN N_DIAS_MULTISAQUE >= {p_t07['min_dias_recorrencia']}
             AND N_PORTADORES_COMPARAVEIS_ANO >= {p_t07['min_comparaveis_ano']}
             AND N_DIAS_MULTISAQUE >= LIMIAR_PRIORIZACAO_DIAS
            THEN TRUE ELSE FALSE
        END AS PRIORITARIO
    FROM rankeado
)
TO '{paths[T07_RECORRENCIA]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        'T07_' || MD5('T07|' || '{VERSAO_REGRAS}' || '|' || UG_ID || '|' || PORTADOR_ID || '|' || CAST(ANO_TRANSACAO AS VARCHAR)) AS ID_SINAL,
        *
    FROM read_parquet('{paths[T07_RECORRENCIA]}')
    WHERE PRIORITARIO
)
TO '{paths[T07_PRIORITARIOS]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        p.ID_SINAL,
        d.ID_SINAL_DIARIO,
        b.ID_TRANSACAO,
        b.UG_ID, b.PORTADOR_ID, b.DATA_DT, b.VALOR_NUM, b.VALOR_CENTAVOS
    FROM read_parquet('{paths[T07_PRIORITARIOS]}') p
    JOIN read_parquet('{paths[T07_SINAIS]}') d
      ON d.UG_ID=p.UG_ID AND d.PORTADOR_ID=p.PORTADOR_ID
     AND EXTRACT(YEAR FROM d.DATA_DT)::INTEGER=p.ANO_TRANSACAO
    JOIN read_parquet('{paths[T07_PONTE]}') b
      ON b.ID_SINAL_DIARIO=d.ID_SINAL_DIARIO
)
TO '{paths[T07_PRIORITARIOS_PONTE]}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

t07_resumo_df = con.execute(f"""
SELECT
    EXTRACT(YEAR FROM DATA_DT)::INTEGER AS ANO,
    COUNT(*) AS N_EPISODIOS_DIARIOS,
    SUM(N_SAQUES) AS N_SAQUES,
    SUM(TOTAL_CENTAVOS)/100.0 AS VALOR_TOTAL
FROM read_parquet('{paths[T07_SINAIS]}')
GROUP BY 1 ORDER BY 1
""").df()

t07_recorrencia_resumo_df = con.execute(f"""
SELECT
    ANO_TRANSACAO,
    COUNT(*) AS N_PORTADORES_COM_MULTISAQUE,
    COUNT_IF(PRIORITARIO) AS N_PORTADORES_PRIORITARIOS,
    MEDIAN(N_DIAS_MULTISAQUE) AS MEDIANA_DIAS,
    MAX(N_DIAS_MULTISAQUE) AS MAX_DIAS
FROM read_parquet('{paths[T07_RECORRENCIA]}')
GROUP BY 1 ORDER BY 1
""").df()

t07_resumo_df['STATUS_PERIODO'] = np.where(
    t07_resumo_df['ANO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

t07_recorrencia_resumo_df['STATUS_PERIODO'] = np.where(
    t07_recorrencia_resumo_df['ANO_TRANSACAO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

salvar_csv(
    t07_resumo_df,
    T07_DIR / 't07_resumo_anual.csv'
)

salvar_csv(
    t07_recorrencia_resumo_df,
    T07_DIR / 't07_recorrencia_resumo.csv'
)
display(t07_resumo_df)
display(t07_recorrencia_resumo_df)
print('✅ T07-A e T07-B concluídas.')

## 1️⃣8️⃣ T08 — Funções da Lei de Newcomb-Benford

A trilha T08 é tratada como um **pipeline**, e não como um único teste:

1. diagnóstico de elegibilidade;
2. primeiro dígito;
3. dois primeiros dígitos;
4. MAD como métrica principal;
5. Z e χ² como diagnósticos auxiliares;
6. *Summation Test*;
7. *Number Duplication Test*;
8. *drill-down*;
9. persistência longitudinal.

### População principal

`COMPRA A/V - R$ - APRES`

Saques, operações internacionais, registros sigilosos e processos geradores diferentes não são misturados à população principal.

### D12

A análise principal de dois primeiros dígitos utiliza valores `≥ R$ 10`.

Também será executada uma análise de sensibilidade com todos os valores positivos.

In [ ]:
MAD_LIMITES_D1 = [
    (
        0.006,
        'Conformidade próxima'
    ),
    (
        0.012,
        'Conformidade aceitável'
    ),
    (
        0.015,
        (
            'Conformidade '
            'marginalmente aceitável'
        )
    ),
    (
        np.inf,
        'Não conformidade'
    ),
]

MAD_LIMITES_D12 = [
    (
        0.0012,
        'Conformidade próxima'
    ),
    (
        0.0018,
        'Conformidade aceitável'
    ),
    (
        0.0022,
        (
            'Conformidade '
            'marginalmente aceitável'
        )
    ),
    (
        np.inf,
        'Não conformidade'
    ),
]

SOMATORIO_ESPERADO = (
    1 / 90
)

Z_CRITICO_5 = (
    1.96
)


def benford_probs(tipo):

    if tipo == 'D1':
        digitos = np.arange(
            1,
            10
        )

    elif tipo == 'D12':
        digitos = np.arange(
            10,
            100
        )

    else:
        raise ValueError(
            'tipo deve ser D1 ou D12'
        )

    probs = np.log10(
        1 + 1 / digitos
    )

    return pd.DataFrame({
        'DIGITO':
            digitos,
        'PROB_ESPERADA':
            probs,
    })


def classificar_mad_benford(
    mad,
    tipo
):
    limites = (
        MAD_LIMITES_D1
        if tipo == 'D1'
        else MAD_LIMITES_D12
    )

    for limite, classe in limites:

        if mad <= limite:
            return classe

    return 'Não classificado'


def completar_frequencias(
    contagens,
    tipo,
    group_cols=None
):
    group_cols = (
        group_cols or []
    )

    probs = benford_probs(
        tipo
    )

    if not group_cols:

        out = probs.merge(
            contagens,
            on='DIGITO',
            how='left'
        )

        out['AC'] = (
            out['AC']
            .fillna(0)
        )

        n = out['AC'].sum()

        out['AP'] = (
            out['AC'] / n
            if n
            else 0
        )

        out['EC'] = (
            out['PROB_ESPERADA']
            * n
        )

        out['DIFERENCA'] = (
            out['AP']
            - out['PROB_ESPERADA']
        )

        out['N'] = n

        return out

    grupos = (
        contagens[
            group_cols
        ]
        .drop_duplicates()
    )

    grupos['_key'] = 1

    probs2 = probs.copy()
    probs2['_key'] = 1

    grade = (
        grupos
        .merge(
            probs2,
            on='_key'
        )
        .drop(
            columns='_key'
        )
    )

    out = grade.merge(
        contagens,
        on=(
            group_cols
            + ['DIGITO']
        ),
        how='left'
    )

    out['AC'] = (
        out['AC']
        .fillna(0)
    )

    out['N'] = (
        out
        .groupby(
            group_cols
        )['AC']
        .transform(
            'sum'
        )
    )

    out['AP'] = (
        out['AC']
        / out['N']
        .replace(
            0,
            np.nan
        )
    )

    out['AP'] = (
        out['AP']
        .fillna(0)
    )

    out['EC'] = (
        out['PROB_ESPERADA']
        * out['N']
    )

    out['DIFERENCA'] = (
        out['AP']
        - out['PROB_ESPERADA']
    )

    return out


def resumir_benford_tabela(
    tabela,
    tipo,
    group_cols=None
):
    group_cols = (
        group_cols or []
    )

    if not group_cols:

        n = (
            float(
                tabela['N'].iloc[0]
            )
            if len(tabela)
            else 0
        )

        mad = float(
            np.mean(
                np.abs(
                    tabela['DIFERENCA']
                )
            )
        )

        chi = float(
            np.nansum(
                (
                    (
                        tabela['AC']
                        - tabela['EC']
                    ) ** 2
                )
                / tabela['EC']
                .replace(
                    0,
                    np.nan
                )
            )
        )

        return pd.DataFrame([
            {
                'N':
                    int(n),
                'MAD':
                    mad,
                'CLASSIFICACAO_MAD':
                    classificar_mad_benford(
                        mad,
                        tipo
                    ),
                'CHI2':
                    chi,
            }
        ])

    tabela2 = tabela.copy()

    tabela2['ABS_DIF'] = (
        tabela2[
            'DIFERENCA'
        ].abs()
    )

    tabela2['CHI_PARTE'] = (
        (
            tabela2['AC']
            - tabela2['EC']
        ) ** 2
        / tabela2['EC']
        .replace(
            0,
            np.nan
        )
    )

    resumo = (
        tabela2
        .groupby(
            group_cols,
            as_index=False
        )
        .agg(
            N=(
                'N',
                'max'
            ),
            MAD=(
                'ABS_DIF',
                'mean'
            ),
            CHI2=(
                'CHI_PARTE',
                'sum'
            ),
        )
    )

    resumo[
        'CLASSIFICACAO_MAD'
    ] = resumo['MAD'].apply(
        lambda x:
            classificar_mad_benford(
                x,
                tipo
            )
    )

    return resumo


def adicionar_z(
    tabela
):
    tabela = tabela.copy()

    n = tabela[
        'N'
    ].astype(
        float
    )

    p0 = tabela[
        'PROB_ESPERADA'
    ].astype(
        float
    )

    p = tabela[
        'AP'
    ].astype(
        float
    )

    diff = (
        p - p0
    ).abs()

    correcao = (
        1
        / (
            2
            * n.replace(
                0,
                np.nan
            )
        )
    )

    numerador = np.where(
        correcao < diff,
        diff - correcao,
        diff
    )

    denominador = np.sqrt(
        p0
        * (
            1 - p0
        )
        / n.replace(
            0,
            np.nan
        )
    )

    tabela['Z'] = (
        numerador
        / denominador
    )

    tabela[
        'Z_MAIOR_1_96'
    ] = (
        tabela['Z']
        > Z_CRITICO_5
    )

    return tabela


print(
    '✅ Funções Benford carregadas.'
)

## 1️⃣9️⃣ T08 — Base digital e elegibilidade

A extração dos dígitos significativos é feita matematicamente a partir da mantissa.

O diagnóstico de elegibilidade não será reduzido a um *score* único. Os indicadores serão exibidos separadamente:

- N;
- valores únicos;
- média;
- mediana;
- assimetria;
- mínimo;
- máximo;
- ordens de magnitude;
- concentração nos cinco valores mais frequentes;
- frequência de valores inteiros e múltiplos de 10/100.

A interpretação final permanece humana e documentada.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW
    v_benford_compras
AS

WITH base AS (
    SELECT
        *,

        VALOR_NUM
        / POWER(
            10,
            FLOOR(
                LOG10(
                    VALOR_NUM
                )
            )
        )
            AS MANTISSA

    FROM stg_cpgf_transacoes

    WHERE
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
        AND ANO_TRANSACAO
            IS NOT NULL
)

SELECT
    *,

    LEAST(
        9,
        GREATEST(
            1,
            CAST(
                FLOOR(
                    MANTISSA
                    + 1e-10
                )
                AS INTEGER
            )
        )
    )
        AS D1,

    LEAST(
        99,
        GREATEST(
            10,
            CAST(
                FLOOR(
                    MANTISSA
                    * 10
                    + 1e-10
                )
                AS INTEGER
            )
        )
    )
        AS D12

FROM base
""")

elegibilidade_global_df = con.execute("""
WITH freq AS (
    SELECT
        VALOR_CENTAVOS,
        COUNT(*)
            AS FREQ

    FROM v_benford_compras

    GROUP BY 1
),

top5 AS (
    SELECT
        SUM(
            FREQ
        ) AS N_TOP5

    FROM (
        SELECT
            FREQ

        FROM freq

        ORDER BY
            FREQ DESC

        LIMIT 5
    )
),

stats AS (
    SELECT
        COUNT(*)
            AS N,

        COUNT(
            DISTINCT VALOR_CENTAVOS
        )
            AS VALORES_UNICOS,

        AVG(
            VALOR_NUM
        )
            AS MEDIA,

        MEDIAN(
            VALOR_NUM
        )
            AS MEDIANA,

        SKEWNESS(
            VALOR_NUM
        )
            AS ASSIMETRIA,

        MIN(
            VALOR_NUM
        )
            AS MINIMO,

        MAX(
            VALOR_NUM
        )
            AS MAXIMO,

        LOG10(
            MAX(
                VALOR_NUM
            )
            / MIN(
                VALOR_NUM
            )
        )
            AS ORDENS_MAGNITUDE,

        COUNT_IF(
            VALOR_CENTAVOS
            % 100 = 0
        )
        * 1.0
        / COUNT(*)
            AS PCT_INTEIROS,

        COUNT_IF(
            VALOR_CENTAVOS
            % 1000 = 0
        )
        * 1.0
        / COUNT(*)
            AS PCT_MULT_10,

        COUNT_IF(
            VALOR_CENTAVOS
            % 10000 = 0
        )
        * 1.0
        / COUNT(*)
            AS PCT_MULT_100

    FROM v_benford_compras
)

SELECT
    stats.*,

    top5.N_TOP5
    * 1.0
    / stats.N
        AS PCT_TOP5_VALORES

FROM
    stats,
    top5
""").df()

salvar_csv(
    elegibilidade_global_df,
    T08_DIR
    / '01_elegibilidade_global.csv'
)

display(
    elegibilidade_global_df
)

n_global = int(
    elegibilidade_global_df
    .loc[
        0,
        'N'
    ]
)

if (
    n_global
    < CONFIG['T08']
      ['min_n_nao_aplicar']
):
    status_global = (
        'NAO_APLICAR'
    )

elif (
    n_global
    < CONFIG['T08']
      ['min_n_formal']
):
    status_global = (
        'EXPLORATORIO'
    )

else:
    status_global = (
        'FORMAL'
    )

robustez_global = (
    'MAIOR'
    if (
        n_global
        >= CONFIG['T08']
           ['min_n_robusto']
    )
    else 'PADRAO'
)

print(
    '📌 Status de tamanho:',
    status_global
)

print(
    '📌 Robustez numérica:',
    robustez_global
)

## 2️⃣0️⃣ T08 — Benford global: D1 e D12

O **MAD** será a medida principal de conformidade.

Z e χ² são calculados como diagnósticos auxiliares.

D12 é executado de duas formas:

- **principal:** valores ≥ R$ 10;
- **sensibilidade:** todos os valores positivos.

### Sensibilidade a arredondamentos — versão 1.2

A análise é repetida em subpopulações sem valores inteiros e sem múltiplos exatos de R$ 5, R$ 10 e R$ 100. O objetivo **não é escolher a população que “melhor adere” a Benford**, mas medir quanto da divergência está associado ao *heaping* em valores arredondados.

In [ ]:
d1_counts = con.execute("""
SELECT
    D1
        AS DIGITO,

    COUNT(*)
        AS AC

FROM v_benford_compras

GROUP BY 1

ORDER BY 1
""").df()

d1_tabela = (
    completar_frequencias(
        d1_counts,
        'D1'
    )
)

d1_tabela = (
    adicionar_z(
        d1_tabela
    )
)

d1_resumo = (
    resumir_benford_tabela(
        d1_tabela,
        'D1'
    )
)

d1_resumo[
    'ANALISE'
] = 'GLOBAL_D1'

d1_resumo[
    'STATUS_TAMANHO'
] = status_global

salvar_csv(
    d1_tabela,
    T08_DIR
    / '02_global_d1_frequencias.csv'
)

salvar_csv(
    d1_resumo,
    T08_DIR
    / '03_global_d1_resumo.csv'
)


min_d12 = (
    CONFIG['T08']
    ['min_valor_d12']
)

d12_counts = con.execute(
    f"""
    SELECT
        D12
            AS DIGITO,

        COUNT(*)
            AS AC

    FROM v_benford_compras

    WHERE
        VALOR_NUM
        >= {min_d12}

    GROUP BY 1

    ORDER BY 1
    """
).df()

d12_tabela = (
    completar_frequencias(
        d12_counts,
        'D12'
    )
)

d12_tabela = (
    adicionar_z(
        d12_tabela
    )
)

d12_resumo = (
    resumir_benford_tabela(
        d12_tabela,
        'D12'
    )
)

d12_resumo[
    'ANALISE'
] = 'GLOBAL_D12_VALOR_GE_10'

d12_resumo[
    'STATUS_TAMANHO'
] = (
    'FORMAL'
    if (
        int(
            d12_resumo
            .loc[
                0,
                'N'
            ]
        )
        >= CONFIG['T08']
           ['min_n_formal']
    )
    else 'EXPLORATORIO'
)

salvar_csv(
    d12_tabela,
    T08_DIR
    / '04_global_d12_frequencias.csv'
)

salvar_csv(
    d12_resumo,
    T08_DIR
    / '05_global_d12_resumo.csv'
)


d12_pos_counts = con.execute("""
SELECT
    D12
        AS DIGITO,

    COUNT(*)
        AS AC

FROM v_benford_compras

GROUP BY 1

ORDER BY 1
""").df()

d12_pos_tabela = (
    completar_frequencias(
        d12_pos_counts,
        'D12'
    )
)

d12_pos_resumo = (
    resumir_benford_tabela(
        d12_pos_tabela,
        'D12'
    )
)

d12_pos_resumo[
    'ANALISE'
] = (
    'GLOBAL_D12_'
    'TODOS_POSITIVOS'
)

salvar_csv(
    d12_pos_tabela,
    T08_DIR
    / '06_global_d12_sensibilidade_positivos.csv'
)

salvar_csv(
    d12_pos_resumo,
    T08_DIR
    / '07_global_d12_sensibilidade_resumo.csv'
)

display(
    d1_resumo
)

display(
    d12_resumo
)

display(
    d12_pos_resumo
)

print(
    '✅ Benford global concluído.'
)


# ============================================================
# 🔄 SENSIBILIDADE D12 A ARREDONDAMENTOS
# ============================================================

def mad_d12_sql(rotulo, filtro_sql):
    cont = con.execute(f"""
        SELECT D12 AS DIGITO, COUNT(*) AS AC
        FROM v_benford_compras
        WHERE VALOR_NUM >= {CONFIG['T08']['min_valor_d12']}
          AND ({filtro_sql})
        GROUP BY 1 ORDER BY 1
    """).df()
    tab = completar_frequencias(cont, 'D12')
    res = resumir_benford_tabela(tab, 'D12')
    res['CENARIO'] = rotulo
    return res

cenarios_arred = [
    ('A_TODOS_GE10', 'TRUE'),
    ('B_CENTAVOS_NAO_ZERO', 'VALOR_CENTAVOS % 100 <> 0'),
    ('C_NAO_MULTIPLO_R5', 'VALOR_CENTAVOS % 500 <> 0'),
    ('D_NAO_MULTIPLO_R10', 'VALOR_CENTAVOS % 1000 <> 0'),
    ('E_NAO_MULTIPLO_R100', 'VALOR_CENTAVOS % 10000 <> 0'),
]

sens_arred = []
for rotulo, filtro in tqdm(cenarios_arred, desc='🔢 Benford — sensibilidade a arredondamentos', unit='cenário'):
    sens_arred.append(mad_d12_sql(rotulo, filtro))

benford_arredondamento_global_df = pd.concat(sens_arred, ignore_index=True)
salvar_csv(benford_arredondamento_global_df, T08_DIR / '07b_sensibilidade_arredondamento_global.csv')
display(benford_arredondamento_global_df)

## 2️⃣1️⃣ T08 — Gráficos observado × esperado

Os gráficos utilizam o padrão visual do Matplotlib sem impor cores fixas.

Eles serão exportados para a pasta `05_graficos`.

In [ ]:
def grafico_benford(
    tabela,
    tipo,
    titulo,
    caminho
):
    tabela = (
        tabela
        .sort_values(
            'DIGITO'
        )
    )

    plt.figure(
        figsize=(
            (10, 5.5)
            if tipo == 'D1'
            else (18, 5.5)
        )
    )

    x = np.arange(
        len(tabela)
    )

    plt.bar(
        x,
        tabela['AP'],
        label='Observado'
    )

    plt.plot(
        x,
        tabela[
            'PROB_ESPERADA'
        ],
        marker='o',
        linewidth=1.2,
        markersize=3,
        label='Esperado — Benford'
    )

    plt.xticks(
        x,
        tabela[
            'DIGITO'
        ].astype(
            str
        ),
        rotation=(
            90
            if tipo == 'D12'
            else 0
        ),
        fontsize=(
            7
            if tipo == 'D12'
            else 10
        )
    )

    plt.title(
        titulo
    )

    plt.xlabel(
        'Dígito significativo'
    )

    plt.ylabel(
        'Proporção'
    )

    plt.legend()

    salvar_fig(
        caminho
    )


if GERAR_GRAFICOS:

    grafico_benford(
        d1_tabela,
        'D1',
        (
            'CPGF — Compras nacionais '
            '— Primeiro dígito'
        ),
        GRAFICOS_DIR
        / 'benford_global_d1.png'
    )

    grafico_benford(
        d12_tabela,
        'D12',
        (
            'CPGF — Compras nacionais '
            '— Dois primeiros dígitos'
        ),
        GRAFICOS_DIR
        / 'benford_global_d12.png'
    )

print(
    '✅ Gráficos Benford globais gerados.'
)

## 2️⃣2️⃣ T08 — Robustez longitudinal por ano

A série anual utiliza o **ano da data efetiva da transação**.

Isso é diferente da competência do extrato e evita tratar ciclos de fatura como se fossem meses civis.

O resultado permite examinar se a conformidade digital é estável ao longo do tempo.

In [ ]:
d1_ano_counts = con.execute("""
SELECT
    ANO_TRANSACAO,

    D1
        AS DIGITO,

    COUNT(*)
        AS AC

FROM v_benford_compras

GROUP BY
    1,
    2

ORDER BY
    1,
    2
""").df()

d1_ano_tab = (
    completar_frequencias(
        d1_ano_counts,
        'D1',
        group_cols=[
            'ANO_TRANSACAO'
        ]
    )
)

d1_ano_resumo = (
    resumir_benford_tabela(
        d1_ano_tab,
        'D1',
        group_cols=[
            'ANO_TRANSACAO'
        ]
    )
    .rename(
        columns={
            'N':
                'N_D1',
            'MAD':
                'MAD_D1',
            'CLASSIFICACAO_MAD':
                'CLASSIFICACAO_D1',
            'CHI2':
                'CHI2_D1',
        }
    )
)


d12_ano_counts = con.execute(
    f"""
    SELECT
        ANO_TRANSACAO,

        D12
            AS DIGITO,

        COUNT(*)
            AS AC

    FROM v_benford_compras

    WHERE
        VALOR_NUM
        >= {CONFIG['T08']['min_valor_d12']}

    GROUP BY
        1,
        2

    ORDER BY
        1,
        2
    """
).df()

d12_ano_tab = (
    completar_frequencias(
        d12_ano_counts,
        'D12',
        group_cols=[
            'ANO_TRANSACAO'
        ]
    )
)

d12_ano_resumo = (
    resumir_benford_tabela(
        d12_ano_tab,
        'D12',
        group_cols=[
            'ANO_TRANSACAO'
        ]
    )
    .rename(
        columns={
            'N':
                'N_D12',
            'MAD':
                'MAD_D12',
            'CLASSIFICACAO_MAD':
                'CLASSIFICACAO_D12',
            'CHI2':
                'CHI2_D12',
        }
    )
)


benford_anual_df = (
    d1_ano_resumo
    .merge(
        d12_ano_resumo,
        on='ANO_TRANSACAO',
        how='outer'
    )
    .sort_values(
        'ANO_TRANSACAO'
    )
)

salvar_csv(
    benford_anual_df,
    T08_DIR
    / '08_benford_anual_resumo.csv'
)

display(
    benford_anual_df
)

if (
    GERAR_GRAFICOS
    and len(
        benford_anual_df
    )
):
    plt.figure(
        figsize=(13, 5.5)
    )

    plt.plot(
        benford_anual_df[
            'ANO_TRANSACAO'
        ],
        benford_anual_df[
            'MAD_D1'
        ],
        marker='o',
        label='MAD — D1'
    )

    plt.axhline(
        0.006,
        linestyle='--',
        label=(
            'D1 — limite '
            'conformidade próxima'
        )
    )

    plt.axhline(
        0.015,
        linestyle=':',
        label=(
            'D1 — não conformidade '
            'acima'
        )
    )

    plt.title(
        (
            'CPGF — MAD do primeiro '
            'dígito ao longo do tempo'
        )
    )

    plt.xlabel(
        'Ano da transação'
    )

    plt.ylabel(
        'MAD'
    )

    plt.legend()

    salvar_fig(
        GRAFICOS_DIR
        / 'benford_mad_d1_anual.png'
    )


    plt.figure(
        figsize=(13, 5.5)
    )

    plt.plot(
        benford_anual_df[
            'ANO_TRANSACAO'
        ],
        benford_anual_df[
            'MAD_D12'
        ],
        marker='o',
        label='MAD — D12'
    )

    plt.axhline(
        0.0012,
        linestyle='--',
        label=(
            'D12 — limite '
            'conformidade próxima'
        )
    )

    plt.axhline(
        0.0022,
        linestyle=':',
        label=(
            'D12 — não conformidade '
            'acima'
        )
    )

    plt.title(
        (
            'CPGF — MAD dos dois primeiros '
            'dígitos ao longo do tempo'
        )
    )

    plt.xlabel(
        'Ano da transação'
    )

    plt.ylabel(
        'MAD'
    )

    plt.legend()

    salvar_fig(
        GRAFICOS_DIR
        / 'benford_mad_d12_anual.png'
    )

print(
    '✅ Benford longitudinal anual concluído.'
)


# Marcação explícita de exercícios completos/parciais
benford_anual_df['PERIODO_COMPLETO'] = benford_anual_df['ANO_TRANSACAO'].between(
    CONFIG['T08']['anos_completos_inicio'],
    CONFIG['T08']['anos_completos_fim']
)

benford_anual_df['STATUS_PERIODO'] = np.where(
    benford_anual_df['PERIODO_COMPLETO'],
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

# Sensibilidade anual a arredondamentos — somente D12
sens_anual = []
for ano in tqdm(sorted(benford_anual_df['ANO_TRANSACAO'].dropna().astype(int).unique()), desc='📆 Benford — arredondamento anual', unit='ano'):
    for rotulo, filtro in cenarios_arred:
        cont = con.execute(f"""
            SELECT D12 AS DIGITO, COUNT(*) AS AC
            FROM v_benford_compras
            WHERE ANO_TRANSACAO = {ano}
              AND VALOR_NUM >= {CONFIG['T08']['min_valor_d12']}
              AND ({filtro})
            GROUP BY 1 ORDER BY 1
        """).df()
        if cont['AC'].sum() == 0:
            continue
        tab = completar_frequencias(cont, 'D12')
        res = resumir_benford_tabela(tab, 'D12')
        res['ANO_TRANSACAO'] = ano
        res['CENARIO'] = rotulo
        sens_anual.append(res)

benford_arredondamento_anual_df = pd.concat(sens_anual, ignore_index=True) if sens_anual else pd.DataFrame()
salvar_csv(benford_arredondamento_anual_df, T08_DIR / '08b_sensibilidade_arredondamento_anual.csv')

salvar_csv(benford_anual_df, T08_DIR / '08_benford_anual_resumo.csv')

## 2️⃣3️⃣ T08 — Benford por Unidade Gestora e exercício

Para cada `UG × ano`, calcula-se:

- tamanho da população;
- D1;
- D12;
- MAD;
- classificação;
- condição de uso: não aplicar / exploratório / formal / robustez maior.

A análise **não aplica Benford abaixo de 300 observações**.

Resultados entre 300 e 999 observações permanecem exploratórios.

In [ ]:
eleg_ug_ano_df = con.execute(
    f"""
    SELECT
        UG_ID,
        ANO_TRANSACAO,

        COUNT(*)
            AS N_D1,

        COUNT_IF(
            VALOR_NUM
            >= {CONFIG['T08']['min_valor_d12']}
        )
            AS N_D12,

        COUNT(
            DISTINCT VALOR_CENTAVOS
        )
            AS VALORES_UNICOS,

        MIN(
            VALOR_NUM
        )
            AS MINIMO,

        MEDIAN(
            VALOR_NUM
        )
            AS MEDIANA,

        AVG(
            VALOR_NUM
        )
            AS MEDIA,

        MAX(
            VALOR_NUM
        )
            AS MAXIMO,

        SKEWNESS(
            VALOR_NUM
        )
            AS ASSIMETRIA,

        CASE
            WHEN
                MIN(
                    VALOR_NUM
                ) > 0
                AND MAX(
                    VALOR_NUM
                ) > MIN(
                    VALOR_NUM
                )

            THEN LOG10(
                MAX(
                    VALOR_NUM
                )
                / MIN(
                    VALOR_NUM
                )
            )

            ELSE NULL
        END
            AS ORDENS_MAGNITUDE

    FROM v_benford_compras

    GROUP BY
        1,
        2
    """
).df()


def status_n(
    n
):
    if (
        n
        < CONFIG['T08']
          ['min_n_nao_aplicar']
    ):
        return (
            'NAO_APLICAR'
        )

    if (
        n
        < CONFIG['T08']
          ['min_n_formal']
    ):
        return (
            'EXPLORATORIO'
        )

    if (
        n
        >= CONFIG['T08']
           ['min_n_robusto']
    ):
        return (
            'FORMAL_ROBUSTEZ_MAIOR'
        )

    return (
        'FORMAL'
    )


eleg_ug_ano_df[
    'STATUS_D1'
] = (
    eleg_ug_ano_df[
        'N_D1'
    ]
    .apply(
        status_n
    )
)

eleg_ug_ano_df[
    'STATUS_D12'
] = (
    eleg_ug_ano_df[
        'N_D12'
    ]
    .apply(
        status_n
    )
)

salvar_csv(
    eleg_ug_ano_df,
    T08_DIR
    / '09_elegibilidade_ug_ano.csv'
)


d1_ug_counts = con.execute(
    f"""
    WITH validos AS (
        SELECT
            UG_ID,
            ANO_TRANSACAO

        FROM v_benford_compras

        GROUP BY
            1,
            2

        HAVING
            COUNT(*)
            >= {CONFIG['T08']['min_n_nao_aplicar']}
    )

    SELECT
        b.UG_ID,
        b.ANO_TRANSACAO,

        b.D1
            AS DIGITO,

        COUNT(*)
            AS AC

    FROM v_benford_compras b

    JOIN validos v
      ON v.UG_ID
            = b.UG_ID
     AND v.ANO_TRANSACAO
            = b.ANO_TRANSACAO

    GROUP BY
        1,
        2,
        3
    """
).df()

d1_ug_tab = (
    completar_frequencias(
        d1_ug_counts,
        'D1',
        group_cols=[
            'UG_ID',
            'ANO_TRANSACAO'
        ]
    )
)

d1_ug_resumo = (
    resumir_benford_tabela(
        d1_ug_tab,
        'D1',
        group_cols=[
            'UG_ID',
            'ANO_TRANSACAO'
        ]
    )
    .rename(
        columns={
            'N':
                'N_D1_CALC',
            'MAD':
                'MAD_D1',
            'CLASSIFICACAO_MAD':
                'CLASSIFICACAO_D1',
            'CHI2':
                'CHI2_D1',
        }
    )
)


d12_ug_counts = con.execute(
    f"""
    WITH validos AS (
        SELECT
            UG_ID,
            ANO_TRANSACAO

        FROM v_benford_compras

        WHERE
            VALOR_NUM
            >= {CONFIG['T08']['min_valor_d12']}

        GROUP BY
            1,
            2

        HAVING
            COUNT(*)
            >= {CONFIG['T08']['min_n_nao_aplicar']}
    )

    SELECT
        b.UG_ID,
        b.ANO_TRANSACAO,

        b.D12
            AS DIGITO,

        COUNT(*)
            AS AC

    FROM v_benford_compras b

    JOIN validos v
      ON v.UG_ID
            = b.UG_ID
     AND v.ANO_TRANSACAO
            = b.ANO_TRANSACAO

    WHERE
        b.VALOR_NUM
        >= {CONFIG['T08']['min_valor_d12']}

    GROUP BY
        1,
        2,
        3
    """
).df()

d12_ug_tab = (
    completar_frequencias(
        d12_ug_counts,
        'D12',
        group_cols=[
            'UG_ID',
            'ANO_TRANSACAO'
        ]
    )
)

d12_ug_resumo = (
    resumir_benford_tabela(
        d12_ug_tab,
        'D12',
        group_cols=[
            'UG_ID',
            'ANO_TRANSACAO'
        ]
    )
    .rename(
        columns={
            'N':
                'N_D12_CALC',
            'MAD':
                'MAD_D12',
            'CLASSIFICACAO_MAD':
                'CLASSIFICACAO_D12',
            'CHI2':
                'CHI2_D12',
        }
    )
)


benford_ug_ano_df = (
    eleg_ug_ano_df
    .merge(
        d1_ug_resumo,
        on=[
            'UG_ID',
            'ANO_TRANSACAO'
        ],
        how='left'
    )
    .merge(
        d12_ug_resumo,
        on=[
            'UG_ID',
            'ANO_TRANSACAO'
        ],
        how='left'
    )
)

salvar_parquet(
    benford_ug_ano_df,
    T08_DIR
    / '10_benford_ug_ano.parquet'
)

salvar_csv(
    benford_ug_ano_df,
    T08_DIR
    / '10_benford_ug_ano.csv'
)

print(
    '✅ UG-anos avaliados:',
    f'{len(benford_ug_ano_df):,}'
)

display(
    benford_ug_ano_df
    .sort_values(
        [
            'MAD_D12',
            'N_D12'
        ],
        ascending=[
            False,
            False
        ],
        na_position='last'
    )
    .head(30)
)

## 2️⃣4️⃣ T08 — Persistência relativa por UG

A versão 1.0 mostrou que a não conformidade absoluta de D12 (`MAD > 0,0022`) ocorria em praticamente todas as UGs formalmente elegíveis. Esse critério, portanto, não discriminava prioridades.

A versão 1.2 preserva a classificação absoluta como **diagnóstico**, mas a priorização institucional usa uma medida relativa:

1. somente `UG × ano` com `N_D12 ≥ 1.000`;
2. somente exercícios completos entre 2013 e 2025;
3. o ranking anual só é válido quando há pelo menos **10 UGs comparáveis**;
4. a UG é marcada como extremo relativo quando está no **decil superior do MAD D12** naquele exercício;
5. a persistência é a proporção de anos comparáveis em que a UG esteve nesse decil.

Esse ranking não transforma Benford em prova de fraude; apenas identifica divergências mais pronunciadas em comparação com pares do mesmo exercício.

In [ ]:
formal = benford_ug_ano_df[
    (benford_ug_ano_df['N_D12'] >= CONFIG['T08']['min_n_formal']) &
    (benford_ug_ano_df['ANO_TRANSACAO'].between(CONFIG['T08']['anos_completos_inicio'], CONFIG['T08']['anos_completos_fim']))
].copy()

formal['N_UG_COMPARAVEIS_ANO'] = formal.groupby('ANO_TRANSACAO')['UG_ID'].transform('count')
formal['PERCENT_RANK_MAD_D12'] = formal.groupby('ANO_TRANSACAO')['MAD_D12'].rank(pct=True, method='average')
formal['LIMIAR_P90_MAD_D12'] = formal.groupby('ANO_TRANSACAO')['MAD_D12'].transform(lambda s: s.quantile(0.90))
formal['ANO_RANK_VALIDO'] = formal['N_UG_COMPARAVEIS_ANO'] >= CONFIG['T08']['min_ugs_comparaveis']
formal['TOP_DECIL_MAD_D12'] = formal['ANO_RANK_VALIDO'] & (formal['MAD_D12'] >= formal['LIMIAR_P90_MAD_D12'])
formal['NAO_CONFORME_D12_ABSOLUTO'] = formal['MAD_D12'] > 0.0022

benford_extremos_relativos_df = formal[formal['TOP_DECIL_MAD_D12']].copy()
salvar_csv(benford_extremos_relativos_df, T08_DIR / '11a_benford_extremos_relativos_ug_ano.csv')

validos = formal[formal['ANO_RANK_VALIDO']].copy()
benford_persistencia_df = (
    validos.groupby('UG_ID', as_index=False)
    .agg(
        ANOS_COMPARAVEIS=('ANO_TRANSACAO','nunique'),
        ANOS_TOP_DECIL=('TOP_DECIL_MAD_D12','sum'),
        PRIMEIRO_ANO=('ANO_TRANSACAO','min'),
        ULTIMO_ANO=('ANO_TRANSACAO','max'),
        MAD_D12_MEDIO=('MAD_D12','mean'),
        MAD_D12_MAX=('MAD_D12','max'),
    )
)

benford_persistencia_df['RATIO_TOP_DECIL'] = (
    benford_persistencia_df['ANOS_TOP_DECIL'] /
    benford_persistencia_df['ANOS_COMPARAVEIS'].replace(0,np.nan)
)

benford_persistencia_df['PERSISTENCIA_RELATIVA_ELEVADA'] = (
    (benford_persistencia_df['ANOS_COMPARAVEIS'] >= CONFIG['T08']['persistencia_min_anos']) &
    (benford_persistencia_df['RATIO_TOP_DECIL'] >= CONFIG['T08']['persistencia_min_ratio'])
)

benford_persistencia_df = benford_persistencia_df.sort_values(
    ['PERSISTENCIA_RELATIVA_ELEVADA','RATIO_TOP_DECIL','ANOS_TOP_DECIL','MAD_D12_MAX'],
    ascending=[False,False,False,False]
)

salvar_csv(benford_persistencia_df, T08_DIR / '11_benford_persistencia_relativa_ug.csv')
display(formal.groupby('ANO_TRANSACAO', as_index=False).agg(N_UG=('UG_ID','count'), RANK_VALIDO=('ANO_RANK_VALIDO','max')))
display(benford_persistencia_df.head(40))
print('📌 Extremos relativos UG-ano:', len(benford_extremos_relativos_df))
print('📌 UGs com persistência relativa elevada:', int(benford_persistencia_df['PERSISTENCIA_RELATIVA_ELEVADA'].sum()) if len(benford_persistencia_df) else 0)

## 2️⃣5️⃣ T08 — Summation Test

Para cada combinação D12, calcula-se a participação daquele dígito na soma financeira total.

A referência ideal é:

`1 / 90 ≈ 0,01111`

O notebook **não cria automaticamente um limite arbitrário de anomalia** para o Summation Test.

As combinações são ordenadas por desvio em relação à referência para apoiar o *drill-down*.

In [ ]:
summation_df = con.execute(
    f"""
    SELECT
        D12
            AS DIGITO,

        COUNT(*)
            AS FREQUENCIA,

        SUM(
            VALOR_NUM
        )
            AS SOMA_VALORES

    FROM v_benford_compras

    WHERE
        VALOR_NUM
        >= {CONFIG['T08']['min_valor_d12']}

    GROUP BY 1

    ORDER BY 1
    """
).df()

todos_d12 = pd.DataFrame({
    'DIGITO':
        np.arange(
            10,
            100
        )
})

summation_df = (
    todos_d12
    .merge(
        summation_df,
        on='DIGITO',
        how='left'
    )
    .fillna(
        {
            'FREQUENCIA':
                0,
            'SOMA_VALORES':
                0,
        }
    )
)

total_soma = (
    summation_df[
        'SOMA_VALORES'
    ].sum()
)

summation_df[
    'SHARE_SOMA'
] = (
    summation_df[
        'SOMA_VALORES'
    ]
    / total_soma
    if total_soma
    else 0
)

summation_df[
    'ESPERADO'
] = SOMATORIO_ESPERADO

summation_df[
    'DESVIO_SOMA'
] = (
    summation_df[
        'SHARE_SOMA'
    ]
    - SOMATORIO_ESPERADO
)

salvar_csv(
    summation_df
    .sort_values(
        'DESVIO_SOMA',
        ascending=False
    ),
    T08_DIR
    / '12_summation_global.csv'
)

display(
    summation_df
    .sort_values(
        'DESVIO_SOMA',
        ascending=False
    )
    .head(20)
)

if GERAR_GRAFICOS:

    s = (
        summation_df
        .sort_values(
            'DIGITO'
        )
    )

    plt.figure(
        figsize=(18, 5.5)
    )

    x = np.arange(
        len(s)
    )

    plt.bar(
        x,
        s[
            'SHARE_SOMA'
        ],
        label='Participação observada'
    )

    plt.axhline(
        SOMATORIO_ESPERADO,
        linestyle='--',
        label='Referência 1/90'
    )

    plt.xticks(
        x,
        s[
            'DIGITO'
        ].astype(
            str
        ),
        rotation=90,
        fontsize=7
    )

    plt.title(
        (
            'CPGF — Summation Test '
            '— compras nacionais'
        )
    )

    plt.xlabel(
        'Dois primeiros dígitos'
    )

    plt.ylabel(
        'Participação na soma'
    )

    plt.legend()

    salvar_fig(
        GRAFICOS_DIR
        / 'benford_summation_global.png'
    )

print(
    '✅ Summation Test concluído.'
)

## 2️⃣6️⃣ T08 — Number Duplication Test

O teste de duplicação ordena os valores monetários exatos pela frequência observada.

Ele funciona como ponte entre:

**pico digital → valor responsável → transações → UG/fornecedor/portador**

Valores arredondados e *price points* podem ser perfeitamente naturais. Portanto, repetição numérica só ganha relevância quando interpretada em conjunto com outras evidências.

In [ ]:
NUMBER_DUP_PARQUET = (
    T08_DIR
    / '13_number_duplication.parquet'
)

number_dup_sql = (
    str(NUMBER_DUP_PARQUET)
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        SELECT
            VALOR_NUM,
            VALOR_CENTAVOS,

            D12
                AS DIGITO_D12,

            COUNT(*)
                AS FREQUENCIA,

            SUM(
                VALOR_NUM
            )
                AS VALOR_TOTAL

        FROM v_benford_compras

        GROUP BY
            1,
            2,
            3

        HAVING
            COUNT(*) >= 2

        ORDER BY
            FREQUENCIA DESC,
            VALOR_TOTAL DESC
    )
    TO '{number_dup_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

number_dup_top_df = con.execute(
    f"""
    SELECT *

    FROM read_parquet(
        '{number_dup_sql}'
    )

    ORDER BY
        FREQUENCIA DESC,
        VALOR_TOTAL DESC

    LIMIT 10000
    """
).df()

salvar_csv(
    number_dup_top_df,
    T08_DIR
    / '13_number_duplication_top10000.csv'
)

display(
    number_dup_top_df
    .head(30)
)

print(
    '✅ Number Duplication Test concluído.'
)

## 2️⃣7️⃣ T08 — Drill-down dos dígitos e valores

A seleção para aprofundamento combina:

- maiores desvios positivos de D12;
- maiores desvios positivos do Summation Test.

Para cada dígito selecionado, são escolhidos os valores exatos mais relevantes segundo frequência e materialidade.

Em seguida, o notebook retorna às transações originais.

In [ ]:
top_n_digitos = (
    CONFIG['T08']
    ['top_digitos_drilldown']
)

top_n_valores = (
    CONFIG['T08']
    ['top_valores_por_digito']
)

top_freq = (
    d12_tabela
    .sort_values(
        'DIFERENCA',
        ascending=False
    )
    .head(
        top_n_digitos
    )[
        'DIGITO'
    ]
    .astype(
        int
    )
    .tolist()
)

top_soma = (
    summation_df
    .sort_values(
        'DESVIO_SOMA',
        ascending=False
    )
    .head(
        top_n_digitos
    )[
        'DIGITO'
    ]
    .astype(
        int
    )
    .tolist()
)

digitos_drilldown = sorted(
    set(
        top_freq
        + top_soma
    )
)

print(
    '🔎 Dígitos selecionados:',
    digitos_drilldown
)

dup_drill = (
    number_dup_top_df[
        number_dup_top_df[
            'DIGITO_D12'
        ]
        .isin(
            digitos_drilldown
        )
    ]
    .copy()
)

valores_selecionados = (
    dup_drill
    .sort_values(
        [
            'DIGITO_D12',
            'FREQUENCIA',
            'VALOR_TOTAL'
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .groupby(
        'DIGITO_D12',
        group_keys=False
    )
    .head(
        top_n_valores
    )
)

salvar_csv(
    valores_selecionados,
    T08_DIR
    / '14_drilldown_valores_selecionados.csv'
)

if len(
    valores_selecionados
):

    lista_centavos = ','.join(
        str(
            int(x)
        )
        for x in (
            valores_selecionados[
                'VALOR_CENTAVOS'
            ]
            .dropna()
            .unique()
        )
    )

    DRILL_TRANSACOES = (
        T08_DIR
        / '15_drilldown_transacoes.parquet'
    )

    drill_path_sql = (
        str(DRILL_TRANSACOES)
        .replace("'", "''")
    )

    con.execute(
        f"""
        COPY (
            SELECT
                ID_TRANSACAO,
                UG_ID,

                "NOME UNIDADE GESTORA"
                    AS NOME_UG,

                PORTADOR_ID,

                "NOME PORTADOR"
                    AS NOME_PORTADOR,

                FAVORECIDO_ID,

                "NOME FAVORECIDO"
                    AS NOME_FAVORECIDO,

                DATA_DT,
                ANO_TRANSACAO,
                VALOR_NUM,
                VALOR_CENTAVOS,
                D12,
                COMPETENCIA_ARQUIVO,
                ARQUIVO_ORIGEM

            FROM v_benford_compras

            WHERE
                VALOR_CENTAVOS
                IN (
                    {lista_centavos}
                )
        )
        TO '{drill_path_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    drill_resumo_df = con.execute(
        f"""
        SELECT
            D12,
            VALOR_NUM,

            COUNT(*)
                AS N,

            COUNT(
                DISTINCT UG_ID
            )
                AS N_UG,

            COUNT(
                DISTINCT FAVORECIDO_ID
            )
                AS N_FORNECEDORES,

            COUNT(
                DISTINCT PORTADOR_ID
            )
                AS N_PORTADORES

        FROM read_parquet(
            '{drill_path_sql}'
        )

        GROUP BY
            1,
            2

        ORDER BY
            N DESC
        """
    ).df()

    salvar_csv(
        drill_resumo_df,
        T08_DIR
        / '15_drilldown_resumo.csv'
    )

    display(
        drill_resumo_df
        .head(40)
    )

else:
    print(
        'ℹ️ Nenhum valor duplicado '
        'selecionado para drill-down.'
    )

print(
    '✅ Drill-down concluído.'
)

## 2️⃣8️⃣ T09 — Proximidade, igualdade e ultrapassagem de referências financeiras

### ✅ Status da versão 1.2: executada integralmente

A T09 continua sendo executada em **dois cenários normativos paralelos** para cada transação:

- `COMPRAS_SERVICOS`;
- `OBRAS_ENGENHARIA`.

A categoria real continua não observável no CSV público. Portanto, nenhum dos dois cenários é escolhido automaticamente como enquadramento jurídico verdadeiro.

### 🆕 Comparação monetária em centavos inteiros

Na V1.2, o valor da transação e as referências normativas são comparados em **centavos inteiros**.

Para cada cenário, existem quatro estados distintos:

- `ABAIXO_FAIXA` — abaixo de 90% da referência;
- `PROXIMO_LIMITE` — entre 90% e menos de 100%;
- `NO_LIMITE` — exatamente igual à referência;
- `ACIMA_LIMITE` — estritamente superior à referência.

Essa separação corrige a semântica da V1.1, na qual `igual` e `acima` apareciam na mesma classe.

### Classificação combinada

A coluna `STATUS_T09` é uma síntese para triagem:

- `ACIMA_AMBOS_CENARIOS`;
- `ACIMA_PELO_MENOS_UM_CENARIO`;
- `NO_LIMITE_PELO_MENOS_UM_CENARIO`;
- `PROXIMO_LIMITE`;
- `ABAIXO_FAIXAS`.

As colunas `STATUS_COMPRAS` e `STATUS_ENGENHARIA` devem ser consideradas a informação primária.

> Estar no limite, próximo ou acima de uma referência **não confirma irregularidade**. O enquadramento depende do objeto da despesa, do ato de concessão e do contexto administrativo.

### 🔗 Camada agregada

O agregado `UG × fornecedor × ano` permanece descritivo. Ele não soma operações para concluir fracionamento; apenas mostra quantas transações se situaram em cada faixa e qual materialidade foi observada.

In [ ]:
assert CONFIG['T09']['habilitado_automaticamente'] is True
assert CONFIG['T09']['classificar_no_limite_separadamente'] is True

DIM_NORMAS_LIMITES = T09_DIR / 'dim_normas_limites.csv'
T09_CENARIOS = T09_DIR / 't09_transacoes_cenarios.parquet'
T09_CLASSIFICADAS = T09_DIR / 't09_transacoes_classificadas.parquet'
T09_SINAIS = T09_DIR / 't09_sinais.parquet'

T09_RESUMO = T09_DIR / 't09_resumo_anual.csv'
T09_RESUMO_CENARIOS = T09_DIR / 't09_resumo_anual_por_cenario.csv'


# ------------------------------------------------------------
# 📚 DIMENSÃO NORMATIVA VERSIONADA
# ------------------------------------------------------------

periodos = [
    # Portaria MF 95/2002 + valores originais da Lei 8.666/1993
    (
        '2002-04-23','2018-07-18',
        'Portaria MF 95/2002 + Lei 8.666/1993',
        'COMPRAS_SERVICOS',
        80000.00,0.10,0.01,'art. 23, II, a'
    ),
    (
        '2002-04-23','2018-07-18',
        'Portaria MF 95/2002 + Lei 8.666/1993',
        'OBRAS_ENGENHARIA',
        150000.00,0.10,0.01,'art. 23, I, a'
    ),

    # Decreto 9.412/2018 — vigência 30 dias após publicação
    (
        '2018-07-19','2023-11-30',
        'Portaria MF 95/2002 + Decreto 9.412/2018',
        'COMPRAS_SERVICOS',
        176000.00,0.10,0.01,'art. 23, II, a'
    ),
    (
        '2018-07-19','2023-11-30',
        'Portaria MF 95/2002 + Decreto 9.412/2018',
        'OBRAS_ENGENHARIA',
        330000.00,0.10,0.01,'art. 23, I, a'
    ),

    # Portaria Normativa MF 1.344/2023
    (
        '2023-12-01','2023-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 11.317/2022',
        'COMPRAS_SERVICOS',
        57208.33,0.50,0.05,'art. 75, II'
    ),
    (
        '2023-12-01','2023-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 11.317/2022',
        'OBRAS_ENGENHARIA',
        114416.65,0.50,0.05,'art. 75, I'
    ),

    (
        '2024-01-01','2024-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 11.871/2023',
        'COMPRAS_SERVICOS',
        59906.02,0.50,0.05,'art. 75, II'
    ),
    (
        '2024-01-01','2024-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 11.871/2023',
        'OBRAS_ENGENHARIA',
        119812.02,0.50,0.05,'art. 75, I'
    ),

    (
        '2025-01-01','2025-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 12.343/2024',
        'COMPRAS_SERVICOS',
        62725.59,0.50,0.05,'art. 75, II'
    ),
    (
        '2025-01-01','2025-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 12.343/2024',
        'OBRAS_ENGENHARIA',
        125451.15,0.50,0.05,'art. 75, I'
    ),

    (
        '2026-01-01','2026-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 12.807/2025',
        'COMPRAS_SERVICOS',
        65492.11,0.50,0.05,'art. 75, II'
    ),
    (
        '2026-01-01','2026-12-31',
        'Portaria Normativa MF 1.344/2023 + Decreto 12.807/2025',
        'OBRAS_ENGENHARIA',
        130984.20,0.50,0.05,'art. 75, I'
    ),
]


normas = []

for (
    ini,
    fim,
    norma,
    categoria,
    ref,
    pct_conc,
    pct_pv,
    dispositivo
) in tqdm(
    periodos,
    desc='⚖️ T09 — montando dimensão normativa',
    unit='regra'
):

    limite_concessao = round(ref * pct_conc, 2)
    limite_pv = round(ref * pct_pv, 2)

    normas.append({
        'VIGENCIA_INICIO':
            ini,

        'VIGENCIA_FIM':
            fim,

        'NORMA_REFERENCIA':
            norma,

        'CATEGORIA_CENARIO':
            categoria,

        'DISPOSITIVO_REFERENCIA':
            dispositivo,

        'VALOR_BASE_REFERENCIA':
            round(ref, 2),

        'PERCENTUAL_CONCESSAO_CPGF':
            pct_conc,

        'PERCENTUAL_PEQUENO_VULTO_CPGF':
            pct_pv,

        'LIMITE_CONCESSAO_CPGF':
            limite_concessao,

        'LIMITE_PEQUENO_VULTO_CPGF':
            limite_pv,

        'LIMITE_CONCESSAO_CENTAVOS':
            int(round(limite_concessao * 100)),

        'LIMITE_PEQUENO_VULTO_CENTAVOS':
            int(round(limite_pv * 100)),

        'APLICABILIDADE':
            'CENARIO_NAO_CLASSIFICADO_PELA_BASE_PUBLICA',
    })


dim_normas_limites_df = pd.DataFrame(normas)

salvar_csv(
    dim_normas_limites_df,
    DIM_NORMAS_LIMITES
)

display(
    dim_normas_limites_df
)


con.register(
    'dim_normas_t09_python',

    dim_normas_limites_df.assign(
        VIGENCIA_INICIO=pd.to_datetime(
            dim_normas_limites_df['VIGENCIA_INICIO']
        ).dt.date,

        VIGENCIA_FIM=pd.to_datetime(
            dim_normas_limites_df['VIGENCIA_FIM']
        ).dt.date,
    )
)


# ------------------------------------------------------------
# 🔎 EXECUÇÃO POR ANO — barra de progresso
# ------------------------------------------------------------

con.execute(
    "DROP TABLE IF EXISTS t09_cenarios_tmp"
)

con.execute("""
CREATE TEMP TABLE t09_cenarios_tmp AS

SELECT
    CAST(NULL AS VARCHAR) AS ID_TRANSACAO,
    CAST(NULL AS VARCHAR) AS UG_ID,
    CAST(NULL AS VARCHAR) AS FAVORECIDO_ID,

    CAST(NULL AS DATE) AS DATA_DT,
    CAST(NULL AS INTEGER) AS ANO_TRANSACAO,

    CAST(NULL AS DOUBLE) AS VALOR_NUM,
    CAST(NULL AS BIGINT) AS VALOR_CENTAVOS,

    CAST(NULL AS VARCHAR) AS CATEGORIA_CENARIO,
    CAST(NULL AS VARCHAR) AS NORMA_REFERENCIA,
    CAST(NULL AS VARCHAR) AS DISPOSITIVO_REFERENCIA,

    CAST(NULL AS DOUBLE) AS VALOR_BASE_REFERENCIA,

    CAST(NULL AS DOUBLE) AS LIMITE_CONCESSAO_CPGF,
    CAST(NULL AS DOUBLE) AS LIMITE_PEQUENO_VULTO_CPGF,

    CAST(NULL AS BIGINT) AS LIMITE_CONCESSAO_CENTAVOS,
    CAST(NULL AS BIGINT) AS LIMITE_PEQUENO_VULTO_CENTAVOS,

    CAST(NULL AS DOUBLE) AS RATIO_PEQUENO_VULTO,
    CAST(NULL AS DOUBLE) AS RATIO_CONCESSAO_CONTEXTO,

    CAST(NULL AS VARCHAR) AS STATUS_CENARIO

WHERE FALSE
""")


anos_t09 = (
    con.execute("""
        SELECT DISTINCT
            ANO_TRANSACAO

        FROM stg_cpgf_transacoes

        WHERE EH_COMPRA_NACIONAL
          AND DATA_DT IS NOT NULL
          AND VALOR_NUM > 0

        ORDER BY 1
    """)
    .df()['ANO_TRANSACAO']
    .dropna()
    .astype(int)
    .tolist()
)


for ano in tqdm(
    anos_t09,
    desc='⚖️ T09 — comparando limites',
    unit='ano'
):

    con.execute(
        f"""
        INSERT INTO t09_cenarios_tmp

        SELECT
            t.ID_TRANSACAO,
            t.UG_ID,
            t.FAVORECIDO_ID,

            t.DATA_DT,
            t.ANO_TRANSACAO,

            CAST(t.VALOR_NUM AS DOUBLE),
            t.VALOR_CENTAVOS,

            n.CATEGORIA_CENARIO,
            n.NORMA_REFERENCIA,
            n.DISPOSITIVO_REFERENCIA,

            n.VALOR_BASE_REFERENCIA,

            n.LIMITE_CONCESSAO_CPGF,
            n.LIMITE_PEQUENO_VULTO_CPGF,

            n.LIMITE_CONCESSAO_CENTAVOS,
            n.LIMITE_PEQUENO_VULTO_CENTAVOS,

            t.VALOR_CENTAVOS
            * 1.0
            / NULLIF(
                n.LIMITE_PEQUENO_VULTO_CENTAVOS,
                0
            ) AS RATIO_PEQUENO_VULTO,

            t.VALOR_CENTAVOS
            * 1.0
            / NULLIF(
                n.LIMITE_CONCESSAO_CENTAVOS,
                0
            ) AS RATIO_CONCESSAO_CONTEXTO,

            CASE
                WHEN t.VALOR_CENTAVOS
                        > n.LIMITE_PEQUENO_VULTO_CENTAVOS
                    THEN 'ACIMA_LIMITE'

                WHEN t.VALOR_CENTAVOS
                        = n.LIMITE_PEQUENO_VULTO_CENTAVOS
                    THEN 'NO_LIMITE'

                WHEN t.VALOR_CENTAVOS
                        * 1.0
                        / NULLIF(
                            n.LIMITE_PEQUENO_VULTO_CENTAVOS,
                            0
                        )
                        >= {CONFIG['T09']['faixa_inferior']}
                    THEN 'PROXIMO_LIMITE'

                ELSE 'ABAIXO_FAIXA'
            END AS STATUS_CENARIO

        FROM stg_cpgf_transacoes t

        JOIN dim_normas_t09_python n
          ON t.DATA_DT
             BETWEEN n.VIGENCIA_INICIO
                 AND n.VIGENCIA_FIM

        WHERE t.EH_COMPRA_NACIONAL
          AND t.VALOR_NUM > 0
          AND t.ANO_TRANSACAO = {ano}
        """
    )


t09_cenarios_path_sql = (
    str(T09_CENARIOS)
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        SELECT *
        FROM t09_cenarios_tmp
    )
    TO '{t09_cenarios_path_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ------------------------------------------------------------
# 🧭 CLASSIFICAÇÃO ÚNICA POR TRANSAÇÃO
# ------------------------------------------------------------

t09_classificadas_path_sql = (
    str(T09_CLASSIFICADAS)
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        WITH p AS (

            SELECT
                ID_TRANSACAO,
                UG_ID,
                FAVORECIDO_ID,
                DATA_DT,
                ANO_TRANSACAO,

                MAX(VALOR_NUM)
                    AS VALOR_NUM,

                MAX(VALOR_CENTAVOS)
                    AS VALOR_CENTAVOS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'COMPRAS_SERVICOS'
                        THEN LIMITE_PEQUENO_VULTO_CPGF
                    END
                )
                    AS LIMITE_PV_COMPRAS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'OBRAS_ENGENHARIA'
                        THEN LIMITE_PEQUENO_VULTO_CPGF
                    END
                )
                    AS LIMITE_PV_ENGENHARIA,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'COMPRAS_SERVICOS'
                        THEN LIMITE_PEQUENO_VULTO_CENTAVOS
                    END
                )
                    AS LIMITE_PV_COMPRAS_CENTAVOS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'OBRAS_ENGENHARIA'
                        THEN LIMITE_PEQUENO_VULTO_CENTAVOS
                    END
                )
                    AS LIMITE_PV_ENGENHARIA_CENTAVOS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'COMPRAS_SERVICOS'
                        THEN LIMITE_CONCESSAO_CPGF
                    END
                )
                    AS LIMITE_CONCESSAO_COMPRAS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'OBRAS_ENGENHARIA'
                        THEN LIMITE_CONCESSAO_CPGF
                    END
                )
                    AS LIMITE_CONCESSAO_ENGENHARIA,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'COMPRAS_SERVICOS'
                        THEN NORMA_REFERENCIA
                    END
                )
                    AS NORMA_COMPRAS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'OBRAS_ENGENHARIA'
                        THEN NORMA_REFERENCIA
                    END
                )
                    AS NORMA_ENGENHARIA,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'COMPRAS_SERVICOS'
                        THEN STATUS_CENARIO
                    END
                )
                    AS STATUS_COMPRAS,

                MAX(
                    CASE
                        WHEN CATEGORIA_CENARIO = 'OBRAS_ENGENHARIA'
                        THEN STATUS_CENARIO
                    END
                )
                    AS STATUS_ENGENHARIA

            FROM t09_cenarios_tmp

            GROUP BY
                ID_TRANSACAO,
                UG_ID,
                FAVORECIDO_ID,
                DATA_DT,
                ANO_TRANSACAO
        )

        SELECT
            *,

            'CENARIOS_PARALELOS_SEM_CATEGORIA'
                AS MODO_APLICABILIDADE,

            'NAO_CONCLUSIVA_SEM_OBJETO_CATEGORIA'
                AS APLICABILIDADE_JURIDICA,

            VALOR_CENTAVOS
            * 1.0
            / NULLIF(
                LIMITE_PV_COMPRAS_CENTAVOS,
                0
            )
                AS RATIO_PV_COMPRAS,

            VALOR_CENTAVOS
            * 1.0
            / NULLIF(
                LIMITE_PV_ENGENHARIA_CENTAVOS,
                0
            )
                AS RATIO_PV_ENGENHARIA,

            CASE
                WHEN STATUS_COMPRAS = 'ACIMA_LIMITE'
                 AND STATUS_ENGENHARIA = 'ACIMA_LIMITE'
                    THEN 'ACIMA_AMBOS_CENARIOS'

                WHEN STATUS_COMPRAS = 'ACIMA_LIMITE'
                  OR STATUS_ENGENHARIA = 'ACIMA_LIMITE'
                    THEN 'ACIMA_PELO_MENOS_UM_CENARIO'

                WHEN STATUS_COMPRAS = 'NO_LIMITE'
                  OR STATUS_ENGENHARIA = 'NO_LIMITE'
                    THEN 'NO_LIMITE_PELO_MENOS_UM_CENARIO'

                WHEN STATUS_COMPRAS = 'PROXIMO_LIMITE'
                  OR STATUS_ENGENHARIA = 'PROXIMO_LIMITE'
                    THEN 'PROXIMO_LIMITE'

                ELSE 'ABAIXO_FAIXAS'
            END AS STATUS_T09,

            CASE
                WHEN STATUS_COMPRAS = 'ACIMA_LIMITE'
                 AND STATUS_ENGENHARIA = 'ACIMA_LIMITE'
                    THEN 'REFORCADO'

                WHEN STATUS_COMPRAS = 'ACIMA_LIMITE'
                  OR STATUS_ENGENHARIA = 'ACIMA_LIMITE'
                    THEN 'ATENCAO'

                WHEN STATUS_COMPRAS = 'NO_LIMITE'
                  OR STATUS_ENGENHARIA = 'NO_LIMITE'
                    THEN 'INFORMATIVO'

                WHEN STATUS_COMPRAS = 'PROXIMO_LIMITE'
                  OR STATUS_ENGENHARIA = 'PROXIMO_LIMITE'
                    THEN 'INFORMATIVO'

                ELSE 'SEM_SINAL'
            END AS NIVEL_TRIAGEM

        FROM p
    )
    TO '{t09_classificadas_path_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ------------------------------------------------------------
# 🚩 SINAIS T09
# ------------------------------------------------------------

t09_sinais_path_sql = (
    str(T09_SINAIS)
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        SELECT
            'T09_' || MD5(
                'T09|'
                || '{VERSAO_REGRAS}'
                || '|'
                || ID_TRANSACAO
                || '|'
                || STATUS_COMPRAS
                || '|'
                || STATUS_ENGENHARIA
            ) AS ID_SINAL,

            *

        FROM read_parquet(
            '{t09_classificadas_path_sql}'
        )

        WHERE STATUS_T09 <> 'ABAIXO_FAIXAS'
    )
    TO '{t09_sinais_path_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ------------------------------------------------------------
# 📊 RESUMOS
# ------------------------------------------------------------

t09_resumo_df = con.execute(
    f"""
    SELECT
        ANO_TRANSACAO,
        STATUS_T09,

        COUNT(*) AS N_TRANSACOES,
        SUM(VALOR_NUM) AS VALOR_TOTAL,

        MEDIAN(RATIO_PV_COMPRAS)
            AS MEDIANA_RATIO_COMPRAS,

        MEDIAN(RATIO_PV_ENGENHARIA)
            AS MEDIANA_RATIO_ENGENHARIA

    FROM read_parquet(
        '{t09_sinais_path_sql}'
    )

    GROUP BY
        ANO_TRANSACAO,
        STATUS_T09

    ORDER BY
        ANO_TRANSACAO,
        STATUS_T09
    """
).df()


t09_resumo_cenarios_df = con.execute(
    f"""
    SELECT
        ANO_TRANSACAO,
        'COMPRAS_SERVICOS' AS CENARIO,
        STATUS_COMPRAS AS STATUS_CENARIO,
        COUNT(*) AS N_TRANSACOES,
        SUM(VALOR_NUM) AS VALOR_TOTAL

    FROM read_parquet(
        '{t09_classificadas_path_sql}'
    )

    GROUP BY
        ANO_TRANSACAO,
        STATUS_COMPRAS

    UNION ALL

    SELECT
        ANO_TRANSACAO,
        'OBRAS_ENGENHARIA' AS CENARIO,
        STATUS_ENGENHARIA AS STATUS_CENARIO,
        COUNT(*) AS N_TRANSACOES,
        SUM(VALOR_NUM) AS VALOR_TOTAL

    FROM read_parquet(
        '{t09_classificadas_path_sql}'
    )

    GROUP BY
        ANO_TRANSACAO,
        STATUS_ENGENHARIA

    ORDER BY
        ANO_TRANSACAO,
        CENARIO,
        STATUS_CENARIO
    """
).df()


t09_resumo_df['STATUS_PERIODO'] = np.where(
    t09_resumo_df['ANO_TRANSACAO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

t09_resumo_cenarios_df['STATUS_PERIODO'] = np.where(
    t09_resumo_cenarios_df['ANO_TRANSACAO'].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)


# ------------------------------------------------------------
# 🔗 AGREGADO DESCRITIVO UG × FORNECEDOR × ANO
# ------------------------------------------------------------

T09_AGREGADOS = (
    T09_DIR
    / 't09_agregados_ug_fornecedor_ano.parquet'
)

t09_agregados_sql = (
    str(T09_AGREGADOS)
    .replace("'", "''")
)

con.execute(
    f"""
    COPY (
        SELECT
            UG_ID,
            FAVORECIDO_ID,
            ANO_TRANSACAO,

            COUNT(*) AS N_COMPRAS,
            SUM(VALOR_NUM) AS VALOR_TOTAL_COMPRAS,

            COUNT_IF(
                STATUS_T09 <> 'ABAIXO_FAIXAS'
            ) AS N_TRANSACOES_T09,

            SUM(
                CASE
                    WHEN STATUS_T09 <> 'ABAIXO_FAIXAS'
                    THEN VALOR_NUM
                    ELSE 0
                END
            ) AS VALOR_TRANSACOES_T09,

            COUNT_IF(
                STATUS_COMPRAS = 'PROXIMO_LIMITE'
            ) AS N_COMPRAS_PROXIMO,

            COUNT_IF(
                STATUS_COMPRAS = 'NO_LIMITE'
            ) AS N_COMPRAS_NO_LIMITE,

            COUNT_IF(
                STATUS_COMPRAS = 'ACIMA_LIMITE'
            ) AS N_COMPRAS_ACIMA,

            COUNT_IF(
                STATUS_ENGENHARIA = 'PROXIMO_LIMITE'
            ) AS N_ENGENHARIA_PROXIMO,

            COUNT_IF(
                STATUS_ENGENHARIA = 'NO_LIMITE'
            ) AS N_ENGENHARIA_NO_LIMITE,

            COUNT_IF(
                STATUS_ENGENHARIA = 'ACIMA_LIMITE'
            ) AS N_ENGENHARIA_ACIMA,

            MAX(RATIO_PV_COMPRAS)
                AS MAX_RATIO_PV_COMPRAS,

            MAX(RATIO_PV_ENGENHARIA)
                AS MAX_RATIO_PV_ENGENHARIA,

            COUNT_IF(
                STATUS_T09 <> 'ABAIXO_FAIXAS'
            )
            * 1.0
            / NULLIF(COUNT(*), 0)
                AS SHARE_TRANSACOES_T09,

            'DESCRITIVO_NAO_CONCLUSIVO'
                AS APLICABILIDADE_AGREGADO

        FROM read_parquet(
            '{t09_classificadas_path_sql}'
        )

        WHERE FAVORECIDO_ID IS NOT NULL

        GROUP BY
            UG_ID,
            FAVORECIDO_ID,
            ANO_TRANSACAO
    )
    TO '{t09_agregados_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ------------------------------------------------------------
# ✅ COBERTURA E CONTROLE DE APLICABILIDADE
# ------------------------------------------------------------

t09_cobertura_df = con.execute("""
SELECT
    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
    ) AS N_COMPRAS_NACIONAIS_POSITIVAS,

    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
        AND DATA_DT IS NOT NULL
    ) AS N_COM_DATA,

    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
        AND DATA_DT IS NULL
    ) AS N_SEM_DATA,

    COUNT_IF(
        EH_COMPRA_NACIONAL
        AND VALOR_NUM > 0
        AND FAVORECIDO_IDENTIFICADO
    ) AS N_FORNECEDOR_IDENTIFICADO

FROM stg_cpgf_transacoes
""").df()


n_cenarios_t09 = int(
    con.execute(
        "SELECT COUNT(*) FROM t09_cenarios_tmp"
    ).fetchone()[0]
)

n_com_data_t09 = int(
    t09_cobertura_df.loc[
        0,
        'N_COM_DATA'
    ]
)

t09_cobertura_df[
    'N_CENARIOS_GERADOS'
] = n_cenarios_t09

t09_cobertura_df[
    'N_CENARIOS_ESPERADOS'
] = n_com_data_t09 * 2

t09_cobertura_df[
    'COBERTURA_CENARIOS_OK'
] = (
    n_cenarios_t09
    == (n_com_data_t09 * 2)
)

t09_cobertura_df[
    'CATEGORIA_OBJETO_OBSERVAVEL'
] = False

t09_cobertura_df[
    'MODO_APLICABILIDADE'
] = 'CENARIOS_PARALELOS_SEM_CATEGORIA'


if n_cenarios_t09 != (
    n_com_data_t09 * 2
):
    raise AssertionError(
        f'T09: eram esperados '
        f'{n_com_data_t09 * 2:,} cenários, '
        f'mas foram gerados '
        f'{n_cenarios_t09:,}. '
        'Verifique lacunas ou sobreposição '
        'na dimensão normativa.'
    )


salvar_csv(
    t09_resumo_df,
    T09_RESUMO
)

salvar_csv(
    t09_resumo_cenarios_df,
    T09_RESUMO_CENARIOS
)

salvar_csv(
    t09_cobertura_df,
    T09_DIR
    / 't09_cobertura_aplicabilidade.csv'
)


display(
    t09_resumo_df
)

display(
    t09_resumo_cenarios_df
)

display(
    t09_cobertura_df
)


print(
    '✅ T09 executada integralmente '
    'em cenários paralelos.'
)

print(
    '✅ Igualdade e ultrapassagem '
    'foram separadas.'
)

print(
    '✅ Cobertura normativa temporal validada: '
    '2 cenários por compra nacional positiva '
    'com data válida.'
)

print(
    '⚠️ A categoria real da despesa continua '
    'não observável no CSV; a saída não '
    'constitui conclusão jurídica.'
)

## 2️⃣9️⃣ Modelo comum de saída das trilhas

Além das tabelas específicas, as trilhas são reunidas em uma tabela-mestre padronizada.

### Campos principais

- `ID_SINAL`;
- `CODIGO_TRILHA`;
- `VERSAO_REGRA`;
- `NIVEL_TRIAGEM`;
- `CODIGO_UG`;
- `CHAVE_ENTIDADE`;
- `PERIODO_INICIAL`;
- `PERIODO_FINAL`;
- `QTD_TRANSACOES`;
- `VALOR_TOTAL_CENTAVOS`;
- `EVIDENCIA_JSON`;
- `PARAMETROS_JSON`;
- `FUNDAMENTO`;
- `LIMITACOES`;
- `TEXTO_CIDADAO`.

Essa camada facilita a integração futura com o dashboard e com o assistente conversacional.

In [ ]:
def json_seguro(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        default=str
    )


partes_master = []


# ------------------------------------------------------------
# T01
# ------------------------------------------------------------

t01_master = con.execute(
    f"""
    SELECT
        ID_SINAL,
        ID_TRANSACAO,
        UG_ID,
        FAVORECIDO_ID,
        DATA_DT,
        VALOR_CENTAVOS,
        NIVEL_TRIAGEM,
        DIA_SEMANA_TXT,
        TRANSACAO

    FROM read_parquet(
        '{t01_path_sql}'
    )
    """
).df()

if len(t01_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t01_master[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T01',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t01_master[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t01_master[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t01_master[
                'FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t01_master[
                    'DATA_DT'
                ]
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t01_master[
                    'DATA_DT'
                ]
            ),
        'QTD_TRANSACOES':
            1,
        'VALOR_TOTAL_CENTAVOS':
            t01_master[
                'VALOR_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Despesa em final de semana: '
                'verificar justificativa documental.'
            ),
        'LIMITACOES':
            (
                'A base pública não contém '
                'a justificativa.'
            ),
        'TEXTO_CIDADAO':
            (
                'Esta compra foi registrada em um final de semana. '
                'O registro isoladamente não permite concluir '
                'que a despesa seja irregular.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t01_master.apply(
            lambda r: json_seguro({
                'dia_semana':
                    r['DIA_SEMANA_TXT'],
                'transacao':
                    r['TRANSACAO'],
                'id_transacao':
                    r['ID_TRANSACAO'],
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro({
            'regra':
                'sábado ou domingo'
        })
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T02
# ------------------------------------------------------------

t02_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{t02_path_sql}'
    )
    """
).df()

if len(t02_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t02_master[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T02',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t02_master[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t02_master[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t02_master[
                'FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t02_master[
                    'DATA_DT'
                ]
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t02_master[
                    'DATA_DT'
                ]
            ),
        'QTD_TRANSACOES':
            1,
        'VALOR_TOTAL_CENTAVOS':
            t02_master[
                'VALOR_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Classificação operacional '
                'de compra parcelada.'
            ),
        'LIMITACOES':
            (
                'A classificação exige conferência '
                'da fatura e documentos.'
            ),
        'TEXTO_CIDADAO':
            (
                'O Portal classificou esta operação como '
                'relacionada a compra parcelada. '
                'A classificação exige conferência documental.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t02_master.apply(
            lambda r: json_seguro({
                'transacao':
                    r['TRANSACAO'],
                'id_transacao':
                    r['ID_TRANSACAO'],
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro({
            'codigo_exato':
                CODIGO_COMPRA_PARCELADA
        })
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T03
# ------------------------------------------------------------

t03_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{t03_grupos_sql}'
    )
    """
).df()

if len(t03_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t03_master[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T03',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t03_master[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t03_master[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t03_master[
                'FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t03_master[
                    'DATA_DT'
                ]
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t03_master[
                    'DATA_DT'
                ]
            ),
        'QTD_TRANSACOES':
            t03_master[
                'N_TRANSACOES'
            ],
        'VALOR_TOTAL_CENTAVOS':
            t03_master[
                'VALOR_TOTAL_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Duplicação anormal como '
                'mecanismo de drill-down.'
            ),
        'LIMITACOES':
            (
                'Transações distintas podem ter '
                'a mesma data, fornecedor e valor.'
            ),
        'TEXTO_CIDADAO':
            (
                'Foram encontradas duas ou mais transações '
                'com o mesmo portador, fornecedor, data e valor. '
                'A repetição pode decorrer de operações legítimas.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t03_master.apply(
            lambda r: json_seguro({
                'portador':
                    r['PORTADOR_ID'],
                'valor_unitario':
                    r['VALOR_UNITARIO'],
                'ocorrencias':
                    int(r['N_TRANSACOES']),
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro(
            CONFIG['T03']
        )
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T04
# ------------------------------------------------------------

t04_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{t04_grupos_sql}'
    )
    """
).df()

if len(t04_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t04_master[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T04',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t04_master[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t04_master[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t04_master[
                'FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t04_master[
                    'DATA_DT'
                ]
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t04_master[
                    'DATA_DT'
                ]
            ),
        'QTD_TRANSACOES':
            t04_master[
                'N_TRANSACOES'
            ],
        'VALOR_TOTAL_CENTAVOS':
            t04_master[
                'VALOR_TOTAL_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Análise conjunta de agentes supridos '
                'da mesma UG.'
            ),
        'LIMITACOES':
            (
                'O Portal não informa o objeto adquirido.'
            ),
        'TEXTO_CIDADAO':
            (
                'Portadores diferentes da mesma unidade realizaram '
                'transações de mesmo valor, no mesmo dia e fornecedor. '
                'O padrão não permite afirmar divisão da mesma despesa.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t04_master.apply(
            lambda r: json_seguro({
                'n_portadores':
                    int(r['N_PORTADORES']),
                'valor_unitario':
                    r['VALOR_UNITARIO'],
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro(
            CONFIG['T04']
        )
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T05
# ------------------------------------------------------------

if len(t05_episodios_df):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t05_episodios_df[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T05',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t05_episodios_df[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t05_episodios_df[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t05_episodios_df[
                'FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t05_episodios_df[
                    'DT_INICIO'
                ]
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t05_episodios_df[
                    'DT_FIM'
                ]
            ),
        'QTD_TRANSACOES':
            t05_episodios_df[
                'N_TRANSACOES'
            ],
        'VALOR_TOTAL_CENTAVOS':
            t05_episodios_df[
                'VALOR_TOTAL_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Recorrência de aquisições na mesma UG '
                'e fornecedor como proxy comportamental.'
            ),
        'LIMITACOES':
            (
                'A base não informa o objeto adquirido; '
                'o sinal não confirma fracionamento.'
            ),
        'TEXTO_CIDADAO':
            (
                'A mesma unidade realizou várias compras de valores '
                'semelhantes junto ao mesmo fornecedor em curto intervalo '
                'e por mais de um portador.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t05_episodios_df.apply(
            lambda r: json_seguro({
                'n_portadores':
                    int(r['N_PORTADORES']),
                'cv': float(r['CV']),
                'share_dentro_faixa_mediana': float(r['SHARE_DENTRO_FAIXA_MEDIANA']),
                'percentil_materialidade_ano': float(r['PERCENTIL_MATERIALIDADE_ANO']),
                'faixa_materialidade': r['FAIXA_MATERIALIDADE'],
                'min_centavos':
                    int(r['MIN_CENTAVOS']),
                'max_centavos':
                    int(r['MAX_CENTAVOS']),
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro({
            'min_transacoes':
                CONFIG['T05']['min_transacoes'],
            'min_portadores':
                CONFIG['T05']['min_portadores'],
            'janela_dias':
                CONFIG['T05']['janela_dias'],
            'cv_base':
                CONFIG['T05']['cv_base'],
            'cv_reforcado':
                CONFIG['T05']['cv_reforcado'],
        })
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T06
# ------------------------------------------------------------

t06_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{t06_sinais_sql}'
    )
    """
).df()

if len(t06_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t06_master[
                'ID_SINAL'
            ],
        'CODIGO_TRILHA':
            'T06',
        'VERSAO_REGRA':
            VERSAO_REGRAS,
        'NIVEL_TRIAGEM':
            t06_master[
                'NIVEL_TRIAGEM'
            ],
        'CODIGO_UG':
            t06_master[
                'UG_ID'
            ],
        'CHAVE_ENTIDADE':
            t06_master[
                'TOP1_FAVORECIDO_ID'
            ],
        'PERIODO_INICIAL':
            pd.to_datetime(
                t06_master[
                    'ANO_TRANSACAO'
                ].astype(str)
                + '-01-01'
            ),
        'PERIODO_FINAL':
            pd.to_datetime(
                t06_master[
                    'ANO_TRANSACAO'
                ].astype(str)
                + '-12-31'
            ),
        'QTD_TRANSACOES':
            t06_master[
                'N_COMPRAS_IDENTIFICADAS'
            ],
        'VALOR_TOTAL_CENTAVOS':
            t06_master[
                'TOTAL_IDENTIFICADO_CENTAVOS'
            ],
        'FUNDAMENTO':
            (
                'Indicador de concentração em fornecedor '
                'associado a boas práticas de diversificação.'
            ),
        'LIMITACOES':
            (
                'Concentração não equivale a direcionamento.'
            ),
        'TEXTO_CIDADAO':
            (
                'Uma parcela elevada das compras identificáveis '
                'desta unidade foi realizada com o mesmo fornecedor.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t06_master.apply(
            lambda r: json_seguro({
                'top1_share': float(r['TOP1_SHARE']),
                'top1_share_qtd': float(r['TOP1_SHARE_QTD']),
                'top5_share':
                    float(r['TOP5_SHARE']),
                'hhi':
                    float(r['HHI']),
                'cobertura_valor':
                    float(r['COBERTURA_VALOR']),
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro(
            CONFIG['T06']
        )
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# T07 — recorrência anual priorizada
# ------------------------------------------------------------

t07_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet('{paths[T07_PRIORITARIOS]}')
    """
).df()

if len(t07_master):
    tmp = pd.DataFrame({
        'ID_SINAL': t07_master['ID_SINAL'],
        'CODIGO_TRILHA': 'T07',
        'VERSAO_REGRA': VERSAO_REGRAS,
        'NIVEL_TRIAGEM': 'ATENCAO',
        'CODIGO_UG': t07_master['UG_ID'],
        'CHAVE_ENTIDADE': t07_master['PORTADOR_ID'],
        'PERIODO_INICIAL': pd.to_datetime(t07_master['ANO_TRANSACAO'].astype(str) + '-01-01'),
        'PERIODO_FINAL': pd.to_datetime(t07_master['ANO_TRANSACAO'].astype(str) + '-12-31'),
        'QTD_TRANSACOES': t07_master['N_SAQUES_EM_EPISODIOS'],
        'VALOR_TOTAL_CENTAVOS': t07_master['VALOR_EPISODIOS_CENTAVOS'],
        'FUNDAMENTO': 'Recorrência anual de dias com múltiplos saques; modalidade de saque exige contexto e autorização próprios.',
        'LIMITACOES': 'A base não contém o ato de concessão nem a justificativa do saque.',
        'TEXTO_CIDADAO': 'O portador repetiu episódios de múltiplos saques em vários dias do exercício e ficou entre os maiores valores relativos do período.',
    })
    tmp['EVIDENCIA_JSON'] = t07_master.apply(lambda r: json_seguro({
        'n_dias_multisaque': int(r['N_DIAS_MULTISAQUE']),
        'n_saques_em_episodios': int(r['N_SAQUES_EM_EPISODIOS']),
        'percent_rank_dias': float(r['PERCENT_RANK_DIAS']),
        'n_comparaveis': int(r['N_PORTADORES_COMPARAVEIS_ANO']),
        'status_periodo': (
            'EXERCICIO_COMPLETO'
            if CONFIG['T08']['anos_completos_inicio']
               <= int(r['ANO_TRANSACAO'])
               <= CONFIG['T08']['anos_completos_fim']
            else 'PERIODO_PARCIAL'
        ),
    }), axis=1)
    tmp['PARAMETROS_JSON'] = json_seguro(CONFIG['T07'])
    partes_master.append(tmp)


# ------------------------------------------------------------
# T08 — extremos relativos válidos
# ------------------------------------------------------------

t08_master = benford_extremos_relativos_df.copy()

if len(t08_master):
    t08_master['ID_SINAL'] = t08_master.apply(
        lambda r: id_sinal('T08', r['UG_ID'], int(r['ANO_TRANSACAO']), round(float(r['MAD_D12']),8)),
        axis=1
    )
    tmp = pd.DataFrame({
        'ID_SINAL': t08_master['ID_SINAL'],
        'CODIGO_TRILHA': 'T08',
        'VERSAO_REGRA': VERSAO_REGRAS,
        'NIVEL_TRIAGEM': 'ATENCAO',
        'CODIGO_UG': t08_master['UG_ID'],
        'CHAVE_ENTIDADE': 'UG:' + t08_master['UG_ID'].astype(str),
        'PERIODO_INICIAL': pd.to_datetime(t08_master['ANO_TRANSACAO'].astype(str)+'-01-01'),
        'PERIODO_FINAL': pd.to_datetime(t08_master['ANO_TRANSACAO'].astype(str)+'-12-31'),
        'QTD_TRANSACOES': t08_master['N_D12'],
        'VALOR_TOTAL_CENTAVOS': pd.NA,
        'FUNDAMENTO': 'Lei de Newcomb-Benford — extremo relativo do MAD D12 entre UGs comparáveis no mesmo exercício.',
        'LIMITACOES': 'Benford não prova fraude; a população e o processo gerador devem ser considerados.',
        'TEXTO_CIDADAO': 'Esta UG ficou entre os maiores afastamentos relativos de Benford em um exercício com número suficiente de unidades comparáveis.',
    })
    tmp['EVIDENCIA_JSON'] = t08_master.apply(lambda r: json_seguro({
        'mad_d12': float(r['MAD_D12']),
        'percent_rank': float(r['PERCENT_RANK_MAD_D12']),
        'n_ugs_comparaveis': int(r['N_UG_COMPARAVEIS_ANO']),
        'ano': int(r['ANO_TRANSACAO']),
    }), axis=1)
    tmp['PARAMETROS_JSON'] = json_seguro(CONFIG['T08'])
    partes_master.append(tmp)


# ------------------------------------------------------------
# T09 — referências financeiras em cenários paralelos
# ------------------------------------------------------------

t09_master = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{str(T09_SINAIS).replace("'","''")}'
    )
    """
).df()

if len(t09_master):

    tmp = pd.DataFrame({
        'ID_SINAL':
            t09_master[
                'ID_SINAL'
            ],

        'CODIGO_TRILHA':
            'T09',

        'VERSAO_REGRA':
            VERSAO_REGRAS,

        'NIVEL_TRIAGEM':
            t09_master[
                'NIVEL_TRIAGEM'
            ],

        'CODIGO_UG':
            t09_master[
                'UG_ID'
            ],

        'CHAVE_ENTIDADE':
            t09_master[
                'FAVORECIDO_ID'
            ],

        'PERIODO_INICIAL':
            pd.to_datetime(
                t09_master[
                    'DATA_DT'
                ]
            ),

        'PERIODO_FINAL':
            pd.to_datetime(
                t09_master[
                    'DATA_DT'
                ]
            ),

        'QTD_TRANSACOES':
            1,

        'VALOR_TOTAL_CENTAVOS':
            t09_master[
                'VALOR_CENTAVOS'
            ].astype('Int64'),

        'FUNDAMENTO':
            (
                'Portaria MF 95/2002 ou Portaria Normativa '
                'MF 1.344/2023, conforme vigência, em '
                'cenários paralelos de categoria.'
            ),

        'LIMITACOES':
            (
                'A categoria/objeto da despesa e o ato de '
                'concessão não estão disponíveis; a '
                'classificação é não conclusiva.'
            ),

        'TEXTO_CIDADAO':
            (
                'O valor foi comparado com referências '
                'financeiras vigentes em dois cenários '
                'possíveis. A situação de cada cenário '
                'é mostrada separadamente como abaixo, '
                'próxima, no limite ou acima. Isso não '
                'confirma irregularidade.'
            ),
    })

    tmp['EVIDENCIA_JSON'] = (
        t09_master.apply(
            lambda r: json_seguro({
                'status_t09':
                    r['STATUS_T09'],

                'status_compras':
                    r['STATUS_COMPRAS'],

                'status_engenharia':
                    r['STATUS_ENGENHARIA'],

                'ratio_compras':
                    float(r['RATIO_PV_COMPRAS']),

                'ratio_engenharia':
                    float(r['RATIO_PV_ENGENHARIA']),

                'limite_pv_compras':
                    float(r['LIMITE_PV_COMPRAS']),

                'limite_pv_engenharia':
                    float(r['LIMITE_PV_ENGENHARIA']),

                'status_periodo':
                    (
                        'EXERCICIO_COMPLETO'
                        if CONFIG['T08']['anos_completos_inicio']
                           <= int(r['ANO_TRANSACAO'])
                           <= CONFIG['T08']['anos_completos_fim']
                        else 'PERIODO_PARCIAL'
                    ),
            }),
            axis=1
        )
    )

    tmp['PARAMETROS_JSON'] = (
        json_seguro(
            CONFIG['T09']
        )
    )

    partes_master.append(
        tmp
    )


# ------------------------------------------------------------
# CONSOLIDAR
# ------------------------------------------------------------

trilha_resultados_df = (
    pd.concat(
        partes_master,
        ignore_index=True
    )
    if partes_master
    else pd.DataFrame()
)

colunas_master = [
    'ID_SINAL',
    'CODIGO_TRILHA',
    'VERSAO_REGRA',
    'NIVEL_TRIAGEM',
    'CODIGO_UG',
    'CHAVE_ENTIDADE',
    'PERIODO_INICIAL',
    'PERIODO_FINAL',
    'QTD_TRANSACOES',
    'VALOR_TOTAL_CENTAVOS',
    'EVIDENCIA_JSON',
    'PARAMETROS_JSON',
    'FUNDAMENTO',
    'LIMITACOES',
    'TEXTO_CIDADAO',
]

trilha_resultados_df = (
    trilha_resultados_df[
        colunas_master
    ]
    if len(trilha_resultados_df)
    else pd.DataFrame(
        columns=colunas_master
    )
)

MASTER_PARQUET = (
    CONVERGENCIA_DIR
    / 'trilha_resultados.parquet'
)

MASTER_CSV = (
    CONVERGENCIA_DIR
    / 'trilha_resultados.csv'
)

salvar_parquet(
    trilha_resultados_df,
    MASTER_PARQUET
)

salvar_csv(
    trilha_resultados_df,
    MASTER_CSV
)

print(
    '✅ Tabela-mestre de sinais:',
    f'{len(trilha_resultados_df):,}',
    'linhas'
)

display(
    trilha_resultados_df
    .groupby(
        'CODIGO_TRILHA',
        as_index=False
    )
    .agg(
        N_SINAIS=(
            'ID_SINAL',
            'count'
        )
    )
)

## 3️⃣0️⃣ Ponte sinal ↔ transação

A rastreabilidade é um requisito central.

A tabela `trilha_resultado_transacao` permite voltar de um sinal às transações que o originaram.

Para T08, a ponte é limitada às transações selecionadas no *drill-down*; a população completa continua acessível pelas chaves `UG × ano`.

In [ ]:
PONTE_MASTER = (
    CONVERGENCIA_DIR
    / 'trilha_resultado_transacao.parquet'
)

ponte_master_sql = (
    str(PONTE_MASTER)
    .replace("'", "''")
)

con.execute("""
DROP TABLE IF EXISTS
    ponte_master_tmp
""")

con.execute("""
CREATE TEMP TABLE
    ponte_master_tmp
(
    CODIGO_TRILHA VARCHAR,
    ID_SINAL VARCHAR,
    ID_TRANSACAO VARCHAR,
    PAPEL_NO_SINAL VARCHAR
)
""")


# T01
con.execute(
    f"""
    INSERT INTO ponte_master_tmp

    SELECT
        'T01',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_sinalizada'

    FROM read_parquet(
        '{t01_path_sql}'
    )
    """
)


# T02
con.execute(
    f"""
    INSERT INTO ponte_master_tmp

    SELECT
        'T02',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_sinalizada'

    FROM read_parquet(
        '{t02_path_sql}'
    )
    """
)


# T03
con.execute(
    f"""
    INSERT INTO ponte_master_tmp

    SELECT
        'T03',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_do_grupo'

    FROM read_parquet(
        '{t03_ponte_sql}'
    )
    """
)


# T04
con.execute(
    f"""
    INSERT INTO ponte_master_tmp

    SELECT
        'T04',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_do_grupo'

    FROM read_parquet(
        '{t04_ponte_sql}'
    )
    """
)


# T05
if (
    T05_PONTE.exists()
    and len(t05_episodios_df)
):
    t05_ponte_path_sql = (
        str(T05_PONTE)
        .replace("'", "''")
    )

    con.execute(
        f"""
        INSERT INTO ponte_master_tmp

        SELECT
            'T05',
            ID_SINAL,
            ID_TRANSACAO,
            'transacao_do_episodio'

        FROM read_parquet(
            '{t05_ponte_path_sql}'
        )
        """
    )


# T06
con.execute(
    f"""
    INSERT INTO ponte_master_tmp

    SELECT
        'T06',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_fornecedor_top1'

    FROM read_parquet(
        '{t06_ponte_sql}'
    )
    """
)


# T07 — somente recorrência anual priorizada
con.execute(
    f"""
    INSERT INTO ponte_master_tmp
    SELECT
        'T07',
        ID_SINAL,
        ID_TRANSACAO,
        'saque_em_episodio_recorrente'
    FROM read_parquet('{paths[T07_PRIORITARIOS_PONTE]}')
    """
)


# T08 — apenas drill-down quando existente
if (
    'DRILL_TRANSACOES'
    in globals()
    and Path(
        DRILL_TRANSACOES
    ).exists()
    and len(t08_master)
):
    t08_bridge = t08_master[
        [
            'ID_SINAL',
            'UG_ID',
            'ANO_TRANSACAO'
        ]
    ].copy()

    con.register(
        't08_bridge_python',
        t08_bridge
    )

    drill_path_sql = (
        str(DRILL_TRANSACOES)
        .replace("'", "''")
    )

    con.execute(
        f"""
        INSERT INTO ponte_master_tmp

        SELECT
            'T08',
            s.ID_SINAL,
            d.ID_TRANSACAO,
            'transacao_drilldown'

        FROM
            t08_bridge_python s

        JOIN
            read_parquet(
                '{drill_path_sql}'
            ) d

          ON d.UG_ID
                = s.UG_ID
         AND d.ANO_TRANSACAO
                = s.ANO_TRANSACAO
        """
    )

    con.unregister(
        't08_bridge_python'
    )


# T09
con.execute(
    f"""
    INSERT INTO ponte_master_tmp
    SELECT
        'T09',
        ID_SINAL,
        ID_TRANSACAO,
        'transacao_comparada_a_referencia'
    FROM read_parquet('{str(T09_SINAIS).replace("'","''")}')
    """
)


con.execute(
    f"""
    COPY (
        SELECT DISTINCT *
        FROM ponte_master_tmp
    )
    TO '{ponte_master_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

ponte_resumo_df = con.execute(
    """
    SELECT
        CODIGO_TRILHA,

        COUNT(*)
            AS N_RELACOES,

        COUNT(
            DISTINCT ID_SINAL
        )
            AS N_SINAIS,

        COUNT(
            DISTINCT ID_TRANSACAO
        )
            AS N_TRANSACOES

    FROM ponte_master_tmp

    GROUP BY 1

    ORDER BY 1
    """
).df()

display(
    ponte_resumo_df
)

print(
    '✅ Ponte sinal ↔ transação criada.'
)

## 3️⃣1️⃣ Convergência entre trilhas — versão 1.2

A convergência continua **sem score único**.

### Visão institucional: `UG × ano`

A contagem principal utiliza apenas as **trilhas núcleo T01–T07**.

T08 e T09 aparecem em colunas independentes:

- `T08_CONTEXTO`;
- `T09_CONTEXTO`.

Assim, uma UG não ganha artificialmente “mais uma trilha” apenas porque possui um extremo relativo de Benford ou alguma transação em faixa T09.

### Visão principal relacional: `UG × fornecedor × ano`

A contagem por fornecedor utiliza **T01–T06**, porque são as trilhas cujo sinal pode ser atribuído diretamente ao fornecedor.

T08 permanece como contexto da `UG × ano`.

T09 aparece como contexto do **mesmo `UG × fornecedor × ano`**, com quantidade de sinais associada.

> A convergência informa sobreposição de evidências independentes; não representa probabilidade de fraude ou irregularidade.

In [ ]:
conv_base = (
    trilha_resultados_df
    .copy()
)

conv_base[
    'ANO'
] = pd.to_datetime(
    conv_base[
        'PERIODO_INICIAL'
    ],
    errors='coerce'
).dt.year


# ============================================================
# 🏛️ VISÃO UG × ANO — TRILHAS NÚCLEO T01–T07
# ============================================================

trilhas_nucleo_ug = [
    'T01',
    'T02',
    'T03',
    'T04',
    'T05',
    'T06',
    'T07',
]

conv_ug_nucleo = (
    conv_base[
        conv_base[
            'CODIGO_TRILHA'
        ]
        .isin(
            trilhas_nucleo_ug
        )
    ]
    .dropna(
        subset=[
            'CODIGO_UG',
            'ANO'
        ]
    )
    .copy()
)


convergencia_ug_ano_df = (
    conv_ug_nucleo
    .groupby(
        [
            'CODIGO_UG',
            'ANO'
        ],
        as_index=False
    )
    .agg(
        N_TRILHAS_NUCLEO=(
            'CODIGO_TRILHA',
            'nunique'
        ),

        TRILHAS_NUCLEO=(
            'CODIGO_TRILHA',
            lambda x:
                ' | '.join(
                    sorted(
                        set(
                            x.astype(str)
                        )
                    )
                )
        ),

        N_SINAIS_NUCLEO=(
            'ID_SINAL',
            'count'
        ),
    )
)


# T08 — contexto estatístico da UG-ano
t08_ug_contexto = (
    conv_base[
        conv_base[
            'CODIGO_TRILHA'
        ].eq('T08')
    ]
    .dropna(
        subset=[
            'CODIGO_UG',
            'ANO'
        ]
    )
    .groupby(
        [
            'CODIGO_UG',
            'ANO'
        ],
        as_index=False
    )
    .agg(
        N_SINAIS_T08_CONTEXTO=(
            'ID_SINAL',
            'count'
        )
    )
)

t08_ug_contexto[
    'T08_CONTEXTO'
] = True


# T09 — contexto normativo-financeiro da UG-ano
t09_ug_contexto = (
    conv_base[
        conv_base[
            'CODIGO_TRILHA'
        ].eq('T09')
    ]
    .dropna(
        subset=[
            'CODIGO_UG',
            'ANO'
        ]
    )
    .groupby(
        [
            'CODIGO_UG',
            'ANO'
        ],
        as_index=False
    )
    .agg(
        N_SINAIS_T09_CONTEXTO=(
            'ID_SINAL',
            'count'
        )
    )
)

t09_ug_contexto[
    'T09_CONTEXTO'
] = True


convergencia_ug_ano_df = (
    convergencia_ug_ano_df
    .merge(
        t08_ug_contexto,
        on=[
            'CODIGO_UG',
            'ANO'
        ],
        how='outer'
    )
    .merge(
        t09_ug_contexto,
        on=[
            'CODIGO_UG',
            'ANO'
        ],
        how='outer'
    )
)


for col in [
    'N_TRILHAS_NUCLEO',
    'N_SINAIS_NUCLEO',
    'N_SINAIS_T08_CONTEXTO',
    'N_SINAIS_T09_CONTEXTO',
]:
    if col in convergencia_ug_ano_df.columns:
        convergencia_ug_ano_df[
            col
        ] = (
            convergencia_ug_ano_df[
                col
            ]
            .fillna(0)
            .astype(int)
        )


convergencia_ug_ano_df[
    'TRILHAS_NUCLEO'
] = (
    convergencia_ug_ano_df[
        'TRILHAS_NUCLEO'
    ]
    .fillna('')
)

convergencia_ug_ano_df[
    'T08_CONTEXTO'
] = (
    convergencia_ug_ano_df[
        'T08_CONTEXTO'
    ]
    .fillna(False)
    .astype(bool)
)

convergencia_ug_ano_df[
    'T09_CONTEXTO'
] = (
    convergencia_ug_ano_df[
        'T09_CONTEXTO'
    ]
    .fillna(False)
    .astype(bool)
)


convergencia_ug_ano_df[
    'STATUS_PERIODO'
] = np.where(
    convergencia_ug_ano_df[
        'ANO'
    ].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)


convergencia_ug_ano_df = (
    convergencia_ug_ano_df
    .sort_values(
        [
            'N_TRILHAS_NUCLEO',
            'T08_CONTEXTO',
            'T09_CONTEXTO',
            'N_SINAIS_NUCLEO'
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
)


salvar_parquet(
    convergencia_ug_ano_df,
    CONVERGENCIA_DIR
    / 'convergencia_ug_ano.parquet'
)

salvar_csv(
    convergencia_ug_ano_df,
    CONVERGENCIA_DIR
    / 'convergencia_ug_ano.csv'
)


# ============================================================
# 🏪 VISÃO UG × FORNECEDOR × ANO — T01–T06
# ============================================================

trilhas_fornecedor = [
    'T01',
    'T02',
    'T03',
    'T04',
    'T05',
    'T06',
]

conv_fornecedor_base = (
    conv_base[
        conv_base[
            'CODIGO_TRILHA'
        ]
        .isin(
            trilhas_fornecedor
        )
        &
        conv_base[
            'CHAVE_ENTIDADE'
        ]
        .notna()
    ]
    .copy()
)


convergencia_fornecedor_df = (
    conv_fornecedor_base
    .groupby(
        [
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO'
        ],
        as_index=False
    )
    .agg(
        N_TRILHAS_FORNECEDOR=(
            'CODIGO_TRILHA',
            'nunique'
        ),

        TRILHAS_FORNECEDOR=(
            'CODIGO_TRILHA',
            lambda x:
                ' | '.join(
                    sorted(
                        set(
                            x.astype(str)
                        )
                    )
                )
        ),

        N_SINAIS_NUCLEO=(
            'ID_SINAL',
            'count'
        ),
    )
)


# T08 como contexto da UG-ano
t08_contexto_fornecedor = (
    t08_ug_contexto[
        [
            'CODIGO_UG',
            'ANO',
            'T08_CONTEXTO',
            'N_SINAIS_T08_CONTEXTO'
        ]
    ]
    .copy()
)


convergencia_fornecedor_df = (
    convergencia_fornecedor_df
    .merge(
        t08_contexto_fornecedor,
        on=[
            'CODIGO_UG',
            'ANO'
        ],
        how='left'
    )
)


# T09 como contexto do MESMO fornecedor-ano
t09_fornecedor_contexto = (
    conv_base[
        conv_base[
            'CODIGO_TRILHA'
        ].eq('T09')
        &
        conv_base[
            'CHAVE_ENTIDADE'
        ].notna()
    ]
    .dropna(
        subset=[
            'CODIGO_UG',
            'ANO'
        ]
    )
    .groupby(
        [
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO'
        ],
        as_index=False
    )
    .agg(
        N_SINAIS_T09_CONTEXTO=(
            'ID_SINAL',
            'count'
        )
    )
)

t09_fornecedor_contexto[
    'T09_CONTEXTO'
] = True


convergencia_fornecedor_df = (
    convergencia_fornecedor_df
    .merge(
        t09_fornecedor_contexto,
        on=[
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO'
        ],
        how='left'
    )
)


convergencia_fornecedor_df[
    'T08_CONTEXTO'
] = (
    convergencia_fornecedor_df[
        'T08_CONTEXTO'
    ]
    .fillna(False)
    .astype(bool)
)

convergencia_fornecedor_df[
    'T09_CONTEXTO'
] = (
    convergencia_fornecedor_df[
        'T09_CONTEXTO'
    ]
    .fillna(False)
    .astype(bool)
)


for col in [
    'N_SINAIS_T08_CONTEXTO',
    'N_SINAIS_T09_CONTEXTO',
]:
    convergencia_fornecedor_df[
        col
    ] = (
        convergencia_fornecedor_df[
            col
        ]
        .fillna(0)
        .astype(int)
    )


convergencia_fornecedor_df[
    'STATUS_PERIODO'
] = np.where(
    convergencia_fornecedor_df[
        'ANO'
    ].between(
        CONFIG['T08']['anos_completos_inicio'],
        CONFIG['T08']['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)


convergencia_fornecedor_df = (
    convergencia_fornecedor_df
    .sort_values(
        [
            'N_TRILHAS_FORNECEDOR',
            'T08_CONTEXTO',
            'T09_CONTEXTO',
            'N_SINAIS_NUCLEO'
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
)


salvar_parquet(
    convergencia_fornecedor_df,
    CONVERGENCIA_DIR
    / 'convergencia_ug_fornecedor_ano.parquet'
)

salvar_csv(
    convergencia_fornecedor_df,
    CONVERGENCIA_DIR
    / 'convergencia_ug_fornecedor_ano.csv'
)


print(
    '📊 Maiores convergências UG × ano'
)

display(
    convergencia_ug_ano_df
    .head(40)
)

print(
    '🏪 Maiores convergências UG × fornecedor × ano'
)

display(
    convergencia_fornecedor_df
    .head(40)
)

print(
    '✅ Convergência V1.2 calculada '
    'com T08 e T09 tratados como contexto.'
)

## 3️⃣2️⃣ V1.3.2 — Governança do Motor

A governança continua investigando:

> **“as trilhas acrescentam informação própria ou parte de sua associação decorre apenas da maior oportunidade de disparo?”**

### Unidades comparáveis

**Matriz A — `UG × fornecedor × ano`**

Inclui T01–T06.

É a unidade principal para examinar relações entre trilhas de compras.

**Matriz B — `UG × ano`**

Inclui T01–T07 como núcleo.

T08 e T09 permanecem como contextos.

### Exposição — tratamento assimétrico e intencional

A V1.3.2 não força a mesma técnica de estratificação em distribuições diferentes.

#### Fornecedor

A distribuição de `N_COMPRAS_FORNECEDOR` contém grande quantidade de empates. Por isso, usa **bandas fixas de contagem**:

`1`, `2`, `3–4`, `5–9`, `10–19`, `20+`.

#### UG

A distribuição de `N_OPERACOES_EFETIVAS` é suficientemente contínua para manter **decis anuais**.

### Raridade

Regras raras permanecem no motor, mas podem ficar fora de PCA/VIF.

### Temporalidade

Os diagnósticos principais continuam utilizando 2013–2025.

In [ ]:
print(
    '🧭 Iniciando camada de governança '
    f'do motor {VERSAO_MOTOR}.'
)

print(
    '🔒 Regras congeladas:',
    VERSAO_REGRAS
)

print(
    '📁 Governança:',
    GOVERNANCA_DIR
)

## 3️⃣3️⃣ 01 — Famílias de evidência

Uma trilha e uma família não são sinônimos.

Duas regras podem produzir sinais distintos, mas compartilhar a mesma dimensão substantiva. A V1.3.1 preserva e testa a **taxonomia inicial** e testa empiricamente sua coerência.

### Separação adicional

A natureza do sinal é distinta da validação.

Exemplos:

- T01 pode confirmar automaticamente que uma transação ocorreu no fim de semana;
- isso **não confirma** que a despesa carecia de justificativa;
- T09 pode confirmar matematicamente que o valor coincidiu com uma referência;
- isso **não confirma** desconformidade jurídica.

Por isso a camada V1.3.1 usa:

`TIPO_EVIDENCIA`

e mantém `STATUS_VALIDACAO` como uma decisão humana posterior.

### Convenção terminológica da V1.3.1

O notebook evita a expressão **“famílias independentes”** como propriedade fixa.

Usa-se **“famílias de evidência”**. O grau de associação entre elas é um resultado empírico produzido por Jaccard, Phi, contribuição marginal e diagnósticos estratificados por exposição.

In [ ]:
catalogo_trilhas_v13_df = pd.DataFrame([
    {
        'CODIGO_TRILHA': 'T01',
        'NOME': 'Despesa realizada em final de semana',
        'FAMILIA': 'F1',
        'FAMILIA_NOME': 'Conformidade operacional observável',
        'TIPO_EVIDENCIA': 'FATO_DETERMINISTICO',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'TRANSACAO',
        'PARAMETRO_USUARIO': 'BLOQUEADO',
        'HIPOTESE_CONTROLE':
            'Despesa em fim de semana demanda exame de justificativa documental.',
        'FUNDAMENTO_NORMATIVO':
            'Guias/manuais de suprimento de fundos: verificação de justificativa para despesas em final de semana.',
        'FUNDAMENTO_CIENTIFICO':
            'Fundamento científico predominantemente arquitetural em auditoria contínua; a condição é normativa.',
        'FUNDAMENTO_METODOLOGICO':
            'Regra determinística sobre DATA_TRANSACAO.',
        'LIMITACAO_CENTRAL':
            'A base pública não contém a justificativa.'
    },
    {
        'CODIGO_TRILHA': 'T02',
        'NOME': 'Transação classificada como compra parcelada',
        'FAMILIA': 'F1',
        'FAMILIA_NOME': 'Conformidade operacional observável',
        'TIPO_EVIDENCIA': 'FATO_DETERMINISTICO',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'TRANSACAO',
        'PARAMETRO_USUARIO': 'BLOQUEADO',
        'HIPOTESE_CONTROLE':
            'Classificação operacional de compra parcelada merece conferência documental.',
        'FUNDAMENTO_NORMATIVO':
            'Guias/manuais de suprimento: conferência de pagamentos à vista, totais e em uma única parcela.',
        'FUNDAMENTO_CIENTIFICO':
            'Fundamento científico predominantemente arquitetural em auditoria contínua; a condição é normativa.',
        'FUNDAMENTO_METODOLOGICO':
            'Correspondência exata do código de transação.',
        'LIMITACAO_CENTRAL':
            'A classificação exige exame de fatura e documentos.'
    },
    {
        'CODIGO_TRILHA': 'T03',
        'NOME': 'Repetição exata de transações',
        'FAMILIA': 'F2',
        'FAMILIA_NOME': 'Repetição e recorrência de aquisições',
        'TIPO_EVIDENCIA': 'PADRAO_COMPORTAMENTAL',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'GRUPO_TRANSACOES',
        'PARAMETRO_USUARIO': 'EXPERIMENTAL',
        'HIPOTESE_CONTROLE':
            'Repetições exatas podem orientar drill-down documental.',
        'FUNDAMENTO_NORMATIVO':
            'Relacionada ao monitoramento do uso do suprimento; não há presunção normativa de duplicidade.',
        'FUNDAMENTO_CIENTIFICO':
            'Auditoria analítica e drill-down de duplicações; interpretação condicionada ao contexto.',
        'FUNDAMENTO_METODOLOGICO':
            'Chave UG + portador + fornecedor + data + valor + tipo de transação.',
        'LIMITACAO_CENTRAL':
            'Transações distintas podem compartilhar atributos idênticos.'
    },
    {
        'CODIGO_TRILHA': 'T04',
        'NOME': 'Repetição multiportador',
        'FAMILIA': 'F2',
        'FAMILIA_NOME': 'Repetição e recorrência de aquisições',
        'TIPO_EVIDENCIA': 'PADRAO_COMPORTAMENTAL',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'UG_FORNECEDOR_DATA_VALOR',
        'PARAMETRO_USUARIO': 'EXPERIMENTAL',
        'HIPOTESE_CONTROLE':
            'Portadores diferentes realizando operações coincidentes podem indicar padrão que merece exame.',
        'FUNDAMENTO_NORMATIVO':
            'Regras de monitoramento de fracionamento recomendam análise conjunta dos supridos da mesma UG.',
        'FUNDAMENTO_CIENTIFICO':
            'Análise relacional de padrões de compras; evidência não conclusiva.',
        'FUNDAMENTO_METODOLOGICO':
            'Mesmo UG, fornecedor, data e valor, com múltiplos portadores.',
        'LIMITACAO_CENTRAL':
            'O objeto adquirido não é observável.'
    },
    {
        'CODIGO_TRILHA': 'T05',
        'NOME': 'Recorrência de aquisições',
        'FAMILIA': 'F2',
        'FAMILIA_NOME': 'Repetição e recorrência de aquisições',
        'TIPO_EVIDENCIA': 'PADRAO_COMPORTAMENTAL',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'UG_FORNECEDOR_EPISODIO',
        'PARAMETRO_USUARIO': 'EXPERIMENTAL',
        'HIPOTESE_CONTROLE':
            'Aquisições recorrentes, multiportador e de valores semelhantes podem indicar padrão de interesse.',
        'FUNDAMENTO_NORMATIVO':
            'Monitoramento de aquisições recorrentes e possível fracionamento no âmbito da UG.',
        'FUNDAMENTO_CIENTIFICO':
            'Detecção de padrões e auditoria analítica; parâmetros são calibrados empiricamente.',
        'FUNDAMENTO_METODOLOGICO':
            'Janela temporal, N, portadores, CV e similaridade robusta em torno da mediana.',
        'LIMITACAO_CENTRAL':
            'Sem objeto da despesa, não é possível confirmar fracionamento.'
    },
    {
        'CODIGO_TRILHA': 'T06',
        'NOME': 'Concentração em fornecedor',
        'FAMILIA': 'F3',
        'FAMILIA_NOME': 'Estrutura e concentração de fornecedor',
        'TIPO_EVIDENCIA': 'PADRAO_ESTRUTURAL',
        'PAPEL_MOTOR': 'NUCLEO',
        'UNIDADE_PRIMARIA': 'UG_ANO',
        'PARAMETRO_USUARIO': 'EXPERIMENTAL',
        'HIPOTESE_CONTROLE':
            'Concentração elevada pode orientar exame da estrutura de fornecedores.',
        'FUNDAMENTO_NORMATIVO':
            'Boas práticas recomendam evitar direcionamento e examinar a razoabilidade das aquisições.',
        'FUNDAMENTO_CIENTIFICO':
            'Métricas de concentração como Top-1 e HHI são utilizadas como indicadores estruturais.',
        'FUNDAMENTO_METODOLOGICO':
            'Top-1 por valor, Top-1 por quantidade, Top-5 e HHI.',
        'LIMITACAO_CENTRAL':
            'Concentração não equivale a favorecimento ou direcionamento.'
    },
    {
        'CODIGO_TRILHA': 'T07',
        'NOME': 'Recorrência de múltiplos saques',
        'FAMILIA': 'F4',
        'FAMILIA_NOME': 'Comportamento de saque',
        'TIPO_EVIDENCIA': 'PADRAO_COMPORTAMENTAL',
        'PAPEL_MOTOR': 'NUCLEO_UG_ANO',
        'UNIDADE_PRIMARIA': 'PORTADOR_ANO',
        'PARAMETRO_USUARIO': 'EXPERIMENTAL',
        'HIPOTESE_CONTROLE':
            'Recorrência de múltiplos saques ao longo do exercício pode orientar exame do ato de concessão.',
        'FUNDAMENTO_NORMATIVO':
            'Saques devem estar relacionados às ações autorizadas na concessão.',
        'FUNDAMENTO_CIENTIFICO':
            'Monitoramento longitudinal de comportamento; priorização relativa por ano.',
        'FUNDAMENTO_METODOLOGICO':
            'Dias com múltiplos saques + percentil anual entre portadores comparáveis.',
        'LIMITACAO_CENTRAL':
            'A base pública não contém a autorização de saque.'
    },
    {
        'CODIGO_TRILHA': 'T08',
        'NOME': 'Lei de Newcomb-Benford',
        'FAMILIA': 'F5',
        'FAMILIA_NOME': 'Contexto estatístico forense',
        'TIPO_EVIDENCIA': 'SINAL_ESTATISTICO',
        'PAPEL_MOTOR': 'CONTEXTO',
        'UNIDADE_PRIMARIA': 'UG_ANO',
        'PARAMETRO_USUARIO': 'BLOQUEADO_METODO',
        'HIPOTESE_CONTROLE':
            'Afastamentos digitais persistentes podem orientar drill-down.',
        'FUNDAMENTO_NORMATIVO':
            'Não é regra jurídica; funciona como técnica analítica auxiliar.',
        'FUNDAMENTO_CIENTIFICO':
            'Nigrini (2012) e literatura de Benford aplicada a auditoria.',
        'FUNDAMENTO_METODOLOGICO':
            'D1, D12, MAD, Z/chi-quadrado auxiliar, Summation, Number Duplication e persistência relativa.',
        'LIMITACAO_CENTRAL':
            'Não conformidade não prova erro ou fraude; arredondamentos afetam a distribuição.'
    },
    {
        'CODIGO_TRILHA': 'T09',
        'NOME': 'Referências financeiras aplicáveis',
        'FAMILIA': 'F6',
        'FAMILIA_NOME': 'Contexto normativo-financeiro',
        'TIPO_EVIDENCIA': 'CONTEXTO_NORMATIVO',
        'PAPEL_MOTOR': 'CONTEXTO',
        'UNIDADE_PRIMARIA': 'TRANSACAO',
        'PARAMETRO_USUARIO': 'PARCIALMENTE_BLOQUEADO',
        'HIPOTESE_CONTROLE':
            'Valores próximos, iguais ou superiores a referências financeiras merecem contextualização.',
        'FUNDAMENTO_NORMATIVO':
            'Portaria MF 95/2002 e Portaria Normativa MF 1.344/2023, conforme vigência.',
        'FUNDAMENTO_CIENTIFICO':
            'Uso de benchmarks normativos como contexto de triagem; não substitui enquadramento jurídico.',
        'FUNDAMENTO_METODOLOGICO':
            'Dois cenários paralelos porque a categoria/objeto não é observável.',
        'LIMITACAO_CENTRAL':
            'A categoria real da despesa não pode ser determinada automaticamente.'
    },
])

familias_v13_df = pd.DataFrame([
    {
        'FAMILIA':
            cod,

        'FAMILIA_NOME':
            dados['nome'],

        'TRILHAS':
            ' | '.join(
                dados['trilhas']
            ),

        'N_TRILHAS':
            len(
                dados['trilhas']
            ),
    }

    for cod, dados
    in MOTOR_CONFIG['familias'].items()
])


salvar_csv(
    catalogo_trilhas_v13_df,
    FAMILIAS_DIR
    / 'catalogo_trilhas_v1_3.csv'
)

salvar_csv(
    familias_v13_df,
    FAMILIAS_DIR
    / 'familias_evidencia.csv'
)

salvar_json(
    {
        'versao_motor':
            VERSAO_MOTOR,

        'familias':
            MOTOR_CONFIG[
                'familias'
            ],

        'principio':
            (
                'Famílias representam dimensões '
                'substantivas e serão testadas '
                'empiricamente; não são assumidas '
                'como independentes por definição.'
            ),
    },
    FAMILIAS_DIR
    / 'familias_evidencia.json'
)

display(
    catalogo_trilhas_v13_df[
        [
            'CODIGO_TRILHA',
            'FAMILIA',
            'FAMILIA_NOME',
            'TIPO_EVIDENCIA',
            'PAPEL_MOTOR',
            'PARAMETRO_USUARIO',
        ]
    ]
)

print(
    '✅ Taxonomia inicial de famílias '
    'e natureza da evidência registrada.'
)

## 3️⃣4️⃣ 02 — Matrizes de flags, exposição e elegibilidade estatística

A V1.3.2 preserva as mesmas unidades analíticas, mas utiliza estratificações adequadas à distribuição observada.

### Matriz `UG × fornecedor × ano`

Além das flags T01–T06, contém:

- `N_COMPRAS_FORNECEDOR`;
- `VALOR_COMPRAS_FORNECEDOR`;
- `N_PORTADORES_FORNECEDOR`;
- `N_DIAS_COMPRA_FORNECEDOR`;
- `BANDA_EXPOSICAO_FORNECEDOR`;
- `ROTULO_BANDA_EXPOSICAO_FORNECEDOR`;
- `ORDEM_BANDA_EXPOSICAO_FORNECEDOR`.

Bandas:

| Ordem | Faixa |
|---:|---|
| 1 | 1 compra |
| 2 | 2 compras |
| 3 | 3–4 compras |
| 4 | 5–9 compras |
| 5 | 10–19 compras |
| 6 | 20+ compras |

Essas bandas são absolutas e interpretáveis; não são chamadas de decis.

### Matriz `UG × ano`

Mantém:

- `N_COMPRAS_UG`;
- `VALOR_COMPRAS_UG`;
- `N_SAQUES_UG`;
- `VALOR_SAQUES_UG`;
- `N_OPERACOES_EFETIVAS`;
- `N_PORTADORES_UG`;
- `N_FORNECEDORES_UG`;
- `N_DIAS_ATIVOS_UG`;
- `PERCENTIL_EXPOSICAO_ANUAL`;
- `DECIL_EXPOSICAO_ANUAL`.

### Elegibilidade estatística

PCA/VIF continuam exigindo:

- pelo menos 30 positivos;
- pelo menos 30 negativos.

A insuficiência estatística não remove uma trilha do motor.

In [ ]:
# ============================================================
# PREPARAR MASTER PARA FLAGS
# ============================================================

master_flags_v13 = (
    trilha_resultados_df[
        [
            'ID_SINAL',
            'CODIGO_TRILHA',
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'PERIODO_INICIAL',
        ]
    ]
    .copy()
)

master_flags_v13['ANO'] = (
    pd.to_datetime(
        master_flags_v13['PERIODO_INICIAL'],
        errors='coerce'
    )
    .dt.year
    .astype('Int64')
)

master_flags_v13['CODIGO_UG'] = (
    master_flags_v13['CODIGO_UG']
    .astype('string')
)

master_flags_v13['CHAVE_ENTIDADE'] = (
    master_flags_v13['CHAVE_ENTIDADE']
    .astype('string')
)


# ============================================================
# HELPERS DE EXPOSIÇÃO
# ============================================================

def adicionar_decil_exposicao_anual_ug(
    df,
    coluna_exposicao,
    coluna_ano='ANO',
    n_decis=10
):
    """
    Decis anuais para UG × ano.

    Rank(pct=True) preserva empates e é adequado aqui porque
    a distribuição de N_OPERACOES_EFETIVAS possui diversidade
    suficiente para formar estratos úteis.
    """
    out = df.copy()

    pct = (
        out.groupby(coluna_ano)[coluna_exposicao]
        .rank(
            method='average',
            pct=True
        )
    )

    out['PERCENTIL_EXPOSICAO_ANUAL'] = pct

    decil = np.ceil(
        pct * n_decis
    )

    decil = np.clip(
        decil,
        1,
        n_decis
    )

    out['DECIL_EXPOSICAO_ANUAL'] = (
        pd.Series(
            decil,
            index=out.index
        )
        .astype('Int64')
    )

    return out


def adicionar_banda_exposicao_fornecedor(
    df,
    coluna_exposicao='N_COMPRAS_FORNECEDOR'
):
    """
    Bandas fixas para UG × fornecedor × ano.

    A opção evita a falsa aparência de dez decis quando a
    distribuição contém grande quantidade de empates em
    valores discretos como 1 e 2 compras.
    """
    out = df.copy()

    regras = (
        MOTOR_CONFIG[
            'diagnostico'
        ]['bandas_exposicao_fornecedor']
    )

    codigo = pd.Series(
        pd.NA,
        index=out.index,
        dtype='string'
    )

    rotulo = pd.Series(
        pd.NA,
        index=out.index,
        dtype='string'
    )

    ordem = pd.Series(
        pd.NA,
        index=out.index,
        dtype='Int64'
    )

    x = out[coluna_exposicao]

    for regra in regras:
        minimo = regra['min']
        maximo = regra['max']

        if maximo is None:
            mask = x >= minimo
        else:
            mask = (
                (x >= minimo)
                &
                (x <= maximo)
            )

        codigo.loc[mask] = regra['codigo']
        rotulo.loc[mask] = regra['rotulo']
        ordem.loc[mask] = regra['ordem']

    out['BANDA_EXPOSICAO_FORNECEDOR'] = codigo
    out['ROTULO_BANDA_EXPOSICAO_FORNECEDOR'] = rotulo
    out['ORDEM_BANDA_EXPOSICAO_FORNECEDOR'] = ordem

    if out[
        'BANDA_EXPOSICAO_FORNECEDOR'
    ].isna().any():
        n_sem_banda = int(
            out[
                'BANDA_EXPOSICAO_FORNECEDOR'
            ].isna().sum()
        )

        raise AssertionError(
            f'Exposição fornecedor: {n_sem_banda:,} '
            'linhas ficaram sem banda.'
        )

    return out


def avaliar_elegibilidade_flags(
    df,
    flags,
    unidade
):
    min_pos = (
        MOTOR_CONFIG[
            'diagnostico'
        ]['min_positivos_estatistica']
    )

    min_neg = (
        MOTOR_CONFIG[
            'diagnostico'
        ]['min_negativos_estatistica']
    )

    linhas = []

    for regra in flags:
        s = (
            df[regra]
            .fillna(0)
            .astype(int)
        )

        n = len(s)
        n_pos = int((s == 1).sum())
        n_neg = int((s == 0).sum())

        if n_pos == 0 or n_neg == 0:
            status = 'SEM_VARIACAO'
        elif n_pos < min_pos or n_neg < min_neg:
            status = 'DIAGNOSTICO_ESTATISTICO_INSUFICIENTE'
        else:
            status = 'SUFICIENTE'

        linhas.append({
            'UNIDADE_ANALISE':
                unidade,

            'REGRA':
                regra,

            'N_UNIVERSO':
                n,

            'N_POSITIVOS':
                n_pos,

            'N_NEGATIVOS':
                n_neg,

            'PREVALENCIA':
                (
                    n_pos / n
                    if n
                    else np.nan
                ),

            'MIN_POSITIVOS_EXIGIDO':
                min_pos,

            'MIN_NEGATIVOS_EXIGIDO':
                min_neg,

            'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO':
                status,

            'ELEGIVEL_PCA_VIF':
                (
                    status
                    == 'SUFICIENTE'
                ),
        })

    return pd.DataFrame(linhas)


# ============================================================
# UNIVERSO + EXPOSIÇÃO UG × FORNECEDOR × ANO
# ============================================================

universo_fornecedor_v13_df = con.execute("""
SELECT
    CAST(UG_ID AS VARCHAR)
        AS CODIGO_UG,

    CAST(FAVORECIDO_ID AS VARCHAR)
        AS CHAVE_ENTIDADE,

    CAST(ANO_TRANSACAO AS INTEGER)
        AS ANO,

    COUNT(*) AS N_COMPRAS_FORNECEDOR,

    SUM(VALOR_CENTAVOS) / 100.0
        AS VALOR_COMPRAS_FORNECEDOR,

    COUNT(DISTINCT PORTADOR_ID)
        AS N_PORTADORES_FORNECEDOR,

    COUNT(DISTINCT DATA_DT)
        AS N_DIAS_COMPRA_FORNECEDOR

FROM stg_cpgf_transacoes

WHERE EH_COMPRA_EFETIVA
  AND NOT EH_AJUSTE_CONTESTACAO
  AND VALOR_CENTAVOS > 0
  AND DATA_DT IS NOT NULL
  AND UG_ID IS NOT NULL
  AND FAVORECIDO_IDENTIFICADO
  AND ANO_TRANSACAO IS NOT NULL

GROUP BY 1,2,3
""").df()


flags_forn_long = (
    master_flags_v13[
        master_flags_v13[
            'CODIGO_TRILHA'
        ].isin(
            MOTOR_CONFIG[
                'matriz_fornecedor'
            ]['trilhas']
        )
        &
        master_flags_v13[
            'CHAVE_ENTIDADE'
        ].notna()
        &
        master_flags_v13[
            'ANO'
        ].notna()
    ][
        [
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO',
            'CODIGO_TRILHA',
        ]
    ]
    .drop_duplicates()
)

flags_forn_pivot = (
    flags_forn_long
    .assign(FLAG=1)
    .pivot_table(
        index=[
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO',
        ],
        columns='CODIGO_TRILHA',
        values='FLAG',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

flags_forn_pivot.columns.name = None

matriz_fornecedor_v13_df = (
    universo_fornecedor_v13_df
    .merge(
        flags_forn_pivot,
        on=[
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO',
        ],
        how='left'
    )
)

for trilha in (
    MOTOR_CONFIG[
        'matriz_fornecedor'
    ]['trilhas']
):
    if trilha not in matriz_fornecedor_v13_df.columns:
        matriz_fornecedor_v13_df[trilha] = 0

    matriz_fornecedor_v13_df[trilha] = (
        matriz_fornecedor_v13_df[trilha]
        .fillna(0)
        .astype('int8')
    )

matriz_fornecedor_v13_df['F1'] = (
    (
        matriz_fornecedor_v13_df[
            ['T01', 'T02']
        ].sum(axis=1)
        > 0
    )
    .astype('int8')
)

matriz_fornecedor_v13_df['F2'] = (
    (
        matriz_fornecedor_v13_df[
            ['T03', 'T04', 'T05']
        ].sum(axis=1)
        > 0
    )
    .astype('int8')
)

matriz_fornecedor_v13_df['F3'] = (
    matriz_fornecedor_v13_df['T06']
    .astype('int8')
)

matriz_fornecedor_v13_df['N_TRILHAS_ATIVAS'] = (
    matriz_fornecedor_v13_df[
        MOTOR_CONFIG[
            'matriz_fornecedor'
        ]['trilhas']
    ]
    .sum(axis=1)
)

matriz_fornecedor_v13_df['N_FAMILIAS_ATIVAS'] = (
    matriz_fornecedor_v13_df[
        ['F1', 'F2', 'F3']
    ]
    .sum(axis=1)
)

matriz_fornecedor_v13_df['STATUS_PERIODO'] = np.where(
    matriz_fornecedor_v13_df['ANO'].between(
        MOTOR_CONFIG[
            'diagnostico'
        ]['anos_completos_inicio'],
        MOTOR_CONFIG[
            'diagnostico'
        ]['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

matriz_fornecedor_v13_df = (
    adicionar_banda_exposicao_fornecedor(
        matriz_fornecedor_v13_df
    )
)


# ============================================================
# CONTEXTOS T08/T09 NA MATRIZ FORNECEDOR
# ============================================================

t08_ctx_ug = (
    master_flags_v13[
        master_flags_v13[
            'CODIGO_TRILHA'
        ].eq('T08')
        &
        master_flags_v13['ANO'].notna()
    ][
        [
            'CODIGO_UG',
            'ANO',
        ]
    ]
    .drop_duplicates()
    .assign(T08_CONTEXTO=1)
)

t09_ctx_forn = (
    master_flags_v13[
        master_flags_v13[
            'CODIGO_TRILHA'
        ].eq('T09')
        &
        master_flags_v13[
            'CHAVE_ENTIDADE'
        ].notna()
        &
        master_flags_v13['ANO'].notna()
    ][
        [
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO',
        ]
    ]
    .drop_duplicates()
    .assign(T09_CONTEXTO=1)
)

matriz_fornecedor_v13_df = (
    matriz_fornecedor_v13_df
    .merge(
        t08_ctx_ug,
        on=[
            'CODIGO_UG',
            'ANO',
        ],
        how='left'
    )
    .merge(
        t09_ctx_forn,
        on=[
            'CODIGO_UG',
            'CHAVE_ENTIDADE',
            'ANO',
        ],
        how='left'
    )
)

for c in [
    'T08_CONTEXTO',
    'T09_CONTEXTO',
]:
    matriz_fornecedor_v13_df[c] = (
        matriz_fornecedor_v13_df[c]
        .fillna(0)
        .astype('int8')
    )


# ============================================================
# UNIVERSO + EXPOSIÇÃO UG × ANO
# ============================================================

universo_ug_v13_df = con.execute("""
SELECT
    CAST(UG_ID AS VARCHAR)
        AS CODIGO_UG,

    CAST(ANO_TRANSACAO AS INTEGER)
        AS ANO,

    COUNT_IF(
        EH_COMPRA_EFETIVA
        AND NOT EH_AJUSTE_CONTESTACAO
        AND VALOR_CENTAVOS > 0
    ) AS N_COMPRAS_UG,

    SUM(
        CASE
            WHEN EH_COMPRA_EFETIVA
             AND NOT EH_AJUSTE_CONTESTACAO
             AND VALOR_CENTAVOS > 0
            THEN VALOR_CENTAVOS
            ELSE 0
        END
    ) / 100.0 AS VALOR_COMPRAS_UG,

    COUNT_IF(
        EH_SAQUE_EFETIVO
        AND NOT EH_AJUSTE_CONTESTACAO
        AND VALOR_CENTAVOS > 0
    ) AS N_SAQUES_UG,

    SUM(
        CASE
            WHEN EH_SAQUE_EFETIVO
             AND NOT EH_AJUSTE_CONTESTACAO
             AND VALOR_CENTAVOS > 0
            THEN VALOR_CENTAVOS
            ELSE 0
        END
    ) / 100.0 AS VALOR_SAQUES_UG,

    COUNT_IF(
        (
            EH_COMPRA_EFETIVA
            OR EH_SAQUE_EFETIVO
        )
        AND NOT EH_AJUSTE_CONTESTACAO
        AND VALOR_CENTAVOS > 0
    ) AS N_OPERACOES_EFETIVAS,

    COUNT(
        DISTINCT CASE
            WHEN (
                EH_COMPRA_EFETIVA
                OR EH_SAQUE_EFETIVO
            )
            AND NOT EH_AJUSTE_CONTESTACAO
            AND VALOR_CENTAVOS > 0
            THEN PORTADOR_ID
        END
    ) AS N_PORTADORES_UG,

    COUNT(
        DISTINCT CASE
            WHEN EH_COMPRA_EFETIVA
             AND NOT EH_AJUSTE_CONTESTACAO
             AND VALOR_CENTAVOS > 0
             AND FAVORECIDO_IDENTIFICADO
            THEN FAVORECIDO_ID
        END
    ) AS N_FORNECEDORES_UG,

    COUNT(
        DISTINCT CASE
            WHEN (
                EH_COMPRA_EFETIVA
                OR EH_SAQUE_EFETIVO
            )
            AND NOT EH_AJUSTE_CONTESTACAO
            AND VALOR_CENTAVOS > 0
            THEN DATA_DT
        END
    ) AS N_DIAS_ATIVOS_UG

FROM stg_cpgf_transacoes

WHERE DATA_DT IS NOT NULL
  AND UG_ID IS NOT NULL
  AND ANO_TRANSACAO IS NOT NULL
  AND NOT EH_AJUSTE_CONTESTACAO
  AND VALOR_CENTAVOS > 0
  AND (
        EH_COMPRA_EFETIVA
        OR EH_SAQUE_EFETIVO
      )

GROUP BY 1,2
""").df()


flags_ug_long = (
    master_flags_v13[
        master_flags_v13[
            'CODIGO_TRILHA'
        ].isin(
            MOTOR_CONFIG[
                'matriz_ug'
            ]['trilhas']
        )
        &
        master_flags_v13['ANO'].notna()
    ][
        [
            'CODIGO_UG',
            'ANO',
            'CODIGO_TRILHA',
        ]
    ]
    .drop_duplicates()
)

flags_ug_pivot = (
    flags_ug_long
    .assign(FLAG=1)
    .pivot_table(
        index=[
            'CODIGO_UG',
            'ANO',
        ],
        columns='CODIGO_TRILHA',
        values='FLAG',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

flags_ug_pivot.columns.name = None

matriz_ug_v13_df = (
    universo_ug_v13_df
    .merge(
        flags_ug_pivot,
        on=[
            'CODIGO_UG',
            'ANO',
        ],
        how='left'
    )
)

for trilha in (
    MOTOR_CONFIG[
        'matriz_ug'
    ]['trilhas']
):
    if trilha not in matriz_ug_v13_df.columns:
        matriz_ug_v13_df[trilha] = 0

    matriz_ug_v13_df[trilha] = (
        matriz_ug_v13_df[trilha]
        .fillna(0)
        .astype('int8')
    )

matriz_ug_v13_df['F1'] = (
    (
        matriz_ug_v13_df[
            ['T01', 'T02']
        ].sum(axis=1)
        > 0
    )
    .astype('int8')
)

matriz_ug_v13_df['F2'] = (
    (
        matriz_ug_v13_df[
            ['T03', 'T04', 'T05']
        ].sum(axis=1)
        > 0
    )
    .astype('int8')
)

matriz_ug_v13_df['F3'] = (
    matriz_ug_v13_df['T06']
    .astype('int8')
)

matriz_ug_v13_df['F4'] = (
    matriz_ug_v13_df['T07']
    .astype('int8')
)

matriz_ug_v13_df['N_TRILHAS_NUCLEO'] = (
    matriz_ug_v13_df[
        MOTOR_CONFIG[
            'matriz_ug'
        ]['trilhas']
    ]
    .sum(axis=1)
)

matriz_ug_v13_df['N_FAMILIAS_NUCLEO'] = (
    matriz_ug_v13_df[
        ['F1', 'F2', 'F3', 'F4']
    ]
    .sum(axis=1)
)


# Contextos UG-ano
ctx_ug = (
    master_flags_v13[
        master_flags_v13[
            'CODIGO_TRILHA'
        ].isin(
            ['T08', 'T09']
        )
        &
        master_flags_v13['ANO'].notna()
    ][
        [
            'CODIGO_UG',
            'ANO',
            'CODIGO_TRILHA',
        ]
    ]
    .drop_duplicates()
    .assign(FLAG=1)
    .pivot_table(
        index=[
            'CODIGO_UG',
            'ANO',
        ],
        columns='CODIGO_TRILHA',
        values='FLAG',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

ctx_ug.columns.name = None

matriz_ug_v13_df = (
    matriz_ug_v13_df
    .merge(
        ctx_ug,
        on=[
            'CODIGO_UG',
            'ANO',
        ],
        how='left',
        suffixes=('', '_CTX')
    )
)

for c in ['T08', 'T09']:
    if c not in matriz_ug_v13_df.columns:
        matriz_ug_v13_df[c] = 0

    matriz_ug_v13_df[f'{c}_CONTEXTO'] = (
        matriz_ug_v13_df[c]
        .fillna(0)
        .astype('int8')
    )

for c in ['T08', 'T09']:
    if c in matriz_ug_v13_df.columns:
        matriz_ug_v13_df.drop(
            columns=c,
            inplace=True
        )

matriz_ug_v13_df['STATUS_PERIODO'] = np.where(
    matriz_ug_v13_df['ANO'].between(
        MOTOR_CONFIG[
            'diagnostico'
        ]['anos_completos_inicio'],
        MOTOR_CONFIG[
            'diagnostico'
        ]['anos_completos_fim']
    ),
    'EXERCICIO_COMPLETO',
    'PERIODO_PARCIAL'
)

matriz_ug_v13_df = (
    adicionar_decil_exposicao_anual_ug(
        matriz_ug_v13_df,
        'N_OPERACOES_EFETIVAS',
        n_decis=MOTOR_CONFIG[
            'diagnostico'
        ]['n_decis_exposicao_ug']
    )
)


# ============================================================
# RECORTES DE EXERCÍCIOS COMPLETOS
# ============================================================

forn_diag = (
    matriz_fornecedor_v13_df[
        matriz_fornecedor_v13_df[
            'STATUS_PERIODO'
        ]
        == 'EXERCICIO_COMPLETO'
    ]
    .copy()
)

ug_diag = (
    matriz_ug_v13_df[
        matriz_ug_v13_df[
            'STATUS_PERIODO'
        ]
        == 'EXERCICIO_COMPLETO'
    ]
    .copy()
)

flags_forn = (
    MOTOR_CONFIG[
        'matriz_fornecedor'
    ]['trilhas']
)

flags_ug = (
    MOTOR_CONFIG[
        'matriz_ug'
    ]['trilhas']
)


# ============================================================
# ELEGIBILIDADE PARA PCA/VIF
# ============================================================

elegibilidade_forn_df = (
    avaliar_elegibilidade_flags(
        forn_diag,
        flags_forn,
        'UG_FORNECEDOR_ANO'
    )
)

elegibilidade_ug_df = (
    avaliar_elegibilidade_flags(
        ug_diag,
        flags_ug,
        'UG_ANO'
    )
)

flags_forn_modelagem = (
    elegibilidade_forn_df.loc[
        elegibilidade_forn_df[
            'ELEGIVEL_PCA_VIF'
        ],
        'REGRA'
    ]
    .tolist()
)

flags_ug_modelagem = (
    elegibilidade_ug_df.loc[
        elegibilidade_ug_df[
            'ELEGIVEL_PCA_VIF'
        ],
        'REGRA'
    ]
    .tolist()
)


# ============================================================
# PERFIL DE EXPOSIÇÃO
# ============================================================

perfil_bandas_fornecedor_df = (
    forn_diag
    .groupby(
        [
            'ORDEM_BANDA_EXPOSICAO_FORNECEDOR',
            'BANDA_EXPOSICAO_FORNECEDOR',
            'ROTULO_BANDA_EXPOSICAO_FORNECEDOR',
        ],
        as_index=False,
        observed=False
    )
    .agg(
        N_UNIDADES=(
            'CHAVE_ENTIDADE',
            'size'
        ),

        N_UG=(
            'CODIGO_UG',
            'nunique'
        ),

        N_FORNECEDORES=(
            'CHAVE_ENTIDADE',
            'nunique'
        ),

        MEDIANA_N_COMPRAS=(
            'N_COMPRAS_FORNECEDOR',
            'median'
        ),

        MEDIA_N_COMPRAS=(
            'N_COMPRAS_FORNECEDOR',
            'mean'
        ),

        MEDIANA_VALOR=(
            'VALOR_COMPRAS_FORNECEDOR',
            'median'
        ),

        T01_PREVALENCIA=(
            'T01',
            'mean'
        ),

        T02_PREVALENCIA=(
            'T02',
            'mean'
        ),

        T03_PREVALENCIA=(
            'T03',
            'mean'
        ),

        T04_PREVALENCIA=(
            'T04',
            'mean'
        ),

        T05_PREVALENCIA=(
            'T05',
            'mean'
        ),

        T06_PREVALENCIA=(
            'T06',
            'mean'
        ),
    )
    .sort_values(
        'ORDEM_BANDA_EXPOSICAO_FORNECEDOR'
    )
)

perfil_decis_ug_df = (
    ug_diag
    .groupby(
        'DECIL_EXPOSICAO_ANUAL',
        as_index=False
    )
    .agg(
        N_UNIDADES=(
            'CODIGO_UG',
            'size'
        ),

        MEDIANA_OPERACOES=(
            'N_OPERACOES_EFETIVAS',
            'median'
        ),

        MEDIA_OPERACOES=(
            'N_OPERACOES_EFETIVAS',
            'mean'
        ),

        T01_PREVALENCIA=(
            'T01',
            'mean'
        ),

        T02_PREVALENCIA=(
            'T02',
            'mean'
        ),

        T03_PREVALENCIA=(
            'T03',
            'mean'
        ),

        T04_PREVALENCIA=(
            'T04',
            'mean'
        ),

        T05_PREVALENCIA=(
            'T05',
            'mean'
        ),

        T06_PREVALENCIA=(
            'T06',
            'mean'
        ),

        T07_PREVALENCIA=(
            'T07',
            'mean'
        ),
    )
    .sort_values(
        'DECIL_EXPOSICAO_ANUAL'
    )
)

if int(
    perfil_bandas_fornecedor_df[
        'N_UNIDADES'
    ].sum()
) != len(forn_diag):
    raise AssertionError(
        'As bandas de exposição fornecedor '
        'não recompõem o universo completo.'
    )


# ============================================================
# EXPORTAR
# ============================================================

salvar_parquet(
    matriz_fornecedor_v13_df,
    MATRIZES_DIR
    / 'matriz_flags_ug_fornecedor_ano.parquet'
)

salvar_csv(
    matriz_fornecedor_v13_df,
    MATRIZES_DIR
    / 'matriz_flags_ug_fornecedor_ano.csv'
)

salvar_parquet(
    matriz_ug_v13_df,
    MATRIZES_DIR
    / 'matriz_flags_ug_ano.parquet'
)

salvar_csv(
    matriz_ug_v13_df,
    MATRIZES_DIR
    / 'matriz_flags_ug_ano.csv'
)

salvar_csv(
    elegibilidade_forn_df,
    MATRIZES_DIR
    / 'elegibilidade_estatistica_fornecedor.csv'
)

salvar_csv(
    elegibilidade_ug_df,
    MATRIZES_DIR
    / 'elegibilidade_estatistica_ug.csv'
)

salvar_csv(
    perfil_bandas_fornecedor_df,
    MATRIZES_DIR
    / 'perfil_bandas_exposicao_fornecedor.csv'
)

salvar_csv(
    perfil_decis_ug_df,
    MATRIZES_DIR
    / 'perfil_decis_exposicao_ug.csv'
)


resumo_matrizes_v13_df = pd.DataFrame([
    {
        'MATRIZ':
            'UG_FORNECEDOR_ANO',

        'N_UNIDADES':
            len(matriz_fornecedor_v13_df),

        'N_COMPLETAS':
            len(forn_diag),

        'N_COM_ALGUMA_TRILHA':
            int(
                (
                    matriz_fornecedor_v13_df[
                        'N_TRILHAS_ATIVAS'
                    ]
                    > 0
                ).sum()
            ),

        'ESTRATIFICACAO_EXPOSICAO':
            'BANDAS_FIXAS_CONTAGEM',

        'N_ESTRATOS':
            int(
                forn_diag[
                    'BANDA_EXPOSICAO_FORNECEDOR'
                ].nunique()
            ),

        'MEDIANA_EXPOSICAO':
            float(
                matriz_fornecedor_v13_df[
                    'N_COMPRAS_FORNECEDOR'
                ].median()
            ),

        'P95_EXPOSICAO':
            float(
                matriz_fornecedor_v13_df[
                    'N_COMPRAS_FORNECEDOR'
                ].quantile(0.95)
            ),
    },
    {
        'MATRIZ':
            'UG_ANO',

        'N_UNIDADES':
            len(matriz_ug_v13_df),

        'N_COMPLETAS':
            len(ug_diag),

        'N_COM_ALGUMA_TRILHA':
            int(
                (
                    matriz_ug_v13_df[
                        'N_TRILHAS_NUCLEO'
                    ]
                    > 0
                ).sum()
            ),

        'ESTRATIFICACAO_EXPOSICAO':
            'DECIL_ANUAL',

        'N_ESTRATOS':
            int(
                ug_diag[
                    'DECIL_EXPOSICAO_ANUAL'
                ].nunique()
            ),

        'MEDIANA_EXPOSICAO':
            float(
                matriz_ug_v13_df[
                    'N_OPERACOES_EFETIVAS'
                ].median()
            ),

        'P95_EXPOSICAO':
            float(
                matriz_ug_v13_df[
                    'N_OPERACOES_EFETIVAS'
                ].quantile(0.95)
            ),
    },
])

salvar_csv(
    resumo_matrizes_v13_df,
    MATRIZES_DIR
    / 'resumo_matrizes_flags.csv'
)


display(
    resumo_matrizes_v13_df
)

print(
    '📦 Perfil das bandas — fornecedor'
)

display(
    perfil_bandas_fornecedor_df
)

print(
    '📊 Perfil dos decis — UG'
)

display(
    perfil_decis_ug_df
)

print(
    '✅ Matrizes V1.3.2 concluídas: '
    'bandas fixas no fornecedor e decis anuais na UG.'
)

## 3️⃣5️⃣ 03 — Sobreposição entre trilhas e controle de exposição

A sobreposição continua sendo calculada por:

- Jaccard;
- Phi;
- interseção;
- `P(A|B)` e `P(B|A)`;
- análise ano a ano.

A V1.3.2 diferencia explicitamente os dois controles de exposição.

### `UG × fornecedor × ano`

Usa:

`BANDA_EXPOSICAO_FORNECEDOR`

com as seis faixas:

`1`, `2`, `3–4`, `5–9`, `10–19`, `20+`.

### `UG × ano`

Usa:

`DECIL_EXPOSICAO_ANUAL`

com dez grupos anuais.

### Elegibilidade no próprio recorte

Cada par recebe ainda avaliação local de suficiência.

Uma relação em determinada banda/decil é marcada como:

- `SUFICIENTE`;
- `CAUTELA_RARIDADE_NO_RECORTE`.

Assim, um coeficiente calculado em uma classe com poucos positivos permanece disponível, mas não recebe interpretação forte.

> As estratificações controlam parte da oportunidade de disparo; não constituem ajuste causal.

In [ ]:
from itertools import combinations


def metricas_binarias_pares(
    df,
    flags,
    unidade,
    ano=None,
    banda_exposicao_fornecedor=None,
    decil_exposicao_anual=None,
):
    linhas = []
    n = len(df)

    for a, b in combinations(flags, 2):
        xa = (
            df[a]
            .fillna(0)
            .astype(int)
            .to_numpy()
        )

        xb = (
            df[b]
            .fillna(0)
            .astype(int)
            .to_numpy()
        )

        n11 = int(
            (
                (xa == 1)
                &
                (xb == 1)
            ).sum()
        )

        n10 = int(
            (
                (xa == 1)
                &
                (xb == 0)
            ).sum()
        )

        n01 = int(
            (
                (xa == 0)
                &
                (xb == 1)
            ).sum()
        )

        n00 = n - n11 - n10 - n01

        uniao = n11 + n10 + n01

        jaccard = (
            n11 / uniao
            if uniao
            else np.nan
        )

        denom_phi = math.sqrt(
            (n11 + n10)
            * (n01 + n00)
            * (n11 + n01)
            * (n10 + n00)
        )

        phi = (
            (
                n11 * n00
                - n10 * n01
            )
            / denom_phi
            if denom_phi
            else np.nan
        )

        n_a = n11 + n10
        n_b = n11 + n01

        min_pos = (
            MOTOR_CONFIG[
                'diagnostico'
            ]['min_positivos_estatistica']
        )

        min_neg = (
            MOTOR_CONFIG[
                'diagnostico'
            ]['min_negativos_estatistica']
        )

        eleg_a_local = (
            n_a >= min_pos
            and (n - n_a) >= min_neg
        )

        eleg_b_local = (
            n_b >= min_pos
            and (n - n_b) >= min_neg
        )

        linhas.append({
            'UNIDADE_ANALISE':
                unidade,

            'ANO':
                ano,

            'BANDA_EXPOSICAO_FORNECEDOR':
                banda_exposicao_fornecedor,

            'DECIL_EXPOSICAO_ANUAL':
                decil_exposicao_anual,

            'REGRA_A':
                a,

            'REGRA_B':
                b,

            'N_UNIVERSO':
                n,

            'N_A':
                n_a,

            'N_B':
                n_b,

            'ELEGIVEL_A_NO_RECORTE':
                eleg_a_local,

            'ELEGIVEL_B_NO_RECORTE':
                eleg_b_local,

            'DIAGNOSTICO_PAR_NO_RECORTE':
                (
                    'SUFICIENTE'
                    if eleg_a_local and eleg_b_local
                    else 'CAUTELA_RARIDADE_NO_RECORTE'
                ),

            'INTERSECAO':
                n11,

            'A_APENAS':
                n10,

            'B_APENAS':
                n01,

            'NENHUMA':
                n00,

            'UNIAO':
                uniao,

            'JACCARD':
                jaccard,

            'PHI':
                phi,

            'P_B_DADO_A':
                (
                    n11 / n_a
                    if n_a
                    else np.nan
                ),

            'P_A_DADO_B':
                (
                    n11 / n_b
                    if n_b
                    else np.nan
                ),
        })

    return pd.DataFrame(linhas)


def adicionar_elegibilidade_par(
    pares,
    elegibilidade
):
    mapa = (
        elegibilidade
        .set_index('REGRA')[
            'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO'
        ]
        .to_dict()
    )

    out = pares.copy()

    out['ELEGIBILIDADE_A_GLOBAL'] = (
        out['REGRA_A']
        .map(mapa)
    )

    out['ELEGIBILIDADE_B_GLOBAL'] = (
        out['REGRA_B']
        .map(mapa)
    )

    out['DIAGNOSTICO_PAR_GLOBAL'] = np.where(
        (
            out['ELEGIBILIDADE_A_GLOBAL']
            == 'SUFICIENTE'
        )
        &
        (
            out['ELEGIBILIDADE_B_GLOBAL']
            == 'SUFICIENTE'
        ),
        'SUFICIENTE',
        'CAUTELA_RARIDADE'
    )

    return out


def matriz_quadrada_metrica(
    pares,
    flags,
    metrica
):
    out = pd.DataFrame(
        np.eye(len(flags)),
        index=flags,
        columns=flags
    )

    for _, r in pares.iterrows():
        a = r['REGRA_A']
        b = r['REGRA_B']
        v = r[metrica]

        out.loc[a, b] = v
        out.loc[b, a] = v

    return (
        out
        .reset_index()
        .rename(
            columns={
                'index':
                    'REGRA'
            }
        )
    )


# ============================================================
# DIAGNÓSTICO GLOBAL — 2013–2025
# ============================================================

sobreposicao_forn_df = (
    adicionar_elegibilidade_par(
        metricas_binarias_pares(
            forn_diag,
            flags_forn,
            'UG_FORNECEDOR_ANO'
        ),
        elegibilidade_forn_df
    )
)

sobreposicao_ug_df = (
    adicionar_elegibilidade_par(
        metricas_binarias_pares(
            ug_diag,
            flags_ug,
            'UG_ANO'
        ),
        elegibilidade_ug_df
    )
)


# ============================================================
# RELAÇÕES POR ANO
# ============================================================

sobreposicao_forn_ano = []

for ano, g in tqdm(
    forn_diag.groupby('ANO'),
    desc='🔗 Sobreposição fornecedor — por ano',
    unit='ano'
):
    if len(g):
        tmp = metricas_binarias_pares(
            g,
            flags_forn,
            'UG_FORNECEDOR_ANO',
            ano=int(ano)
        )

        sobreposicao_forn_ano.append(
            adicionar_elegibilidade_par(
                tmp,
                elegibilidade_forn_df
            )
        )

sobreposicao_ug_ano = []

for ano, g in tqdm(
    ug_diag.groupby('ANO'),
    desc='🔗 Sobreposição UG — por ano',
    unit='ano'
):
    if len(g):
        tmp = metricas_binarias_pares(
            g,
            flags_ug,
            'UG_ANO',
            ano=int(ano)
        )

        sobreposicao_ug_ano.append(
            adicionar_elegibilidade_par(
                tmp,
                elegibilidade_ug_df
            )
        )

sobreposicao_forn_ano_df = (
    pd.concat(
        sobreposicao_forn_ano,
        ignore_index=True
    )
    if sobreposicao_forn_ano
    else pd.DataFrame()
)

sobreposicao_ug_ano_df = (
    pd.concat(
        sobreposicao_ug_ano,
        ignore_index=True
    )
    if sobreposicao_ug_ano
    else pd.DataFrame()
)


# ============================================================
# RELAÇÕES POR BANDA — FORNECEDOR
# ============================================================

sobreposicao_forn_banda = []

for banda, g in tqdm(
    forn_diag.groupby(
        'BANDA_EXPOSICAO_FORNECEDOR',
        dropna=True
    ),
    desc='📦 Sobreposição fornecedor — bandas',
    unit='banda'
):
    if len(g):
        tmp = metricas_binarias_pares(
            g,
            flags_forn,
            'UG_FORNECEDOR_ANO',
            banda_exposicao_fornecedor=str(banda)
        )

        sobreposicao_forn_banda.append(
            adicionar_elegibilidade_par(
                tmp,
                elegibilidade_forn_df
            )
        )

sobreposicao_forn_banda_df = (
    pd.concat(
        sobreposicao_forn_banda,
        ignore_index=True
    )
    if sobreposicao_forn_banda
    else pd.DataFrame()
)


# ============================================================
# RELAÇÕES POR DECIL — UG
# ============================================================

sobreposicao_ug_decil = []

for decil, g in tqdm(
    ug_diag.groupby(
        'DECIL_EXPOSICAO_ANUAL',
        dropna=True
    ),
    desc='📦 Sobreposição UG — decis',
    unit='decil'
):
    if len(g):
        tmp = metricas_binarias_pares(
            g,
            flags_ug,
            'UG_ANO',
            decil_exposicao_anual=int(decil)
        )

        sobreposicao_ug_decil.append(
            adicionar_elegibilidade_par(
                tmp,
                elegibilidade_ug_df
            )
        )

sobreposicao_ug_decil_df = (
    pd.concat(
        sobreposicao_ug_decil,
        ignore_index=True
    )
    if sobreposicao_ug_decil
    else pd.DataFrame()
)


# ============================================================
# FAMÍLIAS DE EVIDÊNCIA
# ============================================================

sobreposicao_familias_forn_df = (
    metricas_binarias_pares(
        forn_diag,
        ['F1', 'F2', 'F3'],
        'FAMILIAS_UG_FORNECEDOR_ANO'
    )
)

sobreposicao_familias_ug_df = (
    metricas_binarias_pares(
        ug_diag,
        ['F1', 'F2', 'F3', 'F4'],
        'FAMILIAS_UG_ANO'
    )
)


# ============================================================
# MATRIZES QUADRADAS
# ============================================================

jaccard_forn_df = (
    matriz_quadrada_metrica(
        sobreposicao_forn_df,
        flags_forn,
        'JACCARD'
    )
)

phi_forn_df = (
    matriz_quadrada_metrica(
        sobreposicao_forn_df,
        flags_forn,
        'PHI'
    )
)

jaccard_ug_df = (
    matriz_quadrada_metrica(
        sobreposicao_ug_df,
        flags_ug,
        'JACCARD'
    )
)

phi_ug_df = (
    matriz_quadrada_metrica(
        sobreposicao_ug_df,
        flags_ug,
        'PHI'
    )
)


# ============================================================
# EXPORTAR
# ============================================================

saidas = {
    'sobreposicao_trilhas_fornecedor.csv':
        sobreposicao_forn_df,

    'sobreposicao_trilhas_ug.csv':
        sobreposicao_ug_df,

    'sobreposicao_trilhas_fornecedor_por_ano.csv':
        sobreposicao_forn_ano_df,

    'sobreposicao_trilhas_ug_por_ano.csv':
        sobreposicao_ug_ano_df,

    'sobreposicao_trilhas_fornecedor_por_banda_exposicao.csv':
        sobreposicao_forn_banda_df,

    'sobreposicao_trilhas_ug_por_decil_exposicao.csv':
        sobreposicao_ug_decil_df,

    'sobreposicao_familias_fornecedor.csv':
        sobreposicao_familias_forn_df,

    'sobreposicao_familias_ug.csv':
        sobreposicao_familias_ug_df,

    'matriz_jaccard_fornecedor.csv':
        jaccard_forn_df,

    'matriz_phi_fornecedor.csv':
        phi_forn_df,

    'matriz_jaccard_ug.csv':
        jaccard_ug_df,

    'matriz_phi_ug.csv':
        phi_ug_df,
}

for nome, df in saidas.items():
    salvar_csv(
        df,
        SOBREPOSICAO_DIR
        / nome
    )


# ============================================================
# GRÁFICOS
# ============================================================

def grafico_heatmap(
    matriz_df,
    titulo,
    caminho,
    vmin=None,
    vmax=None
):
    if not GERAR_GRAFICOS:
        return

    labels = matriz_df['REGRA'].tolist()

    values = (
        matriz_df
        .drop(columns='REGRA')
        .to_numpy(dtype=float)
    )

    plt.figure(
        figsize=(
            max(7, len(labels) * 1.1),
            max(6, len(labels) * 0.9)
        )
    )

    im = plt.imshow(
        values,
        aspect='auto',
        vmin=vmin,
        vmax=vmax
    )

    plt.colorbar(
        im,
        label='Coeficiente'
    )

    plt.xticks(
        np.arange(len(labels)),
        labels
    )

    plt.yticks(
        np.arange(len(labels)),
        labels
    )

    for r in range(len(labels)):
        for c in range(len(labels)):
            if np.isfinite(values[r, c]):
                plt.text(
                    c,
                    r,
                    f'{values[r,c]:.2f}',
                    ha='center',
                    va='center',
                    fontsize=8
                )

    plt.title(titulo)
    salvar_fig(caminho)


grafico_heatmap(
    jaccard_forn_df,
    'Jaccard — UG × fornecedor × ano',
    SOBREPOSICAO_DIR
    / 'heatmap_jaccard_fornecedor.png',
    vmin=0,
    vmax=1
)

grafico_heatmap(
    phi_forn_df,
    'Phi — UG × fornecedor × ano',
    SOBREPOSICAO_DIR
    / 'heatmap_phi_fornecedor.png',
    vmin=-1,
    vmax=1
)

grafico_heatmap(
    jaccard_ug_df,
    'Jaccard — UG × ano',
    SOBREPOSICAO_DIR
    / 'heatmap_jaccard_ug.png',
    vmin=0,
    vmax=1
)

grafico_heatmap(
    phi_ug_df,
    'Phi — UG × ano',
    SOBREPOSICAO_DIR
    / 'heatmap_phi_ug.png',
    vmin=-1,
    vmax=1
)


print(
    '🔗 Maiores Jaccards — fornecedor'
)

display(
    sobreposicao_forn_df
    .sort_values(
        'JACCARD',
        ascending=False
    )
    .head(15)
)

print(
    '📦 Sobreposição por banda de exposição — fornecedor'
)

display(
    sobreposicao_forn_banda_df[
        sobreposicao_forn_banda_df[
            'DIAGNOSTICO_PAR_NO_RECORTE'
        ]
        == 'SUFICIENTE'
    ]
    .sort_values(
        [
            'BANDA_EXPOSICAO_FORNECEDOR',
            'JACCARD'
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(40)
)

print(
    '✅ Sobreposição V1.3.2 concluída.'
)

## 3️⃣6️⃣ 04 — Multicolinearidade com controle de raridade

VIF e índice de condição permanecem diagnósticos exploratórios.

A V1.3.1 acrescenta uma salvaguarda essencial:

> **flags sem quantidade mínima de positivos e negativos não entram em PCA/VIF.**

Elas permanecem no motor, nas tabelas de contribuição marginal e na sobreposição descritiva.

Essa separação evita interpretar como “componente próprio” ou “independência estatística” aquilo que pode ser apenas efeito de raridade extrema, como observado em T02 na V1.3.

A referência exploratória de VIF continua:

- <2,5 → baixo;
- 2,5 a <5 → moderado;
- 5 a <10 → atenção;
- ≥10 → elevado.

In [ ]:
def calcular_vif_flags(
    df,
    flags
):
    x = (
        df[flags]
        .fillna(0)
        .astype(float)
        .to_numpy()
    )

    linhas = []

    for j, regra in enumerate(flags):
        y = x[:, j]

        if np.var(y) == 0:
            vif = np.nan
            r2 = np.nan

        else:
            outros = np.delete(
                x,
                j,
                axis=1
            )

            design = np.column_stack([
                np.ones(len(y)),
                outros
            ])

            beta, *_ = np.linalg.lstsq(
                design,
                y,
                rcond=None
            )

            pred = design @ beta

            sse = float(
                np.sum(
                    (y - pred) ** 2
                )
            )

            sst = float(
                np.sum(
                    (y - y.mean()) ** 2
                )
            )

            r2 = (
                1 - sse / sst
                if sst > 0
                else np.nan
            )

            if np.isfinite(r2) and r2 < 1:
                vif = 1 / (1 - r2)
            elif np.isfinite(r2) and r2 >= 1:
                vif = np.inf
            else:
                vif = np.nan

        linhas.append({
            'REGRA':
                regra,

            'R2_AUXILIAR':
                r2,

            'VIF':
                vif,
        })

    return pd.DataFrame(linhas)


def diagnostico_vif(v):
    if not np.isfinite(v):
        return 'NAO_CALCULAVEL'

    if v < 2.5:
        return 'BAIXO'

    if v < 5:
        return 'MODERADO'

    if v < 10:
        return 'ATENCAO'

    return 'ELEVADO'


def indice_condicao_flags(
    df,
    flags
):
    if len(flags) < 2:
        return (
            np.nan,
            np.array([])
        )

    corr = (
        df[flags]
        .astype(float)
        .corr()
        .to_numpy()
    )

    eig = np.linalg.eigvalsh(corr)

    positivos = eig[
        eig > 1e-12
    ]

    if len(positivos) < 2:
        return (
            np.nan,
            eig
        )

    indice = math.sqrt(
        positivos.max()
        / positivos.min()
    )

    return (
        indice,
        eig
    )


def phi_maximo_por_regra(
    pares,
    flags
):
    linhas = []

    for regra in flags:
        sub = pares[
            (
                pares['REGRA_A']
                == regra
            )
            |
            (
                pares['REGRA_B']
                == regra
            )
        ].copy()

        if len(sub):
            sub['ABS_PHI'] = (
                sub['PHI']
                .abs()
            )

            melhor = (
                sub.sort_values(
                    'ABS_PHI',
                    ascending=False
                )
                .iloc[0]
            )

            outra = (
                melhor['REGRA_B']
                if melhor['REGRA_A']
                   == regra
                else melhor['REGRA_A']
            )

            linhas.append({
                'REGRA':
                    regra,

                'PHI_MAX_ABS':
                    melhor['ABS_PHI'],

                'REGRA_PHI_MAX':
                    outra,
            })
        else:
            linhas.append({
                'REGRA':
                    regra,

                'PHI_MAX_ABS':
                    np.nan,

                'REGRA_PHI_MAX':
                    None,
            })

    return pd.DataFrame(linhas)


def montar_multicol_completo(
    df,
    flags_todos,
    flags_elegiveis,
    elegibilidade,
    pares,
    unidade
):
    if flags_elegiveis:
        vif_eleg = calcular_vif_flags(
            df,
            flags_elegiveis
        )

        indice, eig = indice_condicao_flags(
            df,
            flags_elegiveis
        )

        phi_eleg = phi_maximo_por_regra(
            pares,
            flags_elegiveis
        )

        calc = (
            vif_eleg
            .merge(
                phi_eleg,
                on='REGRA',
                how='left'
            )
        )
    else:
        indice = np.nan
        eig = np.array([])
        calc = pd.DataFrame(
            columns=[
                'REGRA',
                'R2_AUXILIAR',
                'VIF',
                'PHI_MAX_ABS',
                'REGRA_PHI_MAX',
            ]
        )

    base = (
        elegibilidade[
            [
                'REGRA',
                'N_POSITIVOS',
                'N_NEGATIVOS',
                'PREVALENCIA',
                'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO',
                'ELEGIVEL_PCA_VIF',
            ]
        ]
        .copy()
    )

    out = (
        base
        .merge(
            calc,
            on='REGRA',
            how='left'
        )
    )

    out['DIAGNOSTICO_VIF'] = np.where(
        out['ELEGIVEL_PCA_VIF'],
        out['VIF'].apply(
            diagnostico_vif
        ),
        'NAO_CALCULADO_RARIDADE'
    )

    out['INDICE_CONDICAO_GLOBAL'] = indice
    out['UNIDADE_ANALISE'] = unidade

    return (
        out,
        eig
    )


multicol_forn_df, eig_forn = (
    montar_multicol_completo(
        forn_diag,
        flags_forn,
        flags_forn_modelagem,
        elegibilidade_forn_df,
        sobreposicao_forn_df,
        'UG_FORNECEDOR_ANO'
    )
)

multicol_ug_df, eig_ug = (
    montar_multicol_completo(
        ug_diag,
        flags_ug,
        flags_ug_modelagem,
        elegibilidade_ug_df,
        sobreposicao_ug_df,
        'UG_ANO'
    )
)

eig_condicao_df = pd.concat([
    pd.DataFrame({
        'UNIDADE_ANALISE':
            'UG_FORNECEDOR_ANO',

        'AUTOVALOR':
            np.sort(eig_forn)[::-1],
    }),
    pd.DataFrame({
        'UNIDADE_ANALISE':
            'UG_ANO',

        'AUTOVALOR':
            np.sort(eig_ug)[::-1],
    }),
], ignore_index=True)


salvar_csv(
    multicol_forn_df,
    MULTICOL_DIR
    / 'multicolinearidade_fornecedor.csv'
)

salvar_csv(
    multicol_ug_df,
    MULTICOL_DIR
    / 'multicolinearidade_ug.csv'
)

salvar_csv(
    eig_condicao_df,
    MULTICOL_DIR
    / 'autovalores_indice_condicao.csv'
)

salvar_csv(
    elegibilidade_forn_df,
    MULTICOL_DIR
    / 'regras_excluidas_ou_elegiveis_fornecedor.csv'
)

salvar_csv(
    elegibilidade_ug_df,
    MULTICOL_DIR
    / 'regras_excluidas_ou_elegiveis_ug.csv'
)


print(
    '📐 Multicolinearidade — fornecedor'
)

display(multicol_forn_df)

print(
    '📐 Multicolinearidade — UG'
)

display(multicol_ug_df)

print(
    '✅ VIF/índice de condição calculados '
    'apenas para flags estatisticamente elegíveis.'
)

## 3️⃣7️⃣ 05 — Contribuição marginal

A contribuição marginal continua respondendo:

> **“se esta trilha fosse retirada, quantas unidades deixariam de aparecer na fila?”**

São calculados:

- unidades sinalizadas;
- unidades exclusivas;
- percentual exclusivo;
- tamanho da união;
- perda de unidades se a trilha fosse retirada.

### Controle de exposição V1.3.2

A contribuição marginal é repetida:

- por **banda de exposição** em `UG × fornecedor × ano`;
- por **decil anual** em `UG × ano`.

Isso permite verificar se uma regra acrescenta casos próprios em diferentes níveis de oportunidade de disparo.

> Contribuição marginal zero não elimina automaticamente uma regra; fundamento substantivo continua obrigatório.

In [ ]:
def contribuicao_marginal(
    df,
    flags,
    unidade,
    tipo='TRILHA'
):
    soma = (
        df[
            flags
        ]
        .sum(axis=1)
    )

    uniao_total = int(
        (
            soma > 0
        ).sum()
    )

    linhas = []

    for regra in flags:
        ativo = (
            df[regra]
            == 1
        )

        outras = (
            df[
                [
                    c
                    for c in flags
                    if c != regra
                ]
            ]
            .sum(axis=1)
        )

        exclusivo = (
            ativo
            &
            (
                outras == 0
            )
        )

        n_regra = int(
            ativo.sum()
        )

        n_exclusivos = int(
            exclusivo.sum()
        )

        uniao_sem_regra = int(
            (
                outras > 0
            ).sum()
        )

        linhas.append({
            'UNIDADE_ANALISE':
                unidade,

            'TIPO':
                tipo,

            'REGRA_OU_FAMILIA':
                regra,

            'N_UNIVERSO':
                len(df),

            'N_UNIAO_MOTOR':
                uniao_total,

            'N_SINALIZADOS':
                n_regra,

            'N_EXCLUSIVOS':
                n_exclusivos,

            'CONTRIBUICAO_MARGINAL_PCT':
                (
                    n_exclusivos
                    / n_regra
                    if n_regra
                    else np.nan
                ),

            'N_UNIAO_SEM_REGRA':
                uniao_sem_regra,

            'PERDA_UNIDADES_SE_REMOVER':
                (
                    uniao_total
                    - uniao_sem_regra
                ),

            'ZERO_EXCLUSIVOS':
                (
                    n_exclusivos
                    == 0
                ),
        })

    return pd.DataFrame(
        linhas
    )


marginal_forn_df = (
    contribuicao_marginal(
        forn_diag,
        flags_forn,
        'UG_FORNECEDOR_ANO'
    )
)

marginal_ug_df = (
    contribuicao_marginal(
        ug_diag,
        flags_ug,
        'UG_ANO'
    )
)

marginal_fam_forn_df = (
    contribuicao_marginal(
        forn_diag,
        ['F1', 'F2', 'F3'],
        'UG_FORNECEDOR_ANO',
        tipo='FAMILIA'
    )
)

marginal_fam_ug_df = (
    contribuicao_marginal(
        ug_diag,
        ['F1', 'F2', 'F3', 'F4'],
        'UG_ANO',
        tipo='FAMILIA'
    )
)


for nome, df in {
    'contribuicao_marginal_trilhas_fornecedor.csv':
        marginal_forn_df,

    'contribuicao_marginal_trilhas_ug.csv':
        marginal_ug_df,

    'contribuicao_marginal_familias_fornecedor.csv':
        marginal_fam_forn_df,

    'contribuicao_marginal_familias_ug.csv':
        marginal_fam_ug_df,
}.items():
    salvar_csv(
        df,
        MARGINAL_DIR
        / nome
    )


print(
    '➕ Contribuição marginal — fornecedor'
)

display(
    marginal_forn_df
    .sort_values(
        'CONTRIBUICAO_MARGINAL_PCT',
        ascending=False
    )
)

print(
    '➕ Contribuição marginal — UG'
)

display(
    marginal_ug_df
    .sort_values(
        'CONTRIBUICAO_MARGINAL_PCT',
        ascending=False
    )
)

print(
    '✅ Contribuição marginal concluída.'
)


# ============================================================
# CONTRIBUIÇÃO MARGINAL POR BANDA — FORNECEDOR
# ============================================================

marginal_forn_banda = []

for banda, g in tqdm(
    forn_diag.groupby(
        'BANDA_EXPOSICAO_FORNECEDOR',
        dropna=True
    ),
    desc='➕ Marginal fornecedor — bandas',
    unit='banda'
):
    if len(g):
        tmp = contribuicao_marginal(
            g,
            flags_forn,
            'UG_FORNECEDOR_ANO'
        )

        tmp[
            'BANDA_EXPOSICAO_FORNECEDOR'
        ] = str(banda)

        marginal_forn_banda.append(tmp)

marginal_forn_banda_df = (
    pd.concat(
        marginal_forn_banda,
        ignore_index=True
    )
    if marginal_forn_banda
    else pd.DataFrame()
)


# ============================================================
# CONTRIBUIÇÃO MARGINAL POR DECIL — UG
# ============================================================

marginal_ug_decil = []

for decil, g in tqdm(
    ug_diag.groupby(
        'DECIL_EXPOSICAO_ANUAL',
        dropna=True
    ),
    desc='➕ Marginal UG — decis',
    unit='decil'
):
    if len(g):
        tmp = contribuicao_marginal(
            g,
            flags_ug,
            'UG_ANO'
        )

        tmp[
            'DECIL_EXPOSICAO_ANUAL'
        ] = int(decil)

        marginal_ug_decil.append(tmp)

marginal_ug_decil_df = (
    pd.concat(
        marginal_ug_decil,
        ignore_index=True
    )
    if marginal_ug_decil
    else pd.DataFrame()
)


salvar_csv(
    marginal_forn_banda_df,
    MARGINAL_DIR
    / 'contribuicao_marginal_trilhas_fornecedor_por_banda_exposicao.csv'
)

salvar_csv(
    marginal_ug_decil_df,
    MARGINAL_DIR
    / 'contribuicao_marginal_trilhas_ug_por_decil_exposicao.csv'
)

print(
    '✅ Contribuição marginal V1.3.2: '
    'bandas no fornecedor e decis na UG.'
)

## 3️⃣8️⃣ 06 — PCA exploratória das flags elegíveis

A PCA continua sendo diagnóstico exploratório da estrutura conjunta das regras.

Na V1.3.1, porém, **somente flags classificadas como estatisticamente suficientes entram no cálculo**.

Isso evita que uma regra extremamente rara gere artificialmente um componente quase exclusivo.

A saída mantém:

- tabela de elegibilidade;
- lista de regras utilizadas;
- regras não utilizadas por raridade;
- autovalores;
- variância explicada;
- variância acumulada;
- cargas dos componentes.

### Salvaguardas

- o sinal de um componente é arbitrário;
- PCA não prova equivalência;
- carga compartilhada pode refletir exposição comum;
- fundamento substantivo e contribuição marginal continuam prioritários.

In [ ]:
def pca_flags(
    df,
    flags,
    unidade
):
    x = (
        df[
            flags
        ]
        .fillna(0)
        .astype(float)
        .to_numpy()
    )

    medias = x.mean(
        axis=0
    )

    desvios = x.std(
        axis=0,
        ddof=0
    )

    validas = (
        desvios > 0
    )

    flags_validas = [
        f
        for f, ok
        in zip(
            flags,
            validas
        )
        if ok
    ]

    x = x[
        :,
        validas
    ]

    medias = medias[
        validas
    ]

    desvios = desvios[
        validas
    ]

    z = (
        x - medias
    ) / desvios

    corr = np.corrcoef(
        z,
        rowvar=False
    )

    eigvals, eigvecs = (
        np.linalg.eigh(
            corr
        )
    )

    ordem = np.argsort(
        eigvals
    )[::-1]

    eigvals = eigvals[
        ordem
    ]

    eigvecs = eigvecs[
        :,
        ordem
    ]

    eigvals_clip = np.clip(
        eigvals,
        0,
        None
    )

    explained = (
        eigvals_clip
        / eigvals_clip.sum()
    )

    cumulative = np.cumsum(
        explained
    )

    comps = [
        f'CP{i+1}'
        for i in range(
            len(
                eigvals
            )
        )
    ]

    variancia = pd.DataFrame({
        'UNIDADE_ANALISE':
            unidade,

        'COMPONENTE':
            comps,

        'AUTOVALOR':
            eigvals,

        'VARIANCIA_EXPLICADA':
            explained,

        'VARIANCIA_ACUMULADA':
            cumulative,
    })

    # cargas = autovetor * sqrt(autovalor)
    cargas = (
        eigvecs
        * np.sqrt(
            eigvals_clip
        )
    )

    cargas_df = pd.DataFrame(
        cargas,
        index=flags_validas,
        columns=comps
    )

    cargas_df.insert(
        0,
        'REGRA',
        cargas_df.index
    )

    cargas_df.insert(
        0,
        'UNIDADE_ANALISE',
        unidade
    )

    cargas_df.reset_index(
        drop=True,
        inplace=True
    )

    return (
        variancia,
        cargas_df,
        corr,
        flags_validas
    )


pca_var_forn_df, pca_load_forn_df, corr_pca_forn, pca_flags_forn = (
    pca_flags(
        forn_diag,
        flags_forn_modelagem,
        'UG_FORNECEDOR_ANO'
    )
)

pca_var_ug_df, pca_load_ug_df, corr_pca_ug, pca_flags_ug = (
    pca_flags(
        ug_diag,
        flags_ug_modelagem,
        'UG_ANO'
    )
)


salvar_csv(
    pca_var_forn_df,
    PCA_DIR
    / 'pca_variancia_fornecedor.csv'
)

salvar_csv(
    pca_load_forn_df,
    PCA_DIR
    / 'pca_cargas_fornecedor.csv'
)

salvar_csv(
    pca_var_ug_df,
    PCA_DIR
    / 'pca_variancia_ug.csv'
)

salvar_csv(
    pca_load_ug_df,
    PCA_DIR
    / 'pca_cargas_ug.csv'
)

pca_composicao_df = pd.concat([
    elegibilidade_forn_df.assign(
        PCA_UNIDADE='UG_FORNECEDOR_ANO'
    ),
    elegibilidade_ug_df.assign(
        PCA_UNIDADE='UG_ANO'
    ),
], ignore_index=True)

salvar_csv(
    pca_composicao_df,
    PCA_DIR
    / 'pca_elegibilidade_regras.csv'
)


# ============================================================
# GRÁFICOS
# ============================================================

def grafico_variancia_pca(
    df,
    titulo,
    caminho
):
    if not GERAR_GRAFICOS:
        return

    plt.figure(
        figsize=(10, 5.5)
    )

    plt.bar(
        df[
            'COMPONENTE'
        ],
        df[
            'VARIANCIA_EXPLICADA'
        ]
    )

    plt.plot(
        df[
            'COMPONENTE'
        ],
        df[
            'VARIANCIA_ACUMULADA'
        ],
        marker='o',
        label='Acumulada'
    )

    plt.ylabel(
        'Proporção da variância'
    )

    plt.title(
        titulo
    )

    plt.legend()

    salvar_fig(
        caminho
    )


grafico_variancia_pca(
    pca_var_forn_df,
    'PCA — variância explicada — fornecedor',
    PCA_DIR
    / 'pca_variancia_fornecedor.png'
)

grafico_variancia_pca(
    pca_var_ug_df,
    'PCA — variância explicada — UG',
    PCA_DIR
    / 'pca_variancia_ug.png'
)


print(
    '🧩 PCA — fornecedor'
)

display(
    pca_var_forn_df
)

display(
    pca_load_forn_df
)

print(
    '🧩 PCA — UG'
)

display(
    pca_var_ug_df
)

display(
    pca_load_ug_df
)

print(
    '✅ PCA exploratória concluída.'
)

## 3️⃣9️⃣ 07 — Sensibilidade do motor e contrato semântico uniforme

Esta etapa continua separando parâmetros bloqueados de parâmetros experimentais, mas a V1.3.1 corrige uma ambiguidade importante.

Na V1.3, `N_RESULTADOS` podia representar:

- grupos;
- episódios;
- tamanho de população Benford;
- transações próximas a limite.

Essas unidades não são equivalentes.

### Contrato V1.3.1

Cada cenário passa a declarar explicitamente:

- `METRICA`;
- `UNIDADE_CONTAGEM`;
- `VALOR_METRICA`;
- `N_SINAIS_TOTAL_CENARIO`;
- `N_SINAIS_BASELINE`;
- `COMPARAVEL_COM_BASELINE`;
- `TIPO_CONTROLE`;
- `BASELINE`;
- `OBSERVACAO`.

### T05

A grade de 160 combinações permanece como **grade de calibração de grupos representativos**. Ela não é apresentada como se seus totais fossem episódios finais.

O simulador de produção deverá aplicar a lógica completa de construção/deduplicação quando o usuário escolher um cenário específico.

### T08

Os cenários de arredondamento medem **população, MAD e classificação**, não quantidade de sinais.

### T09

A faixa ajustável modifica apenas a categoria de proximidade. O total comparável do cenário é:

`proximidade + no limite + acima`.

Os limites legais continuam bloqueados.

In [ ]:
# ============================================================
# HELPERS DO CONTRATO DE SENSIBILIDADE
# ============================================================

sens_linhas = []


def adicionar_cenario(
    trilha,
    parametro,
    valor_cenario,
    metrica,
    unidade_contagem,
    valor_metrica,
    n_sinais_total_cenario,
    n_sinais_baseline,
    comparavel_com_baseline,
    tipo_controle,
    baseline,
    observacao='',
    metrica_secundaria=None,
    valor_metrica_secundaria=None,
):
    sens_linhas.append({
        'TRILHA':
            trilha,

        'PARAMETRO':
            parametro,

        'VALOR_CENARIO':
            valor_cenario,

        'METRICA':
            metrica,

        'UNIDADE_CONTAGEM':
            unidade_contagem,

        'VALOR_METRICA':
            valor_metrica,

        'METRICA_SECUNDARIA':
            metrica_secundaria,

        'VALOR_METRICA_SECUNDARIA':
            valor_metrica_secundaria,

        'N_SINAIS_TOTAL_CENARIO':
            n_sinais_total_cenario,

        'N_SINAIS_BASELINE':
            n_sinais_baseline,

        'COMPARAVEL_COM_BASELINE':
            bool(
                comparavel_com_baseline
            ),

        'TIPO_CONTROLE':
            tipo_controle,

        'BASELINE':
            bool(
                baseline
            ),

        'OBSERVACAO':
            observacao,
    })


# ============================================================
# T01 / T02 — BLOQUEADOS
# ============================================================

adicionar_cenario(
    'T01',
    'definicao_final_semana',
    'SABADO_DOMINGO',
    'N_SINAIS',
    'TRANSACOES',
    BASELINE_V12['T01'],
    BASELINE_V12['T01'],
    BASELINE_V12['T01'],
    True,
    'BLOQUEADO',
    True,
    'Condição determinística da regra.'
)

adicionar_cenario(
    'T02',
    'codigo_transacao',
    CODIGO_COMPRA_PARCELADA,
    'N_SINAIS',
    'TRANSACOES',
    BASELINE_V12['T02'],
    BASELINE_V12['T02'],
    BASELINE_V12['T02'],
    True,
    'BLOQUEADO',
    True,
    'Código oficial do dicionário de dados.'
)


# ============================================================
# T03 — mínimo de ocorrências
# ============================================================

t03_sens_base = con.execute(
    f"""
    SELECT
        N_TRANSACOES
    FROM read_parquet(
        '{t03_grupos_sql}'
    )
    """
).df()

for limiar in (
    MOTOR_CONFIG[
        'sensibilidade'
    ]['T03_min_ocorrencias']
):
    n = int(
        (
            t03_sens_base[
                'N_TRANSACOES'
            ]
            >= limiar
        ).sum()
    )

    adicionar_cenario(
        'T03',
        'min_ocorrencias',
        limiar,
        'N_GRUPOS_SINALIZADOS',
        'GRUPOS_T03',
        n,
        n,
        BASELINE_V12['T03'],
        True,
        'EXPERIMENTAL',
        (
            limiar
            == CONFIG['T03']
               ['min_ocorrencias']
        ),
        'O cenário é subconjunto direto da saída T03 baseline.'
    )


# ============================================================
# T04 — mínimo de portadores
# ============================================================

t04_sens_base = con.execute(
    f"""
    SELECT
        N_PORTADORES
    FROM read_parquet(
        '{t04_grupos_sql}'
    )
    """
).df()

for limiar in (
    MOTOR_CONFIG[
        'sensibilidade'
    ]['T04_min_portadores']
):
    n = int(
        (
            t04_sens_base[
                'N_PORTADORES'
            ]
            >= limiar
        ).sum()
    )

    adicionar_cenario(
        'T04',
        'min_portadores',
        limiar,
        'N_GRUPOS_SINALIZADOS',
        'GRUPOS_T04',
        n,
        n,
        BASELINE_V12['T04'],
        True,
        'EXPERIMENTAL',
        (
            limiar
            == CONFIG['T04']
               ['min_portadores']
        ),
        'O cenário é subconjunto direto da saída T04 baseline.'
    )


# ============================================================
# T05 — GRADE DE CALIBRAÇÃO, NÃO N DE EPISÓDIOS
# ============================================================

t05_sens_v13_df = (
    t05_sensibilidade_df
    .copy()
)

t05_sens_v13_df[
    'TRILHA'
] = 'T05'

t05_sens_v13_df[
    'TIPO_CONTROLE'
] = 'CALIBRACAO'

t05_sens_v13_df[
    'UNIDADE_CONTAGEM'
] = 'GRUPOS_REPRESENTATIVOS_CALIBRACAO'

t05_sens_v13_df[
    'COMPARAVEL_COM_BASELINE'
] = False

t05_sens_v13_df[
    'N_SINAIS_BASELINE'
] = BASELINE_V12['T05']

t05_sens_v13_df[
    'OBSERVACAO'
] = (
    'A grade seleciona uma janela representativa por '
    'UG-fornecedor-ano e não reproduz a deduplicação final '
    'de episódios. Não comparar diretamente com 1.693 episódios.'
)

t05_sens_v13_df[
    'BASELINE_PARAMETROS'
] = (
    (
        t05_sens_v13_df[
            'janela_dias'
        ]
        == CONFIG['T05']
           ['janela_dias']
    )
    &
    (
        t05_sens_v13_df[
            'min_transacoes'
        ]
        == CONFIG['T05']
           ['min_transacoes']
    )
    &
    (
        t05_sens_v13_df[
            'min_portadores'
        ]
        == CONFIG['T05']
           ['min_portadores']
    )
    &
    np.isclose(
        t05_sens_v13_df[
            'cv_limite'
        ],
        CONFIG['T05']
        ['cv_base']
    )
)

# Linha explícita do contrato para o baseline final.
adicionar_cenario(
    'T05',
    'regra_final_deduplicada',
    '30d|N>=5|port>=2|CV<=0.20',
    'N_EPISODIOS_FINAIS',
    'EPISODIOS_T05',
    BASELINE_V12['T05'],
    BASELINE_V12['T05'],
    BASELINE_V12['T05'],
    True,
    'BASELINE_METODOLOGICA',
    True,
    (
        'O simulador futuro deve reexecutar a lógica completa '
        'de episódios para produzir N comparável.'
    )
)


# ============================================================
# T06 — Top-1 share
# ============================================================

t06_sens_base = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{t06_ind_sql}'
    )
    """
).df()

for share in (
    MOTOR_CONFIG[
        'sensibilidade'
    ]['T06_top1_share']
):
    n = int(
        (
            (
                t06_sens_base[
                    'N_COMPRAS_IDENTIFICADAS'
                ]
                >= CONFIG['T06']
                   ['min_compras_identificadas']
            )
            &
            (
                t06_sens_base[
                    'N_FORNECEDORES'
                ]
                >= CONFIG['T06']
                   ['min_fornecedores']
            )
            &
            (
                t06_sens_base[
                    'COBERTURA_VALOR'
                ]
                >= CONFIG['T06']
                   ['cobertura_min']
            )
            &
            (
                t06_sens_base[
                    'TOP1_SHARE'
                ]
                >= share
            )
        ).sum()
    )

    adicionar_cenario(
        'T06',
        'top1_share',
        share,
        'N_UG_ANO_SINALIZADOS',
        'UG_ANO',
        n,
        n,
        BASELINE_V12['T06'],
        True,
        'EXPERIMENTAL',
        np.isclose(
            share,
            CONFIG['T06']
            ['share_base']
        ),
        'Demais critérios de elegibilidade permanecem fixos.'
    )


# ============================================================
# T07 — dias × percentil
# ============================================================

t07_sens_base = con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{paths[T07_RECORRENCIA]}'
    )
    """
).df()

t07_sens_linhas = []

for min_dias in (
    MOTOR_CONFIG[
        'sensibilidade'
    ]['T07_min_dias']
):
    for q in (
        MOTOR_CONFIG[
            'sensibilidade'
        ]['T07_percentis']
    ):
        n_total = 0

        for ano, g in (
            t07_sens_base
            .groupby(
                'ANO_TRANSACAO'
            )
        ):
            if (
                len(g)
                < CONFIG['T07']
                  ['min_comparaveis_ano']
            ):
                continue

            limiar_q = float(
                g[
                    'N_DIAS_MULTISAQUE'
                ]
                .quantile(q)
            )

            n_total += int(
                (
                    (
                        g[
                            'N_DIAS_MULTISAQUE'
                        ]
                        >= min_dias
                    )
                    &
                    (
                        g[
                            'N_DIAS_MULTISAQUE'
                        ]
                        >= limiar_q
                    )
                ).sum()
            )

        baseline_flag = (
            min_dias
            == CONFIG['T07']
               ['min_dias_recorrencia']
            and
            np.isclose(
                q,
                CONFIG['T07']
                ['percentil_priorizacao']
            )
        )

        t07_sens_linhas.append({
            'TRILHA':
                'T07',

            'MIN_DIAS':
                min_dias,

            'PERCENTIL':
                q,

            'METRICA':
                'N_PORTADOR_ANO_PRIORIZADOS',

            'UNIDADE_CONTAGEM':
                'PORTADOR_ANO',

            'VALOR_METRICA':
                n_total,

            'N_SINAIS_TOTAL_CENARIO':
                n_total,

            'N_SINAIS_BASELINE':
                BASELINE_V12['T07'],

            'COMPARAVEL_COM_BASELINE':
                True,

            'TIPO_CONTROLE':
                'EXPERIMENTAL',

            'BASELINE':
                baseline_flag,
        })

        adicionar_cenario(
            'T07',
            'min_dias_percentil',
            f'{min_dias}|{q:.2f}',
            'N_PORTADOR_ANO_PRIORIZADOS',
            'PORTADOR_ANO',
            n_total,
            n_total,
            BASELINE_V12['T07'],
            True,
            'EXPERIMENTAL',
            baseline_flag,
            'Cenário recalcula limiar relativo dentro de cada ano.'
        )

t07_sens_v13_df = pd.DataFrame(
    t07_sens_linhas
)


# ============================================================
# T08 — SENSIBILIDADE DE POPULAÇÃO, NÃO DE SINAIS
# ============================================================

for _, r in (
    benford_arredondamento_global_df
    .iterrows()
):
    adicionar_cenario(
        'T08',
        'populacao_sensibilidade_arredondamento',
        r['CENARIO'],
        'N_POPULACAO_D12',
        'TRANSACOES_ANALISADAS_BENFORD',
        int(r['N']),
        np.nan,
        BASELINE_V12['T08'],
        False,
        'DIAGNOSTICO_NAO_PARAMETRO',
        (
            r['CENARIO']
            == 'A_TODOS_GE10'
        ),
        (
            'N representa tamanho da população D12; '
            'não representa quantidade de sinais T08.'
        ),
        metrica_secundaria='MAD_D12',
        valor_metrica_secundaria=float(
            r['MAD']
        )
    )


# ============================================================
# T09 — PROXIMIDADE + TOTAL COMPARÁVEL
# ============================================================

t09_sens_base = con.execute(
    f"""
    SELECT
        RATIO_PV_COMPRAS,
        RATIO_PV_ENGENHARIA,
        STATUS_COMPRAS,
        STATUS_ENGENHARIA

    FROM read_parquet(
        '{t09_classificadas_path_sql}'
    )
    """
).df()

fixos_t09 = (
    (
        t09_sens_base[
            'STATUS_COMPRAS'
        ].isin(
            ['NO_LIMITE', 'ACIMA_LIMITE']
        )
    )
    |
    (
        t09_sens_base[
            'STATUS_ENGENHARIA'
        ].isin(
            ['NO_LIMITE', 'ACIMA_LIMITE']
        )
    )
)

n_fixos_t09 = int(
    fixos_t09.sum()
)

for faixa in (
    MOTOR_CONFIG[
        'sensibilidade'
    ]['T09_faixa_proximidade']
):
    prox = (
        (
            t09_sens_base[
                'RATIO_PV_COMPRAS'
            ]
            < 1
        )
        &
        (
            t09_sens_base[
                'RATIO_PV_ENGENHARIA'
            ]
            < 1
        )
        &
        (
            (
                t09_sens_base[
                    'RATIO_PV_COMPRAS'
                ]
                >= faixa
            )
            |
            (
                t09_sens_base[
                    'RATIO_PV_ENGENHARIA'
                ]
                >= faixa
            )
        )
    )

    n_prox = int(
        prox.sum()
    )

    n_total = (
        n_fixos_t09
        + n_prox
    )

    adicionar_cenario(
        'T09',
        'faixa_proximidade',
        faixa,
        'N_TRANSACOES_PROXIMAS',
        'TRANSACOES',
        n_prox,
        n_total,
        BASELINE_V12['T09'],
        True,
        'EXPERIMENTAL',
        np.isclose(
            faixa,
            CONFIG['T09']
            ['faixa_inferior']
        ),
        (
            'N_SINAIS_TOTAL_CENARIO soma proximidade '
            'com casos fixos no limite/acima.'
        ),
        metrica_secundaria='N_FIXOS_NO_LIMITE_OU_ACIMA',
        valor_metrica_secundaria=n_fixos_t09
    )


# ============================================================
# CONTRATO CONSOLIDADO
# ============================================================

sens_motor_df = pd.DataFrame(
    sens_linhas
)

sens_motor_df[
    'VARIACAO_SINAIS_N'
] = np.where(
    sens_motor_df[
        'COMPARAVEL_COM_BASELINE'
    ]
    &
    sens_motor_df[
        'N_SINAIS_TOTAL_CENARIO'
    ].notna(),
    (
        sens_motor_df[
            'N_SINAIS_TOTAL_CENARIO'
        ]
        - sens_motor_df[
            'N_SINAIS_BASELINE'
        ]
    ),
    np.nan
)

sens_motor_df[
    'VARIACAO_SINAIS_PCT'
] = np.where(
    sens_motor_df[
        'COMPARAVEL_COM_BASELINE'
    ]
    &
    sens_motor_df[
        'N_SINAIS_TOTAL_CENARIO'
    ].notna()
    &
    (
        sens_motor_df[
            'N_SINAIS_BASELINE'
        ]
        > 0
    ),
    (
        sens_motor_df[
            'VARIACAO_SINAIS_N'
        ]
        / sens_motor_df[
            'N_SINAIS_BASELINE'
        ]
    ),
    np.nan
)


# Catálogo de controles da interface
parametros_dashboard_v13_df = pd.DataFrame([
    {
        'TRILHA': 'T01',
        'PARAMETRO': 'sábado/domingo',
        'TIPO': 'BLOQUEADO',
        'JUSTIFICATIVA':
            'Condição determinística da regra.'
    },
    {
        'TRILHA': 'T02',
        'PARAMETRO': 'código da transação',
        'TIPO': 'BLOQUEADO',
        'JUSTIFICATIVA':
            'Código oficial do dicionário de dados.'
    },
    {
        'TRILHA': 'T03',
        'PARAMETRO': 'mínimo de ocorrências',
        'TIPO': 'EXPERIMENTAL',
        'JUSTIFICATIVA':
            'Parâmetro de priorização analítica.'
    },
    {
        'TRILHA': 'T04',
        'PARAMETRO': 'mínimo de portadores',
        'TIPO': 'EXPERIMENTAL',
        'JUSTIFICATIVA':
            'Parâmetro de priorização analítica.'
    },
    {
        'TRILHA': 'T05',
        'PARAMETRO': 'janela / N / portadores / CV',
        'TIPO': 'EXPERIMENTAL_RECALCULO_COMPLETO',
        'JUSTIFICATIVA':
            (
                'A grade de calibração não é diretamente '
                'comparável aos episódios finais; o dashboard '
                'deverá recalcular a lógica completa.'
            )
    },
    {
        'TRILHA': 'T06',
        'PARAMETRO': 'Top-1 e filtros de cobertura',
        'TIPO': 'EXPERIMENTAL',
        'JUSTIFICATIVA':
            'Indicadores estruturais, não limites legais.'
    },
    {
        'TRILHA': 'T07',
        'PARAMETRO': 'mínimo de dias / percentil',
        'TIPO': 'EXPERIMENTAL',
        'JUSTIFICATIVA':
            'Priorização relativa anual.'
    },
    {
        'TRILHA': 'T08',
        'PARAMETRO': 'probabilidades e fórmulas Benford',
        'TIPO': 'BLOQUEADO_METODO',
        'JUSTIFICATIVA':
            'Definição metodológica; cenários são diagnósticos.'
    },
    {
        'TRILHA': 'T09',
        'PARAMETRO': 'limites normativos',
        'TIPO': 'BLOQUEADO',
        'JUSTIFICATIVA':
            'Derivados da norma vigente.'
    },
    {
        'TRILHA': 'T09',
        'PARAMETRO': 'faixa de proximidade',
        'TIPO': 'EXPERIMENTAL',
        'JUSTIFICATIVA':
            'Faixa de triagem, não limite legal.'
    },
])


salvar_csv(
    sens_motor_df,
    SENSIBILIDADE_MOTOR_DIR
    / 'sensibilidade_motor_contrato.csv'
)

salvar_csv(
    t05_sens_v13_df,
    SENSIBILIDADE_MOTOR_DIR
    / 'sensibilidade_T05_grade_160_calibracao.csv'
)

salvar_csv(
    t07_sens_v13_df,
    SENSIBILIDADE_MOTOR_DIR
    / 'sensibilidade_T07_dias_percentis.csv'
)

salvar_csv(
    parametros_dashboard_v13_df,
    SENSIBILIDADE_MOTOR_DIR
    / 'catalogo_parametros_interface.csv'
)

display(
    sens_motor_df
)

print(
    '✅ Contrato semântico de sensibilidade concluído.'
)

## 4️⃣0️⃣ 08 — Validação humana estratificada e ciclo de feedback

A validação é a etapa necessária para começar a falar em assertividade.

Os status permanecem:

- `NAO_VALIDADO`
- `EM_ANALISE`
- `CONFIRMADO`
- `JUSTIFICADO`
- `FALSO_POSITIVO`
- `ERRO_DADO`
- `INCONCLUSIVO`

### Amostragem V1.3.1

A amostra passa a ser construída por:

`TRILHA × NIVEL_TRIAGEM`

com até **30 sinais por trilha**, distribuídos de forma balanceada entre os níveis efetivamente observados.

Quando um estrato possui menos casos que sua cota inicial, a sobra é redistribuída entre os demais estratos da mesma trilha.

A seleção dentro do estrato permanece determinística por hash.

### Peso amostral

Para cada estrato:

\[
w_h=\frac{N_h}{n_h}
\]

Esse peso é exportado junto à amostra.

Ele permitirá, futuramente, estimativas ponderadas caso a amostragem estratificada seja utilizada para inferência sobre o universo da trilha.

> Na fase inicial, recomenda-se reportar primeiro resultados por trilha e por nível de triagem. Uma taxa global do motor exige desenho amostral e ponderação explícitos.

In [ ]:
status_validacao_df = pd.DataFrame([
    {
        'STATUS_VALIDACAO':
            'NAO_VALIDADO',
        'DEFINICAO':
            'Nenhuma decisão humana registrada.',
        'EFEITO_MOTOR':
            'Nenhum.'
    },
    {
        'STATUS_VALIDACAO':
            'EM_ANALISE',
        'DEFINICAO':
            'Evidências em coleta ou exame.',
        'EFEITO_MOTOR':
            'Nenhum até decisão.'
    },
    {
        'STATUS_VALIDACAO':
            'CONFIRMADO',
        'DEFINICAO':
            'Evidência suficiente de desconformidade no caso examinado.',
        'EFEITO_MOTOR':
            'Sustenta análise da regra; compõe taxa de confirmação.'
    },
    {
        'STATUS_VALIDACAO':
            'JUSTIFICADO',
        'DEFINICAO':
            'Condição observada possui exceção/contexto válido documentado.',
        'EFEITO_MOTOR':
            'Pode revelar filtro legítimo de elegibilidade.'
    },
    {
        'STATUS_VALIDACAO':
            'FALSO_POSITIVO',
        'DEFINICAO':
            'Regra aplicada aos dados, mas caso é regular no contexto real.',
        'EFEITO_MOTOR':
            'Reexaminar limiar, população ou atributo ausente.'
    },
    {
        'STATUS_VALIDACAO':
            'ERRO_DADO',
        'DEFINICAO':
            'Sinal decorre de problema de cadastro, conversão ou origem.',
        'EFEITO_MOTOR':
            'Corrigir normalização/origem, não o limiar.'
    },
    {
        'STATUS_VALIDACAO':
            'INCONCLUSIVO',
        'DEFINICAO':
            'Evidência disponível ainda não permite decisão.',
        'EFEITO_MOTOR':
            'Reavaliar pacote mínimo de evidências.'
    },
])


protocolo_recalibracao_df = pd.DataFrame([
    {
        'STATUS_VALIDACAO': 'CONFIRMADO',
        'ACAO':
            'Preservar evidência e monitorar estabilidade da regra.'
    },
    {
        'STATUS_VALIDACAO': 'JUSTIFICADO',
        'ACAO':
            'Verificar se a exceção legítima é recorrente e se deve entrar na elegibilidade.'
    },
    {
        'STATUS_VALIDACAO': 'FALSO_POSITIVO',
        'ACAO':
            'Reexaminar um parâmetro por vez e medir volume, Jaccard e contribuição marginal.'
    },
    {
        'STATUS_VALIDACAO': 'ERRO_DADO',
        'ACAO':
            'Corrigir camada de dados; não recalibrar regra por causa de erro de origem.'
    },
    {
        'STATUS_VALIDACAO': 'INCONCLUSIVO',
        'ACAO':
            'Revisar evidência mínima exigida antes de alterar parâmetros.'
    },
])


VALIDACOES_CSV = (
    VALIDACAO_DIR
    / 'validacoes_registradas.csv'
)

colunas_validacao = [
    'ID_VALIDACAO',
    'ID_SINAL',
    'CODIGO_TRILHA',
    'NIVEL_TRIAGEM',
    'ESTRATO_VALIDACAO',
    'PESO_AMOSTRAL',
    'STATUS_VALIDACAO',
    'RESPONSAVEL',
    'DATA_DECISAO',
    'EVIDENCIA_FONTE',
    'EVIDENCIA_URL_OU_REFERENCIA',
    'JUSTIFICATIVA',
    'VERSAO_REGRA',
    'VERSAO_MOTOR',
    'OBSERVACAO',
]

if not VALIDACOES_CSV.exists():
    salvar_csv(
        pd.DataFrame(
            columns=colunas_validacao
        ),
        VALIDACOES_CSV
    )


# ============================================================
# HELPERS DE AMOSTRAGEM
# ============================================================

def ordenar_deterministicamente(
    df,
    salt
):
    out = df.copy()

    out['_ordem_amostra'] = (
        out['ID_SINAL']
        .astype(str)
        .apply(
            lambda x:
                hashlib.sha256(
                    (
                        salt
                        + '|'
                        + x
                    )
                    .encode('utf-8')
                )
                .hexdigest()
        )
    )

    return out.sort_values(
        '_ordem_amostra'
    )


def cotas_balanceadas(
    tamanhos,
    n_total
):
    """
    Distribui n_total entre estratos de forma balanceada,
    respeitando a capacidade de cada estrato.
    """
    estratos = list(
        sorted(
            tamanhos.keys()
        )
    )

    if not estratos:
        return {}

    n_total = min(
        n_total,
        sum(
            tamanhos.values()
        )
    )

    k = len(estratos)
    base = n_total // k
    resto = n_total % k

    cotas = {}

    for idx, estrato in enumerate(estratos):
        desejada = (
            base
            + (
                1
                if idx < resto
                else 0
            )
        )

        cotas[estrato] = min(
            desejada,
            tamanhos[estrato]
        )

    faltam = (
        n_total
        - sum(cotas.values())
    )

    while faltam > 0:
        houve_alocacao = False

        for estrato in estratos:
            if cotas[estrato] < tamanhos[estrato]:
                cotas[estrato] += 1
                faltam -= 1
                houve_alocacao = True

                if faltam == 0:
                    break

        if not houve_alocacao:
            break

    return cotas


# ============================================================
# AMOSTRA TRILHA × NÍVEL DE TRIAGEM
# ============================================================

amostras = []
plano_amostral = []

n_amostra_trilha = (
    MOTOR_CONFIG[
        'validacao'
    ]['n_amostra_por_trilha']
)

for trilha, g_trilha in (
    trilha_resultados_df
    .groupby(
        'CODIGO_TRILHA'
    )
):
    g_trilha = g_trilha.copy()

    g_trilha[
        'NIVEL_TRIAGEM'
    ] = (
        g_trilha[
            'NIVEL_TRIAGEM'
        ]
        .fillna(
            'SEM_NIVEL'
        )
        .astype(str)
    )

    tamanhos = (
        g_trilha[
            'NIVEL_TRIAGEM'
        ]
        .value_counts()
        .to_dict()
    )

    cotas = cotas_balanceadas(
        tamanhos,
        n_amostra_trilha
    )

    for nivel, g_nivel in (
        g_trilha
        .groupby(
            'NIVEL_TRIAGEM'
        )
    ):
        n_pop = len(g_nivel)
        n_sel = int(
            cotas.get(
                nivel,
                0
            )
        )

        if n_sel <= 0:
            continue

        selecionada = (
            ordenar_deterministicamente(
                g_nivel,
                (
                    'AMOSTRA_V131'
                    f'|{trilha}'
                    f'|{nivel}'
                )
            )
            .head(n_sel)
            .drop(
                columns='_ordem_amostra'
            )
            .copy()
        )

        estrato = (
            f'{trilha}|{nivel}'
        )

        peso = (
            n_pop / n_sel
            if n_sel
            else np.nan
        )

        selecionada[
            'ESTRATO_VALIDACAO'
        ] = estrato

        selecionada[
            'N_POP_ESTRATO'
        ] = n_pop

        selecionada[
            'N_AMOSTRA_ESTRATO'
        ] = n_sel

        selecionada[
            'PESO_AMOSTRAL'
        ] = peso

        amostras.append(
            selecionada
        )

        plano_amostral.append({
            'CODIGO_TRILHA':
                trilha,

            'NIVEL_TRIAGEM':
                nivel,

            'ESTRATO_VALIDACAO':
                estrato,

            'N_POP_ESTRATO':
                n_pop,

            'N_AMOSTRA_ESTRATO':
                n_sel,

            'PESO_AMOSTRAL':
                peso,

            'FRACAO_AMOSTRAL':
                (
                    n_sel / n_pop
                    if n_pop
                    else np.nan
                ),
        })


amostra_validacao_v13_df = (
    pd.concat(
        amostras,
        ignore_index=True
    )
    if amostras
    else pd.DataFrame()
)

plano_amostral_v13_df = (
    pd.DataFrame(
        plano_amostral
    )
)


salvar_csv(
    amostra_validacao_v13_df,
    VALIDACAO_DIR
    / 'amostra_validacao_trilha_nivel.csv'
)

salvar_csv(
    plano_amostral_v13_df,
    VALIDACAO_DIR
    / 'plano_amostral_trilha_nivel.csv'
)

salvar_csv(
    status_validacao_df,
    VALIDACAO_DIR
    / 'dicionario_status_validacao.csv'
)

salvar_csv(
    protocolo_recalibracao_df,
    VALIDACAO_DIR
    / 'protocolo_recalibracao.csv'
)


# ============================================================
# MÉTRICAS DE VALIDAÇÃO
# ============================================================

def intervalo_wilson(
    sucessos,
    n,
    z=1.96
):
    if n <= 0:
        return (
            np.nan,
            np.nan
        )

    p = sucessos / n

    denom = (
        1
        + z**2 / n
    )

    centro = (
        p
        + z**2 / (2*n)
    ) / denom

    margem = (
        z
        * math.sqrt(
            (
                p * (1-p)
                + z**2/(4*n)
            )
            / n
        )
        / denom
    )

    return (
        max(
            0,
            centro - margem
        ),
        min(
            1,
            centro + margem
        )
    )


def resumir_validacoes(
    caminho=VALIDACOES_CSV
):
    df = pd.read_csv(
        caminho,
        sep=';',
        encoding='utf-8-sig',
        dtype=str
    )

    if not len(df):
        return pd.DataFrame(
            columns=[
                'CODIGO_TRILHA',
                'NIVEL_TRIAGEM',
                'N_VALIDACOES',
                'N_DECIDIDOS',
                'N_CONFIRMADOS',
                'TAXA_CONFIRMACAO',
                'IC95_INF',
                'IC95_SUP',
            ]
        )

    decisoes = [
        'CONFIRMADO',
        'JUSTIFICADO',
        'FALSO_POSITIVO',
    ]

    linhas = []

    for (
        trilha,
        nivel
    ), g in (
        df.groupby(
            [
                'CODIGO_TRILHA',
                'NIVEL_TRIAGEM',
            ],
            dropna=False
        )
    ):
        decididos = g[
            g[
                'STATUS_VALIDACAO'
            ].isin(
                decisoes
            )
        ]

        n_dec = len(decididos)

        n_conf = int(
            (
                decididos[
                    'STATUS_VALIDACAO'
                ]
                == 'CONFIRMADO'
            ).sum()
        )

        taxa = (
            n_conf / n_dec
            if n_dec
            else np.nan
        )

        lo, hi = intervalo_wilson(
            n_conf,
            n_dec
        )

        linhas.append({
            'CODIGO_TRILHA':
                trilha,

            'NIVEL_TRIAGEM':
                nivel,

            'N_VALIDACOES':
                len(g),

            'N_DECIDIDOS':
                n_dec,

            'N_CONFIRMADOS':
                n_conf,

            'TAXA_CONFIRMACAO':
                taxa,

            'IC95_INF':
                lo,

            'IC95_SUP':
                hi,
        })

    return pd.DataFrame(linhas)


def estimativa_ponderada_amostra(
    validacoes_df,
    plano_df
):
    """
    Estimativa descritiva ponderada pela população do estrato.
    Só usa estratos com decisão CONFIRMADO/JUSTIFICADO/FALSO_POSITIVO.
    """
    if not len(validacoes_df):
        return pd.DataFrame()

    decisoes = [
        'CONFIRMADO',
        'JUSTIFICADO',
        'FALSO_POSITIVO',
    ]

    d = validacoes_df[
        validacoes_df[
            'STATUS_VALIDACAO'
        ].isin(decisoes)
    ].copy()

    if not len(d):
        return pd.DataFrame()

    d['Y_CONFIRMADO'] = (
        d[
            'STATUS_VALIDACAO'
        ]
        .eq('CONFIRMADO')
        .astype(float)
    )

    estrato = (
        d.groupby(
            [
                'CODIGO_TRILHA',
                'ESTRATO_VALIDACAO'
            ],
            as_index=False
        )
        .agg(
            N_DECIDIDOS=(
                'Y_CONFIRMADO',
                'size'
            ),
            TAXA_CONFIRMACAO_ESTRATO=(
                'Y_CONFIRMADO',
                'mean'
            ),
        )
    )

    estrato = (
        estrato
        .merge(
            plano_df[
                [
                    'CODIGO_TRILHA',
                    'ESTRATO_VALIDACAO',
                    'N_POP_ESTRATO',
                ]
            ],
            on=[
                'CODIGO_TRILHA',
                'ESTRATO_VALIDACAO'
            ],
            how='left'
        )
    )

    resultados = []

    for trilha, g in (
        estrato.groupby(
            'CODIGO_TRILHA'
        )
    ):
        peso_total = g[
            'N_POP_ESTRATO'
        ].sum()

        taxa = (
            np.average(
                g[
                    'TAXA_CONFIRMACAO_ESTRATO'
                ],
                weights=g[
                    'N_POP_ESTRATO'
                ]
            )
            if peso_total > 0
            else np.nan
        )

        resultados.append({
            'CODIGO_TRILHA':
                trilha,

            'TAXA_CONFIRMACAO_PONDERADA':
                taxa,

            'N_ESTRATOS_COM_DECISAO':
                len(g),

            'POPULACAO_REPRESENTADA':
                peso_total,

            'OBSERVACAO':
                (
                    'Estimativa ponderada descritiva; '
                    'intervalo global requer tratamento '
                    'amostral específico.'
                ),
        })

    return pd.DataFrame(
        resultados
    )


resumo_validacoes_inicial_df = (
    resumir_validacoes()
)

salvar_csv(
    resumo_validacoes_inicial_df,
    VALIDACAO_DIR
    / 'resumo_validacoes_por_trilha_nivel.csv'
)

display(status_validacao_df)

display(
    plano_amostral_v13_df
)

print(
    '🧪 Amostra total:',
    len(amostra_validacao_v13_df),
    'sinais'
)

print(
    '✅ Amostragem balanceada por trilha × nível '
    'e infraestrutura de validação concluídas.'
)

## 4️⃣1️⃣ 09 — Contrato do dashboard — V1.3.2

A camada final transforma os resultados metodológicos em contrato para a interface.

### Abas recomendadas

1. **Visão geral**
2. **Motor de Trilhas**
3. **Diagnóstico do Motor**
4. **Sinais e Validação**
5. **Metodologia**
6. **Assistente IA**

### Diagnóstico do Motor

A interface deve deixar claro que existem **duas estratégias de exposição**:

**Fornecedor**

`BANDA_EXPOSICAO_FORNECEDOR`

- 1 compra
- 2 compras
- 3–4
- 5–9
- 10–19
- 20+

**UG**

`DECIL_EXPOSICAO_ANUAL`

- decis 1 a 10 dentro de cada exercício.

A interface não deve rotular as bandas de fornecedor como decis ou percentis.

### Motor de Trilhas

Cada card T01–T09 deve exibir:

- hipótese de controle;
- natureza da evidência;
- família;
- unidade de análise;
- fundamento;
- mensuração;
- parâmetro baseline;
- possibilidade de ajuste;
- contribuição marginal;
- elegibilidade estatística;
- limitação central.

### Cenários experimentais

Toda alteração deve ser rotulada:

> **CENÁRIO EXPERIMENTAL — não altera a baseline metodológica.**

### IA

O assistente pode explicar evidências e fundamentos, mas não declarar fraude automaticamente nem tratar associação como causalidade.

In [ ]:
# ============================================================
# CONTRATO DE ABAS
# ============================================================

abas_dashboard_v13_df = pd.DataFrame([
    {
        'ORDEM': 1,
        'ID_ABA': 'visao_geral',
        'ROTULO': 'Visão geral',
        'OBJETIVO':
            'Descrever universo, materialidade, temporalidade, cobertura e distribuição dos sinais.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
    {
        'ORDEM': 2,
        'ID_ABA': 'motor_trilhas',
        'ROTULO': 'Motor de Trilhas',
        'OBJETIVO':
            'Exibir T01–T09, fundamentos, mensuração, baseline e cenários experimentais.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
    {
        'ORDEM': 3,
        'ID_ABA': 'diagnostico_motor',
        'ROTULO': 'Diagnóstico do Motor',
        'OBJETIVO':
            'Exibir Jaccard, Phi, VIF, PCA, contribuição marginal e famílias.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
    {
        'ORDEM': 4,
        'ID_ABA': 'sinais_validacao',
        'ROTULO': 'Sinais e Validação',
        'OBJETIVO':
            'Filtrar sinais, realizar drill-down e registrar validação quando autorizado.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
    {
        'ORDEM': 5,
        'ID_ABA': 'metodologia',
        'ROTULO': 'Metodologia',
        'OBJETIVO':
            'Documentar dados, regras, versões, elegibilidade, limitações, normas e literatura.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
    {
        'ORDEM': 6,
        'ID_ABA': 'assistente_ia',
        'ROTULO': 'Assistente IA',
        'OBJETIVO':
            'Explicar dados, sinais, parâmetros e fundamentos com rastreabilidade.',
        'MODO_PUBLICO': True,
        'MODO_PESQUISADOR': True,
    },
])


# ============================================================
# CONTRATO DE EXIBIÇÃO DAS TRILHAS
# ============================================================

contrato_trilhas_dashboard_df = (
    catalogo_trilhas_v13_df
    .merge(
        elegibilidade_forn_df[
            [
                'REGRA',
                'N_POSITIVOS',
                'PREVALENCIA',
                'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO',
            ]
        ]
        .rename(
            columns={
                'REGRA':
                    'CODIGO_TRILHA',

                'N_POSITIVOS':
                    'N_POSITIVOS_FORNECEDOR',

                'PREVALENCIA':
                    'PREVALENCIA_FORNECEDOR',

                'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO':
                    'ELEGIBILIDADE_ESTATISTICA_FORNECEDOR',
            }
        ),
        on='CODIGO_TRILHA',
        how='left'
    )
    .merge(
        elegibilidade_ug_df[
            [
                'REGRA',
                'N_POSITIVOS',
                'PREVALENCIA',
                'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO',
            ]
        ]
        .rename(
            columns={
                'REGRA':
                    'CODIGO_TRILHA',

                'N_POSITIVOS':
                    'N_POSITIVOS_UG',

                'PREVALENCIA':
                    'PREVALENCIA_UG',

                'ELEGIBILIDADE_DIAGNOSTICO_ESTATISTICO':
                    'ELEGIBILIDADE_ESTATISTICA_UG',
            }
        ),
        on='CODIGO_TRILHA',
        how='left'
    )
    .merge(
        marginal_forn_df[
            [
                'REGRA_OU_FAMILIA',
                'N_SINALIZADOS',
                'N_EXCLUSIVOS',
                'CONTRIBUICAO_MARGINAL_PCT',
            ]
        ]
        .rename(
            columns={
                'REGRA_OU_FAMILIA':
                    'CODIGO_TRILHA',

                'N_SINALIZADOS':
                    'N_UNIDADES_FORNECEDOR',

                'N_EXCLUSIVOS':
                    'N_EXCLUSIVOS_FORNECEDOR',

                'CONTRIBUICAO_MARGINAL_PCT':
                    'MARGINAL_PCT_FORNECEDOR',
            }
        ),
        on='CODIGO_TRILHA',
        how='left'
    )
    .merge(
        marginal_ug_df[
            [
                'REGRA_OU_FAMILIA',
                'N_SINALIZADOS',
                'N_EXCLUSIVOS',
                'CONTRIBUICAO_MARGINAL_PCT',
            ]
        ]
        .rename(
            columns={
                'REGRA_OU_FAMILIA':
                    'CODIGO_TRILHA',

                'N_SINALIZADOS':
                    'N_UNIDADES_UG',

                'N_EXCLUSIVOS':
                    'N_EXCLUSIVOS_UG',

                'CONTRIBUICAO_MARGINAL_PCT':
                    'MARGINAL_PCT_UG',
            }
        ),
        on='CODIGO_TRILHA',
        how='left'
    )
)


# ============================================================
# CONTRATO JSON
# ============================================================

dashboard_contract_v13 = {
    'metadata': {
        'versao_regras':
            VERSAO_REGRAS,

        'versao_motor':
            VERSAO_MOTOR,

        'motor_fingerprint':
            MOTOR_FINGERPRINT,

        'fonte_verdade_regras':
            'metodologia_1.2.0',

        'fonte_verdade_governanca':
            'motor_1.3.2',
    },

    'principios': [
        'Sinal analítico não equivale a fraude ou irregularidade.',
        'Natureza da evidência é separada do status de validação.',
        'T08 e T09 são contextos, não contagem núcleo.',
        'Não existe score opaco obrigatório.',
        'Parâmetros legais são bloqueados.',
        'Cenários experimentais nunca sobrescrevem a baseline.',
        'Redundância estatística não exclui regra automaticamente.',
        'Assertividade depende de validação humana.',
    ],

    'abas':
        abas_dashboard_v13_df
        .to_dict(
            orient='records'
        ),

    'familias':
        familias_v13_df
        .to_dict(
            orient='records'
        ),

    'parametros':
        parametros_dashboard_v13_df
        .to_dict(
            orient='records'
        ),

    'diagnostico_estatistico': {
        'min_positivos':
            MOTOR_CONFIG[
                'diagnostico'
            ]['min_positivos_estatistica'],

        'min_negativos':
            MOTOR_CONFIG[
                'diagnostico'
            ]['min_negativos_estatistica'],

        'exposicao_fornecedor': {
            'tipo':
                'BANDAS_FIXAS_CONTAGEM',

            'variavel':
                'N_COMPRAS_FORNECEDOR',

            'campo':
                'BANDA_EXPOSICAO_FORNECEDOR',

            'bandas':
                MOTOR_CONFIG[
                    'diagnostico'
                ]['bandas_exposicao_fornecedor'],
        },

        'exposicao_ug': {
            'tipo':
                'DECIL_ANUAL',

            'variavel':
                'N_OPERACOES_EFETIVAS',

            'campo':
                'DECIL_EXPOSICAO_ANUAL',

            'n_decis':
                MOTOR_CONFIG[
                    'diagnostico'
                ]['n_decis_exposicao_ug'],
        },

        'regras_elegiveis_pca_vif_fornecedor':
            flags_forn_modelagem,

        'regras_elegiveis_pca_vif_ug':
            flags_ug_modelagem,
    },

    'sensibilidade': {
        'arquivo_contrato':
            'sensibilidade_motor_contrato.csv',

        'grade_T05':
            'calibracao_nao_comparavel_diretamente_com_episodios',

        'T08':
            'populacao_e_MAD_nao_sao_N_sinais',

        'T09':
            'proximidade_ajusta_subconjunto_e_total_do_cenario',
    },

    'validacao': {
        'status_permitidos':
            MOTOR_CONFIG[
                'validacao'
            ]['status_permitidos'],

        'confirmado_automaticamente':
            False,
    },

    'assistente_ia': {
        'pode': [
            'explicar evidência estruturada',
            'explicar parâmetros',
            'recuperar fundamento normativo',
            'comparar períodos e entidades',
            'declarar limitações e abstinência',
        ],

        'nao_pode': [
            'declarar fraude automaticamente',
            'declarar fracionamento sem objeto/documentação',
            'alterar limite legal',
            'tratar correlação como causalidade',
            'ocultar a versão da regra',
        ],
    },
}


# ============================================================
# EXPORTAR CONTRATO
# ============================================================

salvar_csv(
    abas_dashboard_v13_df,
    CONTRATO_DASHBOARD_DIR
    / 'abas_dashboard.csv'
)

salvar_csv(
    contrato_trilhas_dashboard_df,
    CONTRATO_DASHBOARD_DIR
    / 'contrato_trilhas_dashboard.csv'
)

salvar_csv(
    parametros_dashboard_v13_df,
    CONTRATO_DASHBOARD_DIR
    / 'parametros_dashboard.csv'
)

salvar_json(
    dashboard_contract_v13,
    CONTRATO_DASHBOARD_DIR
    / 'dashboard_contract_v1_3_2.json'
)


# ============================================================
# MASTER GOVERNADO PARA FUTURO DASHBOARD
# ============================================================

governado_v13_df = (
    trilha_resultados_df
    .merge(
        catalogo_trilhas_v13_df[
            [
                'CODIGO_TRILHA',
                'FAMILIA',
                'FAMILIA_NOME',
                'TIPO_EVIDENCIA',
                'PAPEL_MOTOR',
            ]
        ],
        on='CODIGO_TRILHA',
        how='left'
    )
)

governado_v13_df[
    'STATUS_VALIDACAO'
] = 'NAO_VALIDADO'

governado_v13_df[
    'VERSAO_MOTOR'
] = VERSAO_MOTOR

salvar_parquet(
    governado_v13_df,
    CONTRATO_DASHBOARD_DIR
    / 'trilha_resultados_governado.parquet'
)


print(
    '🖥️ Contrato do dashboard V1.3.2'
)

display(
    abas_dashboard_v13_df
)

display(
    contrato_trilhas_dashboard_df[
        [
            'CODIGO_TRILHA',
            'FAMILIA',
            'TIPO_EVIDENCIA',
            'PAPEL_MOTOR',
            'PARAMETRO_USUARIO',
        ]
    ]
)

print(
    '✅ Contrato do dashboard criado.'
)

## 4️⃣2️⃣ Controle de regressão — V1.2 × Motor V1.3.2

A V1.3.2 altera somente a governança da exposição.

As regras continuam em `VERSAO_REGRAS = 1.2.0`.

Logo, T01–T09 devem reproduzir exatamente a baseline congelada para o mesmo consolidado.

Qualquer diferença exige investigação.

In [ ]:
contagem_v13_df = (
    trilha_resultados_df
    .groupby(
        'CODIGO_TRILHA',
        as_index=False
    )
    .agg(
        N_V131=(
            'ID_SINAL',
            'count'
        )
    )
)

baseline_v12_df = pd.DataFrame([
    {
        'CODIGO_TRILHA':
            k,

        'N_V12':
            v,
    }

    for k, v
    in BASELINE_V12.items()
])

regressao_v13_df = (
    baseline_v12_df
    .merge(
        contagem_v13_df,
        on='CODIGO_TRILHA',
        how='outer'
    )
)

regressao_v13_df[
    'N_V12'
] = (
    regressao_v13_df[
        'N_V12'
    ]
    .fillna(0)
    .astype(int)
)

regressao_v13_df[
    'N_V131'
] = (
    regressao_v13_df[
        'N_V131'
    ]
    .fillna(0)
    .astype(int)
)

regressao_v13_df[
    'DIFERENCA'
] = (
    regressao_v13_df[
        'N_V131'
    ]
    - regressao_v13_df[
        'N_V12'
    ]
)

regressao_v13_df[
    'REGRESSAO_OK'
] = (
    regressao_v13_df[
        'DIFERENCA'
    ]
    == 0
)

salvar_csv(
    regressao_v13_df,
    GOVERNANCA_DIR
    / 'controle_regressao_v1_2_v1_3_2.csv'
)

display(
    regressao_v13_df
)

if not regressao_v13_df[
    'REGRESSAO_OK'
].all():
    print(
        '⚠️ A V1.3.2 não reproduziu integralmente '
        'a baseline V1.2. Investigue as diferenças '
        'antes de congelar o motor.'
    )
else:
    print(
        '✅ T01–T09 reproduziram a baseline '
        'congelada da V1.2.'
    )

## 4️⃣3️⃣ Gráficos de síntese

Nesta etapa são criados gráficos pequenos e interpretáveis para apoiar o diagnóstico.

Os resultados analíticos completos permanecem em Parquet/CSV.

In [ ]:
if GERAR_GRAFICOS:

    if len(
        t01_resumo_df
    ):
        plt.figure(
            figsize=(12, 5.5)
        )

        plt.plot(
            t01_resumo_df[
                'ANO'
            ],
            t01_resumo_df[
                'N_SINAIS'
            ],
            marker='o'
        )

        plt.title(
            (
                'T01 — Compras em '
                'finais de semana por ano'
            )
        )

        plt.xlabel(
            'Ano'
        )

        plt.ylabel(
            'Quantidade de sinais'
        )

        salvar_fig(
            GRAFICOS_DIR
            / 't01_fim_semana_anual.png'
        )


    recorte_t05 = (
        t05_sensibilidade_df[
            (
                t05_sensibilidade_df[
                    'min_portadores'
                ]
                == 2
            )
            &
            (
                t05_sensibilidade_df[
                    'min_transacoes'
                ]
                == 5
            )
            &
            (
                t05_sensibilidade_df[
                    'cv_limite'
                ]
                .isin(
                    [
                        0.10,
                        0.20,
                        0.30
                    ]
                )
            )
        ]
        .copy()
    )

    if len(
        recorte_t05
    ):
        plt.figure(
            figsize=(11, 5.5)
        )

        for (
            cv_limite,
            g
        ) in recorte_t05.groupby(
            'cv_limite'
        ):
            g = g.sort_values(
                'janela_dias'
            )

            plt.plot(
                g[
                    'janela_dias'
                ],
                g[
                    'N_GRUPOS'
                ],
                marker='o',
                label=(
                    f'CV ≤ '
                    f'{cv_limite:.0%}'
                )
            )

        plt.title(
            (
                'T05 — Sensibilidade '
                'da quantidade de grupos'
            )
        )

        plt.xlabel(
            'Janela temporal (dias)'
        )

        plt.ylabel(
            'Grupos representativos'
        )

        plt.legend()

        salvar_fig(
            GRAFICOS_DIR
            / 't05_sensibilidade_resumo.png'
        )


    t06_ind_df = con.execute(
        f"""
        SELECT
            TOP1_SHARE

        FROM read_parquet(
            '{t06_ind_sql}'
        )

        WHERE
            TOP1_SHARE
            IS NOT NULL
        """
    ).df()

    if len(
        t06_ind_df
    ):
        plt.figure(
            figsize=(10, 5.5)
        )

        plt.hist(
            t06_ind_df[
                'TOP1_SHARE'
            ],
            bins=40
        )

        plt.axvline(
            0.50,
            linestyle='--',
            label='50%'
        )

        plt.axvline(
            0.70,
            linestyle=':',
            label='70%'
        )

        plt.axvline(
            0.80,
            linestyle='-.',
            label='80%'
        )

        plt.title(
            (
                'T06 — Distribuição '
                'da concentração Top-1'
            )
        )

        plt.xlabel(
            (
                'Participação do '
                'maior fornecedor'
            )
        )

        plt.ylabel(
            'UG-anos'
        )

        plt.legend()

        salvar_fig(
            GRAFICOS_DIR
            / 't06_distribuicao_top1.png'
        )


print(
    '✅ Gráficos de síntese concluídos.'
)

## 4️⃣4️⃣ Exportações

Os arquivos completos permanecem prioritariamente em:

- **Parquet** — processamento e dashboard;
- **CSV** — interoperabilidade;
- **Excel** — apenas tabelas-resumo.

Essa divisão evita limitações do Excel e mantém o projeto escalável.

In [ ]:
EXCEL_RESUMO = (
    EXPORTACOES_DIR
    / 'Resumo_Diagnostico_Trilhas_CPGF.xlsx'
)

abas = {
    '00_integridade':
        controle_raw,

    '01_tipos_transacao':
        tipos_transacao_df,

    '02_diagnostico_anual':
        diagnostico_anual_df,

    '03_cobertura_fornec':
        cobertura_fornecedor_df,

    '04_valores_stats':
        estatisticas_valores_df,

    '05_T01_anual': t01_resumo_df,
    '05b_T01_recorrencia': t01_recorrencia_resumo_df,

    '06_T02':
        t02_resumo_df,

    '07_T03': t03_resumo_df,
    '07b_T03_integral_obs': t03_integral_resumo_df,

    '08_T04':
        t04_resumo_df,

    '09_T05_sensibilidade':
        t05_sensibilidade_df,

    '10_T06':
        t06_resumo_df,

    '11_T07': t07_resumo_df,
    '11b_T07_recorrencia': t07_recorrencia_resumo_df,

    '12_Benford_anual': benford_anual_df,
    '12b_Benford_arred': benford_arredondamento_global_df,

    '13_Benford_persist':
        benford_persistencia_df
        .head(
            100000
        ),

    '13b_T09': t09_resumo_df,
    '13c_T09_cenarios': t09_resumo_cenarios_df,
    '13d_T09_cobertura': t09_cobertura_df,

    '14_convergencia_ug':
        convergencia_ug_ano_df
        .head(
            100000
        ),

    '15_converg_fornecedor':
        convergencia_fornecedor_df
        .head(
            100000
        ),

    '16_familias':
        catalogo_trilhas_v13_df,

    '17_sobreposicao_forn':
        sobreposicao_forn_df,

    '18_multicol_forn':
        multicol_forn_df,

    '19_marginal_forn':
        marginal_forn_df,

    '20_pca_variancia':
        pca_var_forn_df,

    '21_sens_motor':
        sens_motor_df
        .head(
            100000
        ),

    '22_regressao_v131':
        regressao_v13_df,

    '23_elegibilidade_forn':
        elegibilidade_forn_df,

    '24_elegibilidade_ug':
        elegibilidade_ug_df,

    '25_plano_validacao':
        plano_amostral_v13_df,

    '26_bandas_fornecedor':
        perfil_bandas_fornecedor_df,

    '27_decis_ug':
        perfil_decis_ug_df,
}

with pd.ExcelWriter(
    EXCEL_RESUMO,
    engine='openpyxl'
) as writer:

    for nome, df in abas.items():

        if df is None:
            continue

        df_excel = (
            df.copy()
        )

        if len(
            df_excel
        ) > 1_000_000:
            df_excel = (
                df_excel
                .head(
                    1_000_000
                )
            )

        df_excel.to_excel(
            writer,
            sheet_name=nome[:31],
            index=False
        )

    pd.DataFrame([
        {
            'campo':
                'data_hora',
            'valor':
                datetime.now()
                .isoformat(),
        },
        {
            'campo':
                'arquivo_entrada',
            'valor':
                str(
                    INPUT_CSV
                ),
        },
        {
            'campo':
                'sha256',
            'valor':
                HASH_INPUT,
        },
        {
            'campo':
                'run_fingerprint',
            'valor':
                RUN_FINGERPRINT,
        },
        {
            'campo':
                'versao_regras',
            'valor':
                VERSAO_REGRAS,
        },
        {
            'campo':
                'versao_preparacao',
            'valor':
                VERSAO_PREPARACAO,
        },
        {
            'campo':
                'versao_motor',
            'valor':
                VERSAO_MOTOR,
        },
        {
            'campo':
                'motor_fingerprint',
            'valor':
                MOTOR_FINGERPRINT,
        },
        {
            'campo':
                'salvaguarda',
            'valor':
                (
                    'As saídas constituem sinais '
                    'analíticos e não comprovam '
                    'fraude, irregularidade ou '
                    'fracionamento.'
                ),
        },
    ]).to_excel(
        writer,
        sheet_name='99_metadata',
        index=False
    )

print(
    '📘 Excel de resumo:',
    EXCEL_RESUMO
)

## 4️⃣5️⃣ Validação final, log e parâmetros

A última etapa registra:

- arquivo e hash de entrada;
- versões das regras;
- configuração completa;
- ambiente;
- caminhos de saída;
- contagens resumidas;
- data/hora.

Também confere a existência dos principais produtos da execução.

In [ ]:
salvar_json(
    CONFIG,
    CONTROLE_DIR
    / 'config_trilhas.json'
)

salvar_json(
    {
        'CODIGO_COMPRA_NACIONAL':
            CODIGO_COMPRA_NACIONAL,

        'CODIGO_COMPRA_INTERNACIONAL':
            CODIGO_COMPRA_INTERNACIONAL,

        'CODIGO_COMPRA_PARCELADA':
            CODIGO_COMPRA_PARCELADA,

        'SAQUES_EFETIVOS':
            SAQUES_EFETIVOS,

        'CODIGOS_AJUSTE_CONTESTACAO':
            CODIGOS_AJUSTE_CONTESTACAO,
    },
    CONTROLE_DIR
    / 'transaction_codes.json'
)


arquivos_criticos = {
    'stg_parquet':
        STG_PARQUET,

    'master_sinais':
        MASTER_PARQUET,

    'ponte_sinal_transacao':
        PONTE_MASTER,

    't05_sensibilidade':
        T05_DIR
        / 't05_sensibilidade_completa.parquet',

    'benford_ug_ano':
        T08_DIR
        / '10_benford_ug_ano.parquet',

    't03_integral_observavel':
        T03_INTEGRAL,

    't03_integral_observavel_ponte':
        T03_INTEGRAL_PONTE,

    't07_recorrencia_prioritaria':
        T07_PRIORITARIOS,

    't09_sinais':
        T09_SINAIS,

    't09_agregados':
        T09_AGREGADOS,

    'excel_resumo':
        EXCEL_RESUMO,

    'matriz_flags_fornecedor':
        MATRIZES_DIR
        / 'matriz_flags_ug_fornecedor_ano.parquet',

    'matriz_flags_ug':
        MATRIZES_DIR
        / 'matriz_flags_ug_ano.parquet',

    'elegibilidade_fornecedor':
        MATRIZES_DIR
        / 'elegibilidade_estatistica_fornecedor.csv',

    'elegibilidade_ug':
        MATRIZES_DIR
        / 'elegibilidade_estatistica_ug.csv',

    'sobreposicao_banda_fornecedor':
        SOBREPOSICAO_DIR
        / 'sobreposicao_trilhas_fornecedor_por_banda_exposicao.csv',

    'sobreposicao_decil_ug':
        SOBREPOSICAO_DIR
        / 'sobreposicao_trilhas_ug_por_decil_exposicao.csv',

    'perfil_bandas_fornecedor':
        MATRIZES_DIR
        / 'perfil_bandas_exposicao_fornecedor.csv',

    'perfil_decis_ug':
        MATRIZES_DIR
        / 'perfil_decis_exposicao_ug.csv',

    'multicolinearidade_fornecedor':
        MULTICOL_DIR
        / 'multicolinearidade_fornecedor.csv',

    'contribuicao_marginal_banda_fornecedor':
        MARGINAL_DIR
        / 'contribuicao_marginal_trilhas_fornecedor_por_banda_exposicao.csv',

    'contribuicao_marginal_decil_ug':
        MARGINAL_DIR
        / 'contribuicao_marginal_trilhas_ug_por_decil_exposicao.csv',

    'sensibilidade_contrato':
        SENSIBILIDADE_MOTOR_DIR
        / 'sensibilidade_motor_contrato.csv',

    'plano_validacao':
        VALIDACAO_DIR
        / 'plano_amostral_trilha_nivel.csv',

    'dashboard_contract':
        CONTRATO_DASHBOARD_DIR
        / 'dashboard_contract_v1_3_2.json',
}

validacao_arquivos = []

for nome, caminho in (
    arquivos_criticos
    .items()
):
    caminho = Path(
        caminho
    )

    validacao_arquivos.append({
        'produto':
            nome,
        'existe':
            caminho.exists(),
        'caminho':
            str(
                caminho
            ),
        'tamanho_mb':
            round(
                caminho.stat().st_size
                / (1024**2),
                2
            )
            if caminho.exists()
            else np.nan,
    })

validacao_arquivos_df = (
    pd.DataFrame(
        validacao_arquivos
    )
)

salvar_csv(
    validacao_arquivos_df,
    CONTROLE_DIR
    / 'validacao_produtos.csv'
)

display(
    validacao_arquivos_df
)

if not validacao_arquivos_df[
    'existe'
].all():
    print(
        '⚠️ Um ou mais produtos '
        'esperados não foram encontrados.'
    )
else:
    print(
        '✅ Produtos críticos encontrados.'
    )


log_final = {
    'data_hora_execucao':
        datetime.now().isoformat(),

    'arquivo_entrada':
        str(
            INPUT_CSV
        ),

    'sha256_input':
        HASH_INPUT,

    'run_fingerprint':
        RUN_FINGERPRINT,

    'versao_regras':
        VERSAO_REGRAS,

    'versao_preparacao':
        VERSAO_PREPARACAO,

    'versao_motor':
        VERSAO_MOTOR,

    'motor_fingerprint':
        MOTOR_FINGERPRINT,

    'n_registros':
        n_registros,

    'n_competencias':
        n_comp,

    'competencia_min':
        comp_min,

    'competencia_max':
        comp_max,

    'result_dir':
        str(
            RESULT_DIR
        ),

    'stg_parquet':
        str(
            STG_PARQUET
        ),

    'excel_resumo':
        str(
            EXCEL_RESUMO
        ),

    't01_sinais':
        int(
            t01_resumo_df[
                'N_SINAIS'
            ].sum()
        )
        if len(
            t01_resumo_df
        )
        else 0,

    't02_sinais':
        int(
            t02_resumo_df[
                'N_SINAIS'
            ].iloc[0]
        )
        if len(
            t02_resumo_df
        )
        else 0,

    't03_grupos':
        int(
            t03_resumo_df[
                'N_GRUPOS'
            ].sum()
        )
        if len(
            t03_resumo_df
        )
        else 0,

    't03b_grupos_integrais_observaveis':
        int(
            t03_integral_resumo_df[
                'N_GRUPOS_INTEGRAIS_OBSERVAVEIS'
            ].iloc[0]
        )
        if len(
            t03_integral_resumo_df
        )
        else 0,

    't04_grupos':
        int(
            t04_resumo_df[
                'N_GRUPOS'
            ].sum()
        )
        if len(
            t04_resumo_df
        )
        else 0,

    't05_episodios':
        int(
            len(
                t05_episodios_df
            )
        ),

    't06_sinais':
        int(
            t06_resumo_df[
                'N_UG_ANO'
            ].sum()
        )
        if len(
            t06_resumo_df
        )
        else 0,

    't07_episodios':
        int(
            t07_resumo_df[
                'N_EPISODIOS_DIARIOS'
            ].sum()
        )
        if len(
            t07_resumo_df
        )
        else 0,

    't08_ugs_persistencia_elevada':
        int(
            benford_persistencia_df['PERSISTENCIA_RELATIVA_ELEVADA'].sum()
        )
        if len(
            benford_persistencia_df
        )
        else 0,

    't09_status': 'EXECUTADO_CENARIOS_PARALELOS_STATUS_SEPARADOS',
    't09_sinais': int(len(t09_master)) if 't09_master' in globals() else 0,

    'governanca_unidades_fornecedor':
        int(
            len(
                matriz_fornecedor_v13_df
            )
        ),

    'governanca_unidades_ug':
        int(
            len(
                matriz_ug_v13_df
            )
        ),

    'governanca_regressao_v12_ok':
        bool(
            regressao_v13_df[
                'REGRESSAO_OK'
            ].all()
        ),

    'governanca_flags_modelagem_fornecedor':
        flags_forn_modelagem,

    'governanca_flags_modelagem_ug':
        flags_ug_modelagem,

    'governanca_exposicao_fornecedor':
        {
            'tipo':
                'BANDAS_FIXAS_CONTAGEM',

            'bandas':
                [
                    b['rotulo']
                    for b in MOTOR_CONFIG[
                        'diagnostico'
                    ]['bandas_exposicao_fornecedor']
                ],
        },

    'governanca_exposicao_ug':
        {
            'tipo':
                'DECIL_ANUAL',

            'n_decis':
                MOTOR_CONFIG[
                    'diagnostico'
                ]['n_decis_exposicao_ug'],
        },

    'amostra_validacao_n':
        int(
            len(
                amostra_validacao_v13_df
            )
        ),

    'salvaguarda':
        (
            'As trilhas produzem sinais analíticos. '
            'Não constituem comprovação automática '
            'de fraude, irregularidade ou fracionamento.'
        ),
}

salvar_json(
    log_final,
    LOGS_DIR
    / 'log_execucao_diagnostico_calibracao.json'
)

display(
    pd.DataFrame(
        [log_final]
    )
    .T
    .rename(
        columns={
            0:
                'valor'
        }
    )
)

print(
    '\n'
    + '=' * 80
)

print(
    '🎉 EXECUÇÃO FINALIZADA'
)

print(
    '=' * 80
)

print(
    '📁 Resultados:',
    RESULT_DIR
)

print(
    '📘 Excel:',
    EXCEL_RESUMO
)

print(
    '📊 Gráficos:',
    GRAFICOS_DIR
)

print(
    '🧾 Log:',
    LOGS_DIR
    / 'log_execucao_diagnostico_calibracao.json'
)

print(
    '🔐 SHA-256:',
    HASH_INPUT
)

print(
    '🧬 Fingerprint regras:',
    RUN_FINGERPRINT[:24]
)

print(
    '🧭 Fingerprint motor:',
    MOTOR_FINGERPRINT[:24]
)

## ✅ Encerramento — Versão 1.3.2

Ao concluir esta execução, o projeto terá:

### Metodologia das trilhas

`VERSAO_REGRAS = 1.2.0`

T01–T09 permanecem congeladas e são reexecutadas integralmente sobre o consolidado bruto.

### Governança do motor

`VERSAO_MOTOR = 1.3.2`

A camada passa a utilizar:

#### `UG × fornecedor × ano`

- exposição medida por `N_COMPRAS_FORNECEDOR`;
- seis bandas fixas:
  - 1;
  - 2;
  - 3–4;
  - 5–9;
  - 10–19;
  - 20+;
- Jaccard/Phi por banda;
- contribuição marginal por banda.

#### `UG × ano`

- exposição medida por `N_OPERACOES_EFETIVAS`;
- decis anuais 1–10;
- Jaccard/Phi por decil;
- contribuição marginal por decil.

### Elementos preservados

- famílias de evidência;
- controle de raridade;
- PCA/VIF somente para flags elegíveis;
- sensibilidade semanticamente tipada;
- validação por trilha × nível;
- pesos amostrais;
- contrato do dashboard;
- regressão rígida T01–T09.

### Critério de congelamento

A governança 1.3.2 poderá ser considerada congelada se:

1. a regressão T01–T09 permanecer integralmente igual à V1.2;
2. as seis bandas de fornecedor recompuserem 100% do universo observável;
3. os diagnósticos por banda confirmarem que T03/T04/T05 permanecem associados, porém não redundantes;
4. os decis UG continuarem mostrando o efeito de exposição identificado na V1.3.1;
5. nenhum novo problema de semântica ou cobertura for identificado.

Cumpridos esses critérios, o próximo passo recomendado é a extração para `src/`, sem nova rodada de calibração do notebook.